# 02 Data Quality Assessment

This notebook is **structure only**. It defines what will eventually be tested and in what order — it does not yet implement any test, does not run any analysis, and does not clean, transform, or modify any data. Each section below is a Markdown heading followed by a code cell containing comments describing the checks that section will eventually contain.

**Pipeline this notebook sits in:**

```
01_source_exploration.ipynb
        │
        │ discoveries
        ▼
02_data_quality.ipynb            <- you are here (structure only, for now)
        │
        │ validated rules
        ▼
profile_source.py
        │
        │ repeatable checks
        ▼
PostgreSQL ingestion
```

Not every check written here will end up automated in `profile_source.py`. Some checks are deterministic structural rules (e.g. `ON_STREAM_HRS > 24` on a daily record is always a violation) and belong in automation with a clear **PASS/FAIL** outcome. Others require engineering interpretation (e.g. an abrupt pressure change could be a shut-in, intervention, restart, well test, or a real problem) and belong in automation only as a **REVIEW** flag, never an automatic FAIL — the automated version should surface it for a human, not judge it. This distinction is revisited explicitly in Section 25.


## Purpose

In [1]:
# Explain that this notebook tests the assumptions and potential issues identified during source exploration (01_source_exploration.ipynb).
# State that the objective is to determine whether the data is suitable for relational modelling and subsequent analysis.
# State that no source values should be modified without an explicit documented decision.

## 1. Load source data

In [2]:
# Load the original source workbook from data/raw.
# Load daily and monthly production sheets.
# Preserve the original source values.
# Verify that expected files and worksheets exist.

## 2. Schema validation

In [3]:
# Confirm expected columns exist.
# Identify missing expected columns.
# Identify unexpected additional columns.
# Compare current source schema with the schema observed during initial exploration.
# Detect column-name changes.
# Detect duplicate column names.

## 3. Data type validation

In [4]:
# Test whether date columns can be parsed as dates.
# Test whether expected numeric columns contain non-numeric values.
# Test whether identifier fields have appropriate representations.
# Check whether integer-like identifiers have been interpreted as floating-point values.
# Document type-conversion failures.
# Do not silently coerce invalid values.

## 4. Dataset grain validation

**Objective:** test whether the Daily Production Data worksheet has the grain *one row per NPD wellbore per production date*.

**Candidate key:** `NPD_WELL_BORE_CODE` + `DATEPRD`.

This section tests, and does not assume, the grain proposed during exploration (`01_source_exploration.ipynb`, Section 8). Any duplicates found are reported only — nothing is removed, modified, aggregated, or corrected here.


In [5]:
# --- TEMPORARY LOADING FOR SECTION 4 ---
# Section 1 (Load source data) has not been implemented yet in this notebook.
# This block loads only what Section 4 needs, and is clearly marked so it can
# be deleted once Section 1 provides `daily_df` for the whole notebook.
if "daily_df" not in globals():
    from pathlib import Path
    import pandas as pd

    PROJECT_ROOT = Path.cwd().parent
    WORKBOOK_PATH = PROJECT_ROOT / "data" / "raw" / "Volve production data.xlsx"
    DAILY_SHEET_NAME = "Daily Production Data"

    if not WORKBOOK_PATH.exists():
        raise FileNotFoundError(f"Source workbook not found at {WORKBOOK_PATH}")

    daily_df = pd.read_excel(WORKBOOK_PATH, sheet_name=DAILY_SHEET_NAME)
    print(f"[Section 4 temporary load] daily_df loaded: {daily_df.shape}")
else:
    print(f"Using daily_df already loaded earlier in the notebook: {daily_df.shape}")


[Section 4 temporary load] daily_df loaded: (15634, 24)


In [6]:
# Confirm the candidate-key columns exist before testing anything else.
required_columns = ["NPD_WELL_BORE_CODE", "DATEPRD"]
missing_columns = [c for c in required_columns if c not in daily_df.columns]

if missing_columns:
    raise ValueError(f"Required column(s) missing from daily_df: {missing_columns}")

print(f"Required candidate-key columns present: {required_columns}")


Required candidate-key columns present: ['NPD_WELL_BORE_CODE', 'DATEPRD']


In [7]:
# Parse DATEPRD into a separate Series for grain testing only.
# daily_df["DATEPRD"] itself is left completely unmodified.
dateprd_parsed = pd.to_datetime(daily_df["DATEPRD"], errors="coerce")

unparseable_mask = dateprd_parsed.isna() & daily_df["DATEPRD"].notna()
n_unparseable = int(unparseable_mask.sum())

print(f"Unparseable DATEPRD values: {n_unparseable}")
if n_unparseable:
    daily_df.loc[unparseable_mask]


Unparseable DATEPRD values: 0


In [8]:
# Check for NULLs in either candidate-key component.
# dateprd_parsed.isna() covers both originally-missing dates and unparseable ones.
null_code_mask = daily_df["NPD_WELL_BORE_CODE"].isna()
null_date_mask = dateprd_parsed.isna()

n_null_code = int(null_code_mask.sum())
n_null_date = int(null_date_mask.sum())
n_missing_key = int((null_code_mask | null_date_mask).sum())

print(f"Rows with missing NPD_WELL_BORE_CODE:        {n_null_code}")
print(f"Rows with missing/unparseable DATEPRD:       {n_null_date}")
print(f"Rows with any missing candidate-key field:   {n_missing_key}")


Rows with missing NPD_WELL_BORE_CODE:        0
Rows with missing/unparseable DATEPRD:       0
Rows with any missing candidate-key field:   0


In [9]:
# Build a validation-only frame: daily_df's columns plus the parsed date,
# used solely to test the candidate key. daily_df is not modified.
grain_check_df = daily_df.copy()
grain_check_df["DATEPRD_parsed"] = dateprd_parsed

key_cols = ["NPD_WELL_BORE_CODE", "DATEPRD_parsed"]
has_complete_key = grain_check_df[key_cols].notna().all(axis=1)
complete_key_df = grain_check_df.loc[has_complete_key]

total_rows = len(grain_check_df)

combination_sizes = complete_key_df.groupby(key_cols).size()
duplicated_combinations = combination_sizes[combination_sizes > 1]
n_distinct_combinations = combination_sizes.shape[0]

dup_key_mask = complete_key_df.duplicated(subset=key_cols, keep=False)
duplicate_rows = complete_key_df.loc[complete_key_df.index[dup_key_mask]]

print(f"Total rows:                                       {total_rows}")
print(f"Rows with missing candidate-key fields:           {n_missing_key}")
print(f"Distinct NPD_WELL_BORE_CODE + DATEPRD combos:     {n_distinct_combinations}")
print(f"Duplicated key combinations:                      {len(duplicated_combinations)}")
print(f"Rows involved in duplicated key combinations:     {len(duplicate_rows)}")


Total rows:                                       15634
Rows with missing candidate-key fields:           0
Distinct NPD_WELL_BORE_CODE + DATEPRD combos:     15634
Duplicated key combinations:                      0
Rows involved in duplicated key combinations:     0


**Rows involved in duplicated wellbore/date combinations** (reported only — not modified):


In [10]:
report_columns = [
    "DATEPRD",
    "NPD_WELL_BORE_CODE",
    "NPD_WELL_BORE_NAME",
    "FLOW_KIND",
    "WELL_TYPE",
    "ON_STREAM_HRS",
    "BORE_OIL_VOL",
    "BORE_GAS_VOL",
    "BORE_WAT_VOL",
    "BORE_WI_VOL",
]
available_report_columns = [c for c in report_columns if c in duplicate_rows.columns]

duplicate_rows_display = duplicate_rows[available_report_columns].sort_values(
    ["NPD_WELL_BORE_CODE", "DATEPRD"]
)
duplicate_rows_display


,DATEPRD,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,FLOW_KIND,WELL_TYPE,ON_STREAM_HRS,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL


In [11]:
# Distinguish exact duplicate rows (identical across every original column)
# from duplicate keys where at least one non-key value differs.
original_columns = daily_df.columns.tolist()

exact_duplicate_mask = duplicate_rows.duplicated(subset=original_columns, keep=False)
exact_duplicate_rows = duplicate_rows.loc[duplicate_rows.index[exact_duplicate_mask]]
conflicting_duplicate_rows = duplicate_rows.loc[~duplicate_rows.index.isin(exact_duplicate_rows.index)]

print(f"Rows that are exact duplicates (identical across all columns): {len(exact_duplicate_rows)}")
print(f"Rows sharing a key but differing in at least one other column: {len(conflicting_duplicate_rows)}")


Rows that are exact duplicates (identical across all columns): 0
Rows sharing a key but differing in at least one other column: 0


In [12]:
# Final grain validation verdict.
if len(duplicated_combinations) > 0:
    grain_verdict = "NOT CONFIRMED"
elif n_missing_key > 0:
    grain_verdict = "REQUIRES REVIEW"
else:
    grain_verdict = "CONFIRMED"

print("=" * 60)
print("SECTION 4 - GRAIN VALIDATION SUMMARY")
print("=" * 60)
print(f"Candidate key:                            NPD_WELL_BORE_CODE + DATEPRD")
print(f"Total rows:                                {total_rows}")
print(f"Rows with missing candidate-key fields:    {n_missing_key}")
print(f"Distinct key combinations:                 {n_distinct_combinations}")
print(f"Duplicated key combinations:               {len(duplicated_combinations)}")
print(f"Rows involved in duplicates:               {len(duplicate_rows)}")
print(f"  - exact duplicate rows:                  {len(exact_duplicate_rows)}")
print(f"  - conflicting duplicate rows:            {len(conflicting_duplicate_rows)}")
print()
print(f"Verdict: {grain_verdict}")


SECTION 4 - GRAIN VALIDATION SUMMARY
Candidate key:                            NPD_WELL_BORE_CODE + DATEPRD
Total rows:                                15634
Rows with missing candidate-key fields:    0
Distinct key combinations:                 15634
Duplicated key combinations:               0
Rows involved in duplicates:               0
  - exact duplicate rows:                  0
  - conflicting duplicate rows:            0

Verdict: CONFIRMED


**Interpretation for later relational modelling:**

The candidate key `NPD_WELL_BORE_CODE + DATEPRD` produced zero duplicated combinations and zero rows with missing key fields across all rows tested, so the grain **CONFIRMED**: one row per NPD wellbore per production date. This means `NPD_WELL_BORE_CODE + DATEPRD` is a validated candidate for the daily production fact table's primary key, and no deduplication or grain-repair logic is needed before that table can be built on this worksheet. This result only covers the Daily Production Data worksheet in isolation — it does not confirm or test the monthly worksheet's grain (Section 5) or any relationship between the two (deferred to later sections).


## 5. Monthly grain validation

**Objective:** test whether the Monthly Production Data worksheet has the grain *one row per NPD wellbore per year/month*.

**Candidate key:** `NPDCode` + `Year` + `Month`.

This section tests, and does not assume, the grain proposed during exploration (`01_source_exploration.ipynb`, Section 8). Any duplicates found are reported only — nothing is removed, modified, aggregated, or corrected here.


In [13]:
# --- TEMPORARY LOADING FOR SECTION 5 ---
# Section 1 (Load source data) has not been implemented yet in this notebook.
# This block loads only what Section 5 needs, and is clearly marked so it can
# be deleted once Section 1 provides `monthly_df` for the whole notebook.
if "monthly_df" not in globals():
    from pathlib import Path
    import pandas as pd

    PROJECT_ROOT = Path.cwd().parent
    WORKBOOK_PATH = PROJECT_ROOT / "data" / "raw" / "Volve production data.xlsx"
    MONTHLY_SHEET_NAME = "Monthly Production Data"

    if not WORKBOOK_PATH.exists():
        raise FileNotFoundError(f"Source workbook not found at {WORKBOOK_PATH}")

    monthly_df = pd.read_excel(WORKBOOK_PATH, sheet_name=MONTHLY_SHEET_NAME)
    print(f"[Section 5 temporary load] monthly_df loaded: {monthly_df.shape}")
else:
    print(f"Using monthly_df already loaded earlier in the notebook: {monthly_df.shape}")


[Section 5 temporary load] monthly_df loaded: (527, 10)


In [14]:
# Confirm the candidate-key columns exist before testing anything else.
required_columns = ["NPDCode", "Year", "Month"]
missing_columns = [c for c in required_columns if c not in monthly_df.columns]

if missing_columns:
    raise ValueError(f"Required column(s) missing from monthly_df: {missing_columns}")

print(f"Required candidate-key columns present: {required_columns}")


Required candidate-key columns present: ['NPDCode', 'Year', 'Month']


In [15]:
# Check for NULLs in each candidate-key component individually, then combined.
null_npdcode_mask = monthly_df["NPDCode"].isna()
null_year_mask = monthly_df["Year"].isna()
null_month_mask = monthly_df["Month"].isna()

n_null_npdcode = int(null_npdcode_mask.sum())
n_null_year = int(null_year_mask.sum())
n_null_month = int(null_month_mask.sum())

has_complete_key = ~(null_npdcode_mask | null_year_mask | null_month_mask)
n_missing_key = int((~has_complete_key).sum())

print(f"Rows with missing NPDCode:                  {n_null_npdcode}")
print(f"Rows with missing Year:                     {n_null_year}")
print(f"Rows with missing Month:                    {n_null_month}")
print(f"Rows with any missing candidate-key field:  {n_missing_key}")


Rows with missing NPDCode:                  1
Rows with missing Year:                     1
Rows with missing Month:                    1
Rows with any missing candidate-key field:  1


In [16]:
# Inspect the rows with an incomplete key, if any - reported only.
missing_key_rows = monthly_df.loc[~has_complete_key]
print(f"Rows with an incomplete candidate key: {len(missing_key_rows)}")
missing_key_rows

Rows with an incomplete candidate key: 1


,Wellbore name,NPDCode,Year,Month,On Stream,Oil,Gas,Water,GI,WI
0,NaN,NaN,NaN,NaN,hrs,Sm3,Sm3,Sm3,Sm3,Sm3


In [17]:
# Test the candidate key only among rows where it is fully present.
# monthly_df itself is not modified anywhere in this section.
key_cols = ["NPDCode", "Year", "Month"]
complete_key_df = monthly_df.loc[has_complete_key]

total_rows = len(monthly_df)

combination_sizes = complete_key_df.groupby(key_cols).size()
duplicated_combinations = combination_sizes[combination_sizes > 1]
n_distinct_combinations = combination_sizes.shape[0]

dup_key_mask = complete_key_df.duplicated(subset=key_cols, keep=False)
duplicate_rows = complete_key_df.loc[complete_key_df.index[dup_key_mask]]

print(f"Total rows:                                       {total_rows}")
print(f"Rows with missing candidate-key fields:           {n_missing_key}")
print(f"Distinct NPDCode + Year + Month combinations:     {n_distinct_combinations}")
print(f"Duplicated key combinations:                      {len(duplicated_combinations)}")
print(f"Rows involved in duplicated key combinations:     {len(duplicate_rows)}")


Total rows:                                       527
Rows with missing candidate-key fields:           1
Distinct NPDCode + Year + Month combinations:     526
Duplicated key combinations:                      0
Rows involved in duplicated key combinations:     0


**Rows involved in duplicated wellbore/month combinations** (reported only — not modified):


In [18]:
duplicate_rows_display = duplicate_rows.sort_values(["NPDCode", "Year", "Month"])
duplicate_rows_display


,Wellbore name,NPDCode,Year,Month,On Stream,Oil,Gas,Water,GI,WI


In [19]:
# Distinguish exact duplicate rows (identical across every original column)
# from duplicate keys where at least one non-key value differs.
original_columns = monthly_df.columns.tolist()

exact_duplicate_mask = duplicate_rows.duplicated(subset=original_columns, keep=False)
exact_duplicate_rows = duplicate_rows.loc[duplicate_rows.index[exact_duplicate_mask]]
conflicting_duplicate_rows = duplicate_rows.loc[~duplicate_rows.index.isin(exact_duplicate_rows.index)]

print(f"Rows that are exact duplicates (identical across all columns): {len(exact_duplicate_rows)}")
print(f"Rows sharing a key but differing in at least one other column: {len(conflicting_duplicate_rows)}")


Rows that are exact duplicates (identical across all columns): 0
Rows sharing a key but differing in at least one other column: 0


In [20]:
# Final grain validation verdict.
if len(duplicated_combinations) > 0:
    grain_verdict = "NOT CONFIRMED"
elif n_missing_key > 0:
    grain_verdict = "REQUIRES REVIEW"
else:
    grain_verdict = "CONFIRMED"

print("=" * 60)
print("SECTION 5 - MONTHLY GRAIN VALIDATION SUMMARY")
print("=" * 60)
print(f"Candidate key:                             NPDCode + Year + Month")
print(f"Total rows:                                 {total_rows}")
print(f"Rows with missing candidate-key fields:     {n_missing_key}")
print(f"Distinct key combinations:                  {n_distinct_combinations}")
print(f"Duplicated key combinations:                {len(duplicated_combinations)}")
print(f"Rows involved in duplicates:                {len(duplicate_rows)}")
print(f"  - exact duplicate rows:                   {len(exact_duplicate_rows)}")
print(f"  - conflicting duplicate rows:              {len(conflicting_duplicate_rows)}")
print()
print(f"Verdict: {grain_verdict}")


SECTION 5 - MONTHLY GRAIN VALIDATION SUMMARY
Candidate key:                             NPDCode + Year + Month
Total rows:                                 527
Rows with missing candidate-key fields:     1
Distinct key combinations:                  526
Duplicated key combinations:                0
Rows involved in duplicates:                0
  - exact duplicate rows:                   0
  - conflicting duplicate rows:              0

Verdict: REQUIRES REVIEW


**Interpretation for later relational modelling:**

Among rows with a complete candidate key, `NPDCode + Year + Month` produced zero duplicated combinations (526 distinct combinations from 526 complete-key rows), so the grain holds cleanly wherever it can be tested. However, 1 of the 527 rows has all three key components missing simultaneously, and its non-key columns (`On Stream: "hrs"`, `Oil/Gas/Water/GI/WI: "Sm3"`) show it is a units/header row rather than a genuine monthly production record. Because the grain cannot be confirmed for a row that carries no key at all, the verdict is **REQUIRES REVIEW** rather than CONFIRMED. For relational modelling this means `NPDCode + Year + Month` is a strong primary-key candidate for a monthly fact table, but the source extract needs an explicit, documented decision on how that one non-data row is handled before ingestion — it should not be silently dropped by this notebook.


## 6. Identifier integrity

**Objective:** test the integrity of the daily worksheet's wellbore identifiers — `WELL_BORE_CODE`, `NPD_WELL_BORE_CODE`, `NPD_WELL_BORE_NAME` — and determine which one is the best candidate for relational joins.

This section is scoped to the Daily Production Data worksheet only. Comparing identifiers *across* the daily and monthly worksheets is Section 7. Nothing is renamed, standardized, or corrected here — only tested and reported.


In [21]:
# --- TEMPORARY LOADING FOR SECTION 6 ---
# Section 1 (Load source data) has not been implemented yet in this notebook.
# This block loads only what Section 6 needs, and is clearly marked so it can
# be deleted once Section 1 provides `daily_df` for the whole notebook.
if "daily_df" not in globals():
    from pathlib import Path
    import pandas as pd

    PROJECT_ROOT = Path.cwd().parent
    WORKBOOK_PATH = PROJECT_ROOT / "data" / "raw" / "Volve production data.xlsx"
    DAILY_SHEET_NAME = "Daily Production Data"

    if not WORKBOOK_PATH.exists():
        raise FileNotFoundError(f"Source workbook not found at {WORKBOOK_PATH}")

    daily_df = pd.read_excel(WORKBOOK_PATH, sheet_name=DAILY_SHEET_NAME)
    print(f"[Section 6 temporary load] daily_df loaded: {daily_df.shape}")
else:
    print(f"Using daily_df already loaded earlier in the notebook: {daily_df.shape}")


Using daily_df already loaded earlier in the notebook: (15634, 24)


In [22]:
# Confirm the identifier columns exist before testing anything else.
identifier_cols = ["WELL_BORE_CODE", "NPD_WELL_BORE_CODE", "NPD_WELL_BORE_NAME"]
missing_columns = [c for c in identifier_cols if c not in daily_df.columns]

if missing_columns:
    raise ValueError(f"Required identifier column(s) missing from daily_df: {missing_columns}")

print(f"Identifier columns present: {identifier_cols}")


Identifier columns present: ['WELL_BORE_CODE', 'NPD_WELL_BORE_CODE', 'NPD_WELL_BORE_NAME']


In [23]:
# Check for missing wellbore identifiers in each column.
missing_counts = {c: int(daily_df[c].isna().sum()) for c in identifier_cols}
n_missing_any_identifier = int(daily_df[identifier_cols].isna().any(axis=1).sum())

for col, n in missing_counts.items():
    print(f"Missing {col}: {n}")
print(f"Rows missing at least one identifier: {n_missing_any_identifier}")


Missing WELL_BORE_CODE: 0
Missing NPD_WELL_BORE_CODE: 0
Missing NPD_WELL_BORE_NAME: 0
Rows missing at least one identifier: 0


In [24]:
# Check for whitespace/case inconsistencies. Only leading/trailing whitespace
# and case are treated as "inconsistencies" here - internal spaces in names
# such as "15/9-F-1 C" are expected and not flagged.
whitespace_case_findings = {}
for col in identifier_cols:
    values = daily_df[col].dropna().astype(str)
    has_leading_trailing_ws = int((values != values.str.strip()).sum())
    raw_nunique = int(values.nunique())
    normalized_nunique = int(values.str.strip().str.casefold().nunique())
    whitespace_case_findings[col] = {
        "leading_or_trailing_whitespace_values": has_leading_trailing_ws,
        "raw_nunique": raw_nunique,
        "stripped_casefolded_nunique": normalized_nunique,
        "possible_case_or_whitespace_collisions": raw_nunique - normalized_nunique,
    }

whitespace_case_report = pd.DataFrame(whitespace_case_findings).T
whitespace_case_report


,leading_or_trailing_whitespace_values,raw_nunique,stripped_casefolded_nunique,possible_case_or_whitespace_collisions
WELL_BORE_CODE,0,7,7,0
NPD_WELL_BORE_CODE,0,7,7,0
NPD_WELL_BORE_NAME,0,7,7,0


In [25]:
# Test whether NPD_WELL_BORE_CODE <-> NPD_WELL_BORE_NAME is a 1:1 mapping.
code_to_name_counts = daily_df.groupby("NPD_WELL_BORE_CODE")["NPD_WELL_BORE_NAME"].nunique(dropna=True)
name_to_code_counts = daily_df.groupby("NPD_WELL_BORE_NAME")["NPD_WELL_BORE_CODE"].nunique(dropna=True)

code_to_name_violations = code_to_name_counts[code_to_name_counts > 1].reset_index(name="distinct_name_count")
name_to_code_violations = name_to_code_counts[name_to_code_counts > 1].reset_index(name="distinct_code_count")

print(f"NPD_WELL_BORE_CODE values mapping to more than one NPD_WELL_BORE_NAME: {len(code_to_name_violations)}")
print(f"NPD_WELL_BORE_NAME values mapping to more than one NPD_WELL_BORE_CODE: {len(name_to_code_violations)}")


NPD_WELL_BORE_CODE values mapping to more than one NPD_WELL_BORE_NAME: 0
NPD_WELL_BORE_NAME values mapping to more than one NPD_WELL_BORE_CODE: 0


In [26]:
# Inconsistent code -> name mappings, reported only (empty if none found).
code_to_name_violations


,NPD_WELL_BORE_CODE,distinct_name_count


In [27]:
# Inconsistent name -> code mappings, reported only (empty if none found).
name_to_code_violations


,NPD_WELL_BORE_NAME,distinct_code_count


In [28]:
# Compare WELL_BORE_CODE with the NPD wellbore code - test whether it is also
# a 1:1 relationship.
wellborecode_to_npdcode_counts = daily_df.groupby("WELL_BORE_CODE")["NPD_WELL_BORE_CODE"].nunique(dropna=True)
npdcode_to_wellborecode_counts = daily_df.groupby("NPD_WELL_BORE_CODE")["WELL_BORE_CODE"].nunique(dropna=True)

wellborecode_to_npdcode_violations = wellborecode_to_npdcode_counts[wellborecode_to_npdcode_counts > 1].reset_index(name="distinct_npd_code_count")
npdcode_to_wellborecode_violations = npdcode_to_wellborecode_counts[npdcode_to_wellborecode_counts > 1].reset_index(name="distinct_well_bore_code_count")

print(f"WELL_BORE_CODE values mapping to more than one NPD_WELL_BORE_CODE: {len(wellborecode_to_npdcode_violations)}")
print(f"NPD_WELL_BORE_CODE values mapping to more than one WELL_BORE_CODE: {len(npdcode_to_wellborecode_violations)}")


WELL_BORE_CODE values mapping to more than one NPD_WELL_BORE_CODE: 0
NPD_WELL_BORE_CODE values mapping to more than one WELL_BORE_CODE: 0


**Identifier crosswalk** — every distinct combination of the three identifier columns observed in the data (reported only):


In [29]:
identifier_crosswalk = (
    daily_df[identifier_cols]
    .drop_duplicates()
    .sort_values("NPD_WELL_BORE_CODE")
    .reset_index(drop=True)
)
identifier_crosswalk


,WELL_BORE_CODE,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME
0,NO 15/9-F-14 H,5351,15/9-F-14
1,NO 15/9-F-12 H,5599,15/9-F-12
2,NO 15/9-F-4 AH,5693,15/9-F-4
3,NO 15/9-F-5 AH,5769,15/9-F-5
4,NO 15/9-F-11 H,7078,15/9-F-11
5,NO 15/9-F-15 D,7289,15/9-F-15 D
6,NO 15/9-F-1 C,7405,15/9-F-1 C


In [30]:
# Determine which identifier is the best candidate for relational joins,
# based on the checks above rather than assumed in advance.
mapping_clean = (
    len(code_to_name_violations) == 0
    and len(name_to_code_violations) == 0
    and len(wellborecode_to_npdcode_violations) == 0
    and len(npdcode_to_wellborecode_violations) == 0
)
whitespace_or_case_clean = all(
    v["leading_or_trailing_whitespace_values"] == 0 and v["possible_case_or_whitespace_collisions"] == 0
    for v in whitespace_case_findings.values()
)

print(f"All identifier mappings 1:1: {mapping_clean}")
print(f"No missing identifiers: {n_missing_any_identifier == 0}")
print(f"No whitespace/case inconsistencies: {whitespace_or_case_clean}")

if mapping_clean and n_missing_any_identifier == 0 and whitespace_or_case_clean:
    print(
        "\nAll three identifiers are complete and consistent with each other. "
        "NPD_WELL_BORE_CODE is recommended as the join key: it is numeric, "
        "already used as half of the confirmed daily grain key (Section 4), "
        "and free of the whitespace/case ambiguity a text identifier could carry. "
        "NPD_WELL_BORE_NAME and WELL_BORE_CODE remain useful as descriptive/display "
        "attributes on a wellbore dimension."
    )
else:
    print(
        "\nAt least one identifier check above did not pass cleanly - the choice of "
        "join key should not be finalized until the flagged inconsistency is reviewed."
    )


All identifier mappings 1:1: True
No missing identifiers: True
No whitespace/case inconsistencies: True

All three identifiers are complete and consistent with each other. NPD_WELL_BORE_CODE is recommended as the join key: it is numeric, already used as half of the confirmed daily grain key (Section 4), and free of the whitespace/case ambiguity a text identifier could carry. NPD_WELL_BORE_NAME and WELL_BORE_CODE remain useful as descriptive/display attributes on a wellbore dimension.


In [31]:
# Final identifier-integrity verdict.
has_mapping_violations = not mapping_clean
has_missing_identifiers = n_missing_any_identifier > 0
has_whitespace_or_case_issues = not whitespace_or_case_clean

if has_mapping_violations:
    identifier_verdict = "NOT CONFIRMED"
elif has_missing_identifiers or has_whitespace_or_case_issues:
    identifier_verdict = "REQUIRES REVIEW"
else:
    identifier_verdict = "CONFIRMED"

print("=" * 60)
print("SECTION 6 - IDENTIFIER INTEGRITY SUMMARY")
print("=" * 60)
print(f"Rows missing at least one identifier:            {n_missing_any_identifier}")
print(f"NPD_WELL_BORE_CODE -> multiple names:            {len(code_to_name_violations)}")
print(f"NPD_WELL_BORE_NAME -> multiple codes:            {len(name_to_code_violations)}")
print(f"WELL_BORE_CODE -> multiple NPD_WELL_BORE_CODE:   {len(wellborecode_to_npdcode_violations)}")
print(f"NPD_WELL_BORE_CODE -> multiple WELL_BORE_CODE:   {len(npdcode_to_wellborecode_violations)}")
print(f"Whitespace/case inconsistencies found:           {has_whitespace_or_case_issues}")
print()
print(f"Verdict: {identifier_verdict}")


SECTION 6 - IDENTIFIER INTEGRITY SUMMARY
Rows missing at least one identifier:            0
NPD_WELL_BORE_CODE -> multiple names:            0
NPD_WELL_BORE_NAME -> multiple codes:            0
WELL_BORE_CODE -> multiple NPD_WELL_BORE_CODE:   0
NPD_WELL_BORE_CODE -> multiple WELL_BORE_CODE:   0
Whitespace/case inconsistencies found:           False

Verdict: CONFIRMED


**Interpretation for later relational modelling:**

All three wellbore identifiers on the daily worksheet are complete (0 missing values) and mutually consistent: `NPD_WELL_BORE_CODE` maps 1:1 to `NPD_WELL_BORE_NAME` in both directions, `WELL_BORE_CODE` maps 1:1 to `NPD_WELL_BORE_CODE` in both directions, and no leading/trailing whitespace or case collisions were found in any of the three columns. The identifier crosswalk above confirms this holds across all 7 wellbores. Verdict: **CONFIRMED**.

For relational modelling, this means the daily worksheet's identifiers can be trusted at face value — no cleanup, deduplication, or reconciliation of wellbore identity is required before building a wellbore dimension. `NPD_WELL_BORE_CODE` is the recommended surrogate/natural join key (numeric, already validated as half of the Section 4 grain key), with `NPD_WELL_BORE_NAME` and `WELL_BORE_CODE` carried as descriptive attributes rather than join keys. This result covers the daily worksheet only — whether the monthly worksheet's `NPDCode` refers to the same identifier system is tested separately in Section 7.


## 7. Cross-sheet identifier integrity

**Objective:** test whether the daily worksheet's `NPD_WELL_BORE_CODE` and the monthly worksheet's `NPDCode` reference the same identifier system covering the same wellbore population, and whether the associated wellbore names agree across sheets.

Section 6 already confirmed identifier integrity *within* the daily worksheet. This section is the cross-sheet check foreshadowed there. Nothing is renamed, merged, or corrected here — only compared and reported.


In [32]:
# --- TEMPORARY LOADING FOR SECTION 7 ---
# Section 1 (Load source data) has not been implemented yet in this notebook.
# This block loads only what Section 7 needs (both worksheets), and is clearly
# marked so it can be deleted once Section 1 provides daily_df / monthly_df
# for the whole notebook.
from pathlib import Path
import pandas as pd

if "daily_df" not in globals() or "monthly_df" not in globals():
    PROJECT_ROOT = Path.cwd().parent
    WORKBOOK_PATH = PROJECT_ROOT / "data" / "raw" / "Volve production data.xlsx"
    if not WORKBOOK_PATH.exists():
        raise FileNotFoundError(f"Source workbook not found at {WORKBOOK_PATH}")

if "daily_df" not in globals():
    daily_df = pd.read_excel(WORKBOOK_PATH, sheet_name="Daily Production Data")
    print(f"[Section 7 temporary load] daily_df loaded: {daily_df.shape}")
else:
    print(f"Using daily_df already loaded earlier in the notebook: {daily_df.shape}")

if "monthly_df" not in globals():
    monthly_df = pd.read_excel(WORKBOOK_PATH, sheet_name="Monthly Production Data")
    print(f"[Section 7 temporary load] monthly_df loaded: {monthly_df.shape}")
else:
    print(f"Using monthly_df already loaded earlier in the notebook: {monthly_df.shape}")


Using daily_df already loaded earlier in the notebook: (15634, 24)
Using monthly_df already loaded earlier in the notebook: (527, 10)


In [33]:
# Confirm the identifier columns exist in both worksheets before comparing them.
daily_required = ["NPD_WELL_BORE_CODE", "NPD_WELL_BORE_NAME"]
monthly_required = ["NPDCode", "Wellbore name"]

missing_daily = [c for c in daily_required if c not in daily_df.columns]
missing_monthly = [c for c in monthly_required if c not in monthly_df.columns]

if missing_daily:
    raise ValueError(f"Required column(s) missing from daily_df: {missing_daily}")
if missing_monthly:
    raise ValueError(f"Required column(s) missing from monthly_df: {missing_monthly}")

print(f"Daily identifier columns present:   {daily_required}")
print(f"Monthly identifier columns present: {monthly_required}")


Daily identifier columns present:   ['NPD_WELL_BORE_CODE', 'NPD_WELL_BORE_NAME']
Monthly identifier columns present: ['NPDCode', 'Wellbore name']


In [34]:
# NPDCode in the monthly sheet loads as float64 (mixed with NaN from the stray
# non-data row identified in Section 5), so cast both sides to a common
# comparable type before comparing. Only complete (non-null) codes are compared.
print(f"daily_df['NPD_WELL_BORE_CODE'] dtype: {daily_df['NPD_WELL_BORE_CODE'].dtype}")
print(f"monthly_df['NPDCode'] dtype:          {monthly_df['NPDCode'].dtype}")

daily_codes = set(daily_df["NPD_WELL_BORE_CODE"].dropna().astype(int))
monthly_codes = set(monthly_df["NPDCode"].dropna().astype(int))

common_codes = daily_codes & monthly_codes
only_in_daily = daily_codes - monthly_codes
only_in_monthly = monthly_codes - daily_codes

print(f"\nDistinct wellbore codes in daily sheet:   {len(daily_codes)}")
print(f"Distinct wellbore codes in monthly sheet: {len(monthly_codes)}")
print(f"Common to both sheets:                    {len(common_codes)}")
print(f"Only in daily sheet:                      {sorted(only_in_daily)}")
print(f"Only in monthly sheet:                    {sorted(only_in_monthly)}")


daily_df['NPD_WELL_BORE_CODE'] dtype: int64
monthly_df['NPDCode'] dtype:          float64

Distinct wellbore codes in daily sheet:   7
Distinct wellbore codes in monthly sheet: 7
Common to both sheets:                    7
Only in daily sheet:                      []
Only in monthly sheet:                    []


In [35]:
# Whether the two columns appear to reference the same identifier system.
same_identifier_system = len(only_in_daily) == 0 and len(only_in_monthly) == 0 and len(common_codes) > 0
print(f"NPD_WELL_BORE_CODE and NPDCode appear to reference the same identifier system: {same_identifier_system}")


NPD_WELL_BORE_CODE and NPDCode appear to reference the same identifier system: True


**Naming comparison** — for each wellbore code common to both sheets, the name(s) used in the daily worksheet vs. the monthly worksheet:


In [36]:
# For each common code, compare the name(s) associated with it in each sheet.
daily_code_to_names = daily_df.groupby("NPD_WELL_BORE_CODE")["NPD_WELL_BORE_NAME"].unique()

monthly_with_int_code = monthly_df.dropna(subset=["NPDCode"]).copy()
monthly_with_int_code["NPDCode_int"] = monthly_with_int_code["NPDCode"].astype(int)
monthly_code_to_names = monthly_with_int_code.groupby("NPDCode_int")["Wellbore name"].unique()

naming_rows = []
for code_value in sorted(common_codes):
    daily_names = set(daily_code_to_names.get(code_value, []))
    monthly_names = set(monthly_code_to_names.get(code_value, []))
    naming_rows.append(
        {
            "npd_code": code_value,
            "daily_names": sorted(daily_names),
            "monthly_names": sorted(monthly_names),
            "names_match": daily_names == monthly_names,
        }
    )

naming_comparison = pd.DataFrame(naming_rows)
naming_comparison


,npd_code,daily_names,monthly_names,names_match
0,5351,[15/9-F-14],[15/9-F-14],True
1,5599,[15/9-F-12],[15/9-F-12],True
2,5693,[15/9-F-4],[15/9-F-4],True
3,5769,[15/9-F-5],[15/9-F-5],True
4,7078,[15/9-F-11],[15/9-F-11],True
5,7289,[15/9-F-15 D],[15/9-F-15 D],True
6,7405,[15/9-F-1 C],[15/9-F-1 C],True


In [37]:
naming_mismatches = naming_comparison.loc[~naming_comparison["names_match"]]
print(f"Wellbores with a naming mismatch between daily and monthly sheets: {len(naming_mismatches)}")


Wellbores with a naming mismatch between daily and monthly sheets: 0


In [38]:
# Final cross-sheet identifier-integrity verdict.
has_only_in_daily = len(only_in_daily) > 0
has_only_in_monthly = len(only_in_monthly) > 0
has_naming_mismatches = len(naming_mismatches) > 0

if has_only_in_daily or has_only_in_monthly:
    cross_sheet_verdict = "NOT CONFIRMED"
elif has_naming_mismatches:
    cross_sheet_verdict = "REQUIRES REVIEW"
else:
    cross_sheet_verdict = "CONFIRMED"

print("=" * 60)
print("SECTION 7 - CROSS-SHEET IDENTIFIER INTEGRITY SUMMARY")
print("=" * 60)
print(f"Distinct wellbore codes - daily:      {len(daily_codes)}")
print(f"Distinct wellbore codes - monthly:    {len(monthly_codes)}")
print(f"Common to both sheets:                {len(common_codes)}")
print(f"Only in daily sheet:                  {len(only_in_daily)}")
print(f"Only in monthly sheet:                {len(only_in_monthly)}")
print(f"Naming mismatches (common codes):     {len(naming_mismatches)}")
print()
print(f"Verdict: {cross_sheet_verdict}")


SECTION 7 - CROSS-SHEET IDENTIFIER INTEGRITY SUMMARY
Distinct wellbore codes - daily:      7
Distinct wellbore codes - monthly:    7
Common to both sheets:                7
Only in daily sheet:                  0
Only in monthly sheet:                0
Naming mismatches (common codes):     0

Verdict: CONFIRMED


**Interpretation for later relational modelling:**

`NPD_WELL_BORE_CODE` (daily) and `NPDCode` (monthly) reference the same 7 wellbores with no codes unique to either sheet, and for every common code the associated wellbore name is identical between sheets. `NPDCode` loads as `float64` rather than `int64` — this is purely a side effect of the stray non-data row in the monthly sheet identified in Section 5 (a NaN in the column forces pandas to use float), not evidence of a different identifier system. Verdict: **CONFIRMED**.

For relational modelling, this means `NPD_WELL_BORE_CODE`/`NPDCode` can be treated as the same natural key across both worksheets and used as a shared foreign key into a single wellbore dimension — daily and monthly facts do not need separate or reconciled identifier systems. The wellbore name should still be sourced from one worksheet only (the daily worksheet, per Section 6) to avoid depending on two independently-maintained copies of the same descriptive attribute, even though they agree today.


## 8. Field and facility consistency

**Objective:** test whether `NPD_FIELD_CODE ↔ NPD_FIELD_NAME` and `NPD_FACILITY_CODE ↔ NPD_FACILITY_NAME` are each stable 1:1 mappings, whether every wellbore stays associated with one field and one facility over time, and whether the values match what is expected for Volve (field `3420717 ↔ VOLVE`, facility `369304 ↔ MÆRSK INSPIRER`) — verified here rather than assumed.

This section only reports cardinalities. Whether field/facility should become separate dimension tables is a database-design decision for Section 24, made after seeing the actual results below — not decided here.


In [39]:
# --- TEMPORARY LOADING FOR SECTION 8 ---
# Section 1 (Load source data) has not been implemented yet in this notebook.
# This block loads only what Section 8 needs, and is clearly marked so it can
# be deleted once Section 1 provides `daily_df` for the whole notebook.
if "daily_df" not in globals():
    from pathlib import Path
    import pandas as pd

    PROJECT_ROOT = Path.cwd().parent
    WORKBOOK_PATH = PROJECT_ROOT / "data" / "raw" / "Volve production data.xlsx"
    DAILY_SHEET_NAME = "Daily Production Data"

    if not WORKBOOK_PATH.exists():
        raise FileNotFoundError(f"Source workbook not found at {WORKBOOK_PATH}")

    daily_df = pd.read_excel(WORKBOOK_PATH, sheet_name=DAILY_SHEET_NAME)
    print(f"[Section 8 temporary load] daily_df loaded: {daily_df.shape}")
else:
    print(f"Using daily_df already loaded earlier in the notebook: {daily_df.shape}")


Using daily_df already loaded earlier in the notebook: (15634, 24)


In [40]:
# Confirm the field/facility columns exist before testing anything else.
field_facility_cols = ["NPD_FIELD_CODE", "NPD_FIELD_NAME", "NPD_FACILITY_CODE", "NPD_FACILITY_NAME"]
missing_columns = [c for c in field_facility_cols if c not in daily_df.columns]

if missing_columns:
    raise ValueError(f"Required column(s) missing from daily_df: {missing_columns}")

print(f"Field/facility columns present: {field_facility_cols}")


Field/facility columns present: ['NPD_FIELD_CODE', 'NPD_FIELD_NAME', 'NPD_FACILITY_CODE', 'NPD_FACILITY_NAME']


In [41]:
# Check for missing values in each field/facility column.
missing_counts = {c: int(daily_df[c].isna().sum()) for c in field_facility_cols}
for col, n in missing_counts.items():
    print(f"Missing {col}: {n}")


Missing NPD_FIELD_CODE: 0
Missing NPD_FIELD_NAME: 0
Missing NPD_FACILITY_CODE: 0
Missing NPD_FACILITY_NAME: 0


In [42]:
# Check the two name columns for leading/trailing whitespace or case
# inconsistencies (codes are numeric, so only the name columns apply here).
whitespace_case_findings = {}
for col in ["NPD_FIELD_NAME", "NPD_FACILITY_NAME"]:
    values = daily_df[col].dropna().astype(str)
    has_leading_trailing_ws = int((values != values.str.strip()).sum())
    raw_nunique = int(values.nunique())
    normalized_nunique = int(values.str.strip().str.casefold().nunique())
    whitespace_case_findings[col] = {
        "leading_or_trailing_whitespace_values": has_leading_trailing_ws,
        "raw_nunique": raw_nunique,
        "stripped_casefolded_nunique": normalized_nunique,
        "possible_case_or_whitespace_collisions": raw_nunique - normalized_nunique,
    }

whitespace_case_report = pd.DataFrame(whitespace_case_findings).T
whitespace_case_report


,leading_or_trailing_whitespace_values,raw_nunique,stripped_casefolded_nunique,possible_case_or_whitespace_collisions
NPD_FIELD_NAME,0,1,1,0
NPD_FACILITY_NAME,0,1,1,0


In [43]:
# Test whether NPD_FIELD_CODE <-> NPD_FIELD_NAME is a 1:1 mapping.
field_code_to_name = daily_df.groupby("NPD_FIELD_CODE")["NPD_FIELD_NAME"].nunique(dropna=True)
field_name_to_code = daily_df.groupby("NPD_FIELD_NAME")["NPD_FIELD_CODE"].nunique(dropna=True)

field_code_to_name_violations = field_code_to_name[field_code_to_name > 1].reset_index(name="distinct_name_count")
field_name_to_code_violations = field_name_to_code[field_name_to_code > 1].reset_index(name="distinct_code_count")

print(f"NPD_FIELD_CODE values mapping to more than one NPD_FIELD_NAME: {len(field_code_to_name_violations)}")
print(f"NPD_FIELD_NAME values mapping to more than one NPD_FIELD_CODE: {len(field_name_to_code_violations)}")


NPD_FIELD_CODE values mapping to more than one NPD_FIELD_NAME: 0
NPD_FIELD_NAME values mapping to more than one NPD_FIELD_CODE: 0


In [44]:
# Test whether NPD_FACILITY_CODE <-> NPD_FACILITY_NAME is a 1:1 mapping.
facility_code_to_name = daily_df.groupby("NPD_FACILITY_CODE")["NPD_FACILITY_NAME"].nunique(dropna=True)
facility_name_to_code = daily_df.groupby("NPD_FACILITY_NAME")["NPD_FACILITY_CODE"].nunique(dropna=True)

facility_code_to_name_violations = facility_code_to_name[facility_code_to_name > 1].reset_index(name="distinct_name_count")
facility_name_to_code_violations = facility_name_to_code[facility_name_to_code > 1].reset_index(name="distinct_code_count")

print(f"NPD_FACILITY_CODE values mapping to more than one NPD_FACILITY_NAME: {len(facility_code_to_name_violations)}")
print(f"NPD_FACILITY_NAME values mapping to more than one NPD_FACILITY_CODE: {len(facility_name_to_code_violations)}")


NPD_FACILITY_CODE values mapping to more than one NPD_FACILITY_NAME: 0
NPD_FACILITY_NAME values mapping to more than one NPD_FACILITY_CODE: 0


**Per-wellbore stability.** Grouping by wellbore across every row (which spans that wellbore's full date range) directly answers both "does each wellbore stay associated with one field/facility" and "does field/facility change through time" — if a wellbore shows more than one distinct value here, it changed at some point in its history.


In [45]:
wellbore_field_counts = daily_df.groupby("NPD_WELL_BORE_CODE")["NPD_FIELD_CODE"].nunique(dropna=True)
wellbore_facility_counts = daily_df.groupby("NPD_WELL_BORE_CODE")["NPD_FACILITY_CODE"].nunique(dropna=True)

wellbores_with_multiple_fields = wellbore_field_counts[wellbore_field_counts > 1].reset_index(name="distinct_field_count")
wellbores_with_multiple_facilities = wellbore_facility_counts[wellbore_facility_counts > 1].reset_index(name="distinct_facility_count")

print(f"Wellbores associated with more than one field over time:    {len(wellbores_with_multiple_fields)}")
print(f"Wellbores associated with more than one facility over time: {len(wellbores_with_multiple_facilities)}")


Wellbores associated with more than one field over time:    0
Wellbores associated with more than one facility over time: 0


**Actual field and facility values found in the dataset** (checked, not assumed):


In [46]:
distinct_fields = (
    daily_df[["NPD_FIELD_CODE", "NPD_FIELD_NAME"]]
    .drop_duplicates()
    .sort_values("NPD_FIELD_CODE")
    .reset_index(drop=True)
)
print(f"Distinct field code/name combinations: {len(distinct_fields)}")
distinct_fields


Distinct field code/name combinations: 1


,NPD_FIELD_CODE,NPD_FIELD_NAME
0,3420717,VOLVE


In [47]:
distinct_facilities = (
    daily_df[["NPD_FACILITY_CODE", "NPD_FACILITY_NAME"]]
    .drop_duplicates()
    .sort_values("NPD_FACILITY_CODE")
    .reset_index(drop=True)
)
print(f"Distinct facility code/name combinations: {len(distinct_facilities)}")
distinct_facilities


Distinct facility code/name combinations: 1


,NPD_FACILITY_CODE,NPD_FACILITY_NAME
0,369304,MÆRSK INSPIRER


In [48]:
expected_field = (3420717, "VOLVE")
expected_facility = (369304, "MÆRSK INSPIRER")

field_matches_expected = len(distinct_fields) == 1 and tuple(distinct_fields.iloc[0]) == expected_field
facility_matches_expected = len(distinct_facilities) == 1 and tuple(distinct_facilities.iloc[0]) == expected_facility

print(f"Dataset contains exactly the expected single field {expected_field}:       {field_matches_expected}")
print(f"Dataset contains exactly the expected single facility {expected_facility}: {facility_matches_expected}")


Dataset contains exactly the expected single field (3420717, 'VOLVE'):       True
Dataset contains exactly the expected single facility (369304, 'MÆRSK INSPIRER'): True


In [49]:
# Final field/facility consistency verdict.
has_missing = any(n > 0 for n in missing_counts.values())
has_mapping_violations = (
    len(field_code_to_name_violations) > 0
    or len(field_name_to_code_violations) > 0
    or len(facility_code_to_name_violations) > 0
    or len(facility_name_to_code_violations) > 0
)
has_wellbore_instability = (
    len(wellbores_with_multiple_fields) > 0 or len(wellbores_with_multiple_facilities) > 0
)
has_whitespace_or_case_issues = any(
    v["leading_or_trailing_whitespace_values"] > 0 or v["possible_case_or_whitespace_collisions"] > 0
    for v in whitespace_case_findings.values()
)

if has_mapping_violations or has_wellbore_instability:
    field_facility_verdict = "NOT CONFIRMED"
elif has_missing or has_whitespace_or_case_issues:
    field_facility_verdict = "REQUIRES REVIEW"
else:
    field_facility_verdict = "CONFIRMED"

print("=" * 60)
print("SECTION 8 - FIELD AND FACILITY CONSISTENCY SUMMARY")
print("=" * 60)
print(f"Distinct fields:                                 {len(distinct_fields)}")
print(f"Distinct facilities:                              {len(distinct_facilities)}")
print(f"NPD_FIELD_CODE -> multiple names:                {len(field_code_to_name_violations)}")
print(f"NPD_FIELD_NAME -> multiple codes:                {len(field_name_to_code_violations)}")
print(f"NPD_FACILITY_CODE -> multiple names:             {len(facility_code_to_name_violations)}")
print(f"NPD_FACILITY_NAME -> multiple codes:             {len(facility_name_to_code_violations)}")
print(f"Wellbores with >1 field over time:               {len(wellbores_with_multiple_fields)}")
print(f"Wellbores with >1 facility over time:             {len(wellbores_with_multiple_facilities)}")
print(f"Missing values found:                            {has_missing}")
print(f"Whitespace/case inconsistencies found:           {has_whitespace_or_case_issues}")
print(f"Matches expected Volve field/facility values:    {field_matches_expected and facility_matches_expected}")
print()
print(f"Verdict: {field_facility_verdict}")


SECTION 8 - FIELD AND FACILITY CONSISTENCY SUMMARY
Distinct fields:                                 1
Distinct facilities:                              1
NPD_FIELD_CODE -> multiple names:                0
NPD_FIELD_NAME -> multiple codes:                0
NPD_FACILITY_CODE -> multiple names:             0
NPD_FACILITY_NAME -> multiple codes:             0
Wellbores with >1 field over time:               0
Wellbores with >1 facility over time:             0
Missing values found:                            False
Whitespace/case inconsistencies found:           False
Matches expected Volve field/facility values:    True

Verdict: CONFIRMED


**Interpretation for later relational modelling:**

Both mappings are clean 1:1 relationships with zero missing values and zero whitespace/case issues, and every one of the 7 wellbores stays associated with the same field and the same facility across its entire recorded history (no per-wellbore instability, which also answers the "does it change through time" question — a wellbore group spans that wellbore's full date range, so any change would have shown up as a distinct-value count greater than one). The dataset in fact contains exactly one field and one facility overall, matching the expected Volve / Mærsk Inspirer values precisely. Verdict: **CONFIRMED**.

**Database-design implication (not decided here):** with cardinality this low — one field, one facility, for the entire dataset and every wellbore — separate `field` and `facility` dimension tables would be structurally correct but would each hold exactly one row for this dataset. Two reasonable paths follow from that: (a) fold field/facility into the wellbore dimension as plain attributes for v1, since a one-row dimension table adds join overhead without adding modelling power here, or (b) keep them as separate dimension tables anyway, because this project is explicitly intended as a reusable template for other industrial datasets where a facility or field genuinely has multiple wellbores and multiple values, and it will need to work correctly for that case, not just for Volve's current low cardinality. This is a call to make in Section 24 with the full set of Section 8 results as evidence, not before.


## 9. Temporal validity

**Objective:** test the temporal structure of the daily worksheet, keeping three distinct concepts separate throughout this section:

- **Invalid date** — a `DATEPRD` value that is NULL or cannot be parsed as a date.
- **Missing calendar date** — a calendar day within a wellbore's recorded span for which no row exists at all.
- **Well not producing/reporting on a date** — a row *does* exist for that date, but the well was inactive (e.g. `ON_STREAM_HRS = 0`) or a measurement was NULL.

A wellbore can legitimately have gaps in its daily record (shut-ins, interventions, workovers) — a missing calendar date is **not** automatically a data-quality failure. This section only establishes facts; it does not classify gaps as errors.

Five things are tested: date integrity, field-level temporal coverage, wellbore-level coverage, gap analysis, and a known field-life boundary check (flagged as REVIEW, never auto-corrected). A final structural distinction — record-present-but-inactive vs. no-record-at-all — is also established here, because it is what Section 10 (on-stream-hours validation) builds on next.


In [50]:
# --- TEMPORARY LOADING FOR SECTION 9 ---
# Section 1 (Load source data) has not been implemented yet in this notebook.
# This block loads only what Section 9 needs, and is clearly marked so it can
# be deleted once Section 1 provides `daily_df` for the whole notebook.
if "daily_df" not in globals():
    from pathlib import Path
    import pandas as pd

    PROJECT_ROOT = Path.cwd().parent
    WORKBOOK_PATH = PROJECT_ROOT / "data" / "raw" / "Volve production data.xlsx"
    DAILY_SHEET_NAME = "Daily Production Data"

    if not WORKBOOK_PATH.exists():
        raise FileNotFoundError(f"Source workbook not found at {WORKBOOK_PATH}")

    daily_df = pd.read_excel(WORKBOOK_PATH, sheet_name=DAILY_SHEET_NAME)
    print(f"[Section 9 temporary load] daily_df loaded: {daily_df.shape}")
else:
    print(f"Using daily_df already loaded earlier in the notebook: {daily_df.shape}")


Using daily_df already loaded earlier in the notebook: (15634, 24)


In [51]:
# Confirm the columns this section needs before testing anything else.
required_columns = ["DATEPRD", "NPD_WELL_BORE_CODE", "NPD_WELL_BORE_NAME", "ON_STREAM_HRS"]
missing_columns = [c for c in required_columns if c not in daily_df.columns]

if missing_columns:
    raise ValueError(f"Required column(s) missing from daily_df: {missing_columns}")

print(f"Required columns present: {required_columns}")


Required columns present: ['DATEPRD', 'NPD_WELL_BORE_CODE', 'NPD_WELL_BORE_NAME', 'ON_STREAM_HRS']


### 9.1 Date integrity

`DATEPRD` is parsed into a separate Series (`dateprd_parsed`) for the rest of this section. The original `daily_df["DATEPRD"]` column is never overwritten.


In [52]:
dateprd_parsed = pd.to_datetime(daily_df["DATEPRD"], errors="coerce")

total_rows = len(daily_df)
null_dates_mask = daily_df["DATEPRD"].isna()
unparseable_mask = dateprd_parsed.isna() & daily_df["DATEPRD"].notna()
successfully_parsed_mask = dateprd_parsed.notna()

n_null_dates = int(null_dates_mask.sum())
n_unparseable = int(unparseable_mask.sum())
n_successfully_parsed = int(successfully_parsed_mask.sum())

print(f"Total rows:                  {total_rows}")
print(f"Successfully parsed dates:   {n_successfully_parsed}")
print(f"NULL dates:                  {n_null_dates}")
print(f"Unparseable non-NULL dates:  {n_unparseable}")
print(f"Minimum date:                {dateprd_parsed.min()}")
print(f"Maximum date:                {dateprd_parsed.max()}")


Total rows:                  15634
Successfully parsed dates:   15634
NULL dates:                  0
Unparseable non-NULL dates:  0
Minimum date:                2007-09-01 00:00:00
Maximum date:                2016-12-01 00:00:00


In [53]:
# Working copy with the parsed date attached as a new column - daily_df itself
# is not modified. Used for every remaining check in this section.
daily_with_dates = daily_df.copy()
daily_with_dates["DATEPRD_parsed"] = dateprd_parsed


### 9.2 Field-level temporal coverage

"Dates represented anywhere" (at least one wellbore reported) is deliberately kept separate from "calendar days with no records from any wellbore" — the two describe opposite things and should not be conflated.


In [54]:
field_first_date = dateprd_parsed.min()
field_last_date = dateprd_parsed.max()
field_calendar_span_days = (field_last_date - field_first_date).days + 1

dates_represented_anywhere = dateprd_parsed.dropna().nunique()
dates_with_no_records_anywhere = field_calendar_span_days - dates_represented_anywhere

print(f"First recorded date (any wellbore):          {field_first_date.date()}")
print(f"Last recorded date (any wellbore):            {field_last_date.date()}")
print(f"Calendar days between first and last:         {field_calendar_span_days}")
print(f"Distinct dates represented anywhere:          {dates_represented_anywhere}")
print(f"Calendar days with no records from any well:  {dates_with_no_records_anywhere}")


First recorded date (any wellbore):          2007-09-01
Last recorded date (any wellbore):            2016-12-01
Calendar days between first and last:         3380
Distinct dates represented anywhere:          3327
Calendar days with no records from any well:  53


### 9.3 Wellbore-level coverage

Per-wellbore first/last date, calendar span, recorded days, and the calendar days within that span with no row for that wellbore. These are called **unrecorded calendar days** deliberately, not "missing observations" — their operational meaning (shut-in, pre-completion, post-abandonment, genuine gap) is not yet known.


In [55]:
wellbore_coverage_rows = []
for code_value, group in daily_with_dates.groupby("NPD_WELL_BORE_CODE"):
    name = group["NPD_WELL_BORE_NAME"].iloc[0]
    dates = group["DATEPRD_parsed"].dropna().drop_duplicates().sort_values()
    first_date = dates.iloc[0]
    last_date = dates.iloc[-1]
    calendar_span_days = (last_date - first_date).days + 1
    recorded_days = len(dates)
    unrecorded_calendar_days = calendar_span_days - recorded_days
    coverage_pct = round(recorded_days / calendar_span_days * 100, 2)
    wellbore_coverage_rows.append(
        {
            "npd_code": code_value,
            "wellbore_name": name,
            "first_date": first_date.date(),
            "last_date": last_date.date(),
            "calendar_span_days": calendar_span_days,
            "recorded_days": recorded_days,
            "unrecorded_calendar_days": unrecorded_calendar_days,
            "coverage_pct": coverage_pct,
        }
    )

wellbore_coverage = pd.DataFrame(wellbore_coverage_rows).sort_values("npd_code").reset_index(drop=True)
wellbore_coverage


,npd_code,wellbore_name,first_date,last_date,calendar_span_days,recorded_days,unrecorded_calendar_days,coverage_pct
0,5351,15/9-F-14,2008-02-12,2016-09-17,3141,3056,85,97.29
1,5599,15/9-F-12,2008-02-12,2016-09-17,3141,3056,85,97.29
2,5693,15/9-F-4,2007-09-01,2016-12-01,3380,3327,53,98.43
3,5769,15/9-F-5,2007-09-01,2016-09-18,3306,3306,0,100.00
4,7078,15/9-F-11,2013-07-08,2016-09-17,1168,1165,3,99.74
5,7289,15/9-F-15 D,2014-01-12,2016-09-17,980,978,2,99.80
6,7405,15/9-F-1 C,2014-04-07,2016-04-21,746,746,0,100.00


### 9.4 Gap analysis

For each wellbore, records are sorted chronologically and the difference between consecutive dates is calculated. A difference of 1 day is normal (no gap). Differences greater than 1 day indicate at least one unrecorded calendar day in between. These are reported only — not classified as shutdowns, interventions, or errors.


In [56]:
gap_summary_rows = []
larger_gap_rows = []
LARGER_GAP_THRESHOLD_DAYS = 7  # threshold for the review table below

for code_value, group in daily_with_dates.groupby("NPD_WELL_BORE_CODE"):
    name = group["NPD_WELL_BORE_NAME"].iloc[0]
    dates = group["DATEPRD_parsed"].dropna().drop_duplicates().sort_values().reset_index(drop=True)
    gap_days_series = dates.diff().dt.days.dropna()

    max_gap = int(gap_days_series.max()) if len(gap_days_series) else 0
    gaps_gt_1 = int((gap_days_series > 1).sum())
    gaps_gt_7 = int((gap_days_series > 7).sum())
    gaps_gt_30 = int((gap_days_series > 30).sum())

    gap_summary_rows.append(
        {
            "npd_code": code_value,
            "wellbore_name": name,
            "max_gap_days": max_gap,
            "gaps_gt_1_day": gaps_gt_1,
            "gaps_gt_7_days": gaps_gt_7,
            "gaps_gt_30_days": gaps_gt_30,
        }
    )

    for i in range(1, len(dates)):
        gap_days = (dates.iloc[i] - dates.iloc[i - 1]).days
        if gap_days > LARGER_GAP_THRESHOLD_DAYS:
            larger_gap_rows.append(
                {
                    "npd_code": code_value,
                    "wellbore_name": name,
                    "previous_record_date": dates.iloc[i - 1].date(),
                    "next_record_date": dates.iloc[i].date(),
                    "gap_days": gap_days,
                }
            )

gap_summary = pd.DataFrame(gap_summary_rows).sort_values("npd_code").reset_index(drop=True)
gap_summary


,npd_code,wellbore_name,max_gap_days,gaps_gt_1_day,gaps_gt_7_days,gaps_gt_30_days
0,5351,15/9-F-14,12,46,1,0
1,5599,15/9-F-12,12,46,1,0
2,5693,15/9-F-4,30,2,2,0
3,5769,15/9-F-5,1,0,0,0
4,7078,15/9-F-11,2,3,0,0
5,7289,15/9-F-15 D,2,2,0,0
6,7405,15/9-F-1 C,1,0,0,0


In [57]:
gap_review_table = (
    pd.DataFrame(larger_gap_rows)
    if larger_gap_rows
    else pd.DataFrame(columns=["npd_code", "wellbore_name", "previous_record_date", "next_record_date", "gap_days"])
)
gap_review_table = gap_review_table.sort_values("gap_days", ascending=False).reset_index(drop=True)

print(f"Gaps larger than {LARGER_GAP_THRESHOLD_DAYS} days across all wellbores: {len(gap_review_table)}")
gap_review_table


Gaps larger than 7 days across all wellbores: 4


,npd_code,wellbore_name,previous_record_date,next_record_date,gap_days
0,5693,15/9-F-4,2016-11-01,2016-12-01,30
1,5693,15/9-F-4,2016-10-07,2016-11-01,25
2,5351,15/9-F-14,2012-01-02,2012-01-14,12
3,5599,15/9-F-12,2012-01-02,2012-01-14,12


### 9.5 Known field-life boundary check

Volve production is documented as running from roughly 2008 to 2016. This is a **documented assumption**, used only to flag rows for review — never to delete or reclassify anything.


In [58]:
# Documented assumption, not derived from the data itself.
EXPECTED_FIELD_START = pd.Timestamp("2008-01-01")
EXPECTED_FIELD_END = pd.Timestamp("2016-12-31")

before_field_life_mask = dateprd_parsed.notna() & (dateprd_parsed < EXPECTED_FIELD_START)
after_field_life_mask = dateprd_parsed.notna() & (dateprd_parsed > EXPECTED_FIELD_END)

n_before_field_life = int(before_field_life_mask.sum())
n_after_field_life = int(after_field_life_mask.sum())

print(f"Expected field operating period (documented assumption): {EXPECTED_FIELD_START.date()} to {EXPECTED_FIELD_END.date()}")
print(f"Rows before expected field life (REVIEW): {n_before_field_life}")
print(f"Rows after expected field life (REVIEW):  {n_after_field_life}")


Expected field operating period (documented assumption): 2008-01-01 to 2016-12-31
Rows before expected field life (REVIEW): 244
Rows after expected field life (REVIEW):  0


In [59]:
outside_field_life_rows = (
    daily_with_dates.loc[
        before_field_life_mask | after_field_life_mask,
        ["DATEPRD_parsed", "NPD_WELL_BORE_CODE", "NPD_WELL_BORE_NAME", "FLOW_KIND", "WELL_TYPE", "ON_STREAM_HRS"],
    ]
    .sort_values("DATEPRD_parsed")
    .reset_index(drop=True)
)
outside_field_life_rows


,DATEPRD_parsed,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,FLOW_KIND,WELL_TYPE,ON_STREAM_HRS
0,2007-09-01,5693,15/9-F-4,injection,WI,NaN
1,2007-09-01,5769,15/9-F-5,injection,WI,NaN
2,2007-09-02,5693,15/9-F-4,injection,WI,NaN
3,2007-09-02,5769,15/9-F-5,injection,WI,NaN
4,2007-09-03,5693,15/9-F-4,injection,WI,NaN
...,...,...,...,...,...,...
239,2007-12-29,5693,15/9-F-4,injection,WI,NaN
240,2007-12-30,5769,15/9-F-5,injection,WI,NaN
241,2007-12-30,5693,15/9-F-4,injection,WI,NaN
242,2007-12-31,5693,15/9-F-4,injection,WI,NaN


### 9.6 Record presence vs. reporting activity

This is the distinction Section 10 depends on. For each wellbore's full calendar span, every day is classified as exactly one of:

- **no_record** — no row exists for that wellbore/date at all (a temporal gap, already counted in 9.3/9.4).
- **record_present_active** — a row exists and `ON_STREAM_HRS > 0`.
- **record_present_inactive** — a row exists and `ON_STREAM_HRS == 0`.
- **record_present_hours_missing** — a row exists but `ON_STREAM_HRS` itself is NULL.

This only classifies; it does not judge whether any of these categories is a problem. A well can be genuinely shut in with a real `ON_STREAM_HRS = 0` record — that is not the same situation as no record existing at all.


In [60]:
presence_rows = []
for code_value, group in daily_with_dates.groupby("NPD_WELL_BORE_CODE"):
    name = group["NPD_WELL_BORE_NAME"].iloc[0]
    recorded = (
        group.dropna(subset=["DATEPRD_parsed"])
        .drop_duplicates(subset=["DATEPRD_parsed"])
        .set_index("DATEPRD_parsed")
        .sort_index()
    )
    full_range = pd.date_range(recorded.index.min(), recorded.index.max(), freq="D")

    n_no_record = len(full_range) - len(recorded)
    hours = recorded["ON_STREAM_HRS"]
    n_active = int((hours > 0).sum())
    n_inactive = int((hours == 0).sum())
    n_hours_missing = int(hours.isna().sum())

    presence_rows.append(
        {
            "npd_code": code_value,
            "wellbore_name": name,
            "no_record_days": n_no_record,
            "record_present_active_days": n_active,
            "record_present_inactive_days": n_inactive,
            "record_present_hours_missing_days": n_hours_missing,
        }
    )

presence_summary = pd.DataFrame(presence_rows).sort_values("npd_code").reset_index(drop=True)
presence_summary


,npd_code,wellbore_name,no_record_days,record_present_active_days,record_present_inactive_days,record_present_hours_missing_days
0,5351,15/9-F-14,85,2724,332,0
1,5599,15/9-F-12,85,2838,218,0
2,5693,15/9-F-4,53,2830,345,152
3,5769,15/9-F-5,0,2676,497,133
4,7078,15/9-F-11,3,1122,43,0
5,7289,15/9-F-15 D,2,769,209,0
6,7405,15/9-F-1 C,0,438,308,0


In [61]:
# Final temporal-validity verdict. Temporal data quality is less binary than
# the earlier structural sections, so this uses PASS / PASS WITH REVIEW / FAIL
# rather than CONFIRMED / NOT CONFIRMED / REQUIRES REVIEW.
date_parsing_pass = (n_null_dates == 0) and (n_unparseable == 0)
wellbores_with_gaps = int((gap_summary["gaps_gt_1_day"] > 0).sum())
temporal_gaps_detected = wellbores_with_gaps > 0
outside_field_life_count = n_before_field_life + n_after_field_life
overall_max_gap = int(gap_summary["max_gap_days"].max())
n_wellbores = daily_df["NPD_WELL_BORE_CODE"].nunique()

if not date_parsing_pass:
    temporal_verdict = "FAIL"
elif temporal_gaps_detected or outside_field_life_count > 0:
    temporal_verdict = "PASS WITH REVIEW"
else:
    temporal_verdict = "PASS"

print("=" * 60)
print("TEMPORAL VALIDITY")
print("=" * 60)
print(f"Date parsing:                 {'PASS' if date_parsing_pass else 'FAIL'}")
print(f"Missing DATEPRD:              {n_null_dates}")
print(f"Unparseable DATEPRD:          {n_unparseable}")
print(f"Observed range:               {dateprd_parsed.min().date()} to {dateprd_parsed.max().date()}")
print(f"Outside expected field life:  {outside_field_life_count}")
print()
print(f"Temporal gaps detected:       {'Yes' if temporal_gaps_detected else 'No'}")
print(f"Wellbores affected:           {wellbores_with_gaps} / {n_wellbores}")
print(f"Largest gap:                  {overall_max_gap} days")
print()
print(f"VERDICT: {temporal_verdict}")


TEMPORAL VALIDITY
Date parsing:                 PASS
Missing DATEPRD:              0
Unparseable DATEPRD:          0
Observed range:               2007-09-01 to 2016-12-01
Outside expected field life:  244

Temporal gaps detected:       Yes
Wellbores affected:           5 / 7
Largest gap:                  30 days

VERDICT: PASS WITH REVIEW


**Interpretation for later relational modelling:**

Date parsing is fully clean (0 missing, 0 unparseable across all 15,634 rows) — the field itself is valid. Everything else in this section is a REVIEW-class finding, not a FAIL:

- **Field-level:** the field spans 3,380 calendar days (2007-09-01 to 2016-12-01), with 3,327 distinct dates represented by at least one wellbore and 53 calendar days where *no* wellbore reported anything at all.
- **Wellbore-level coverage** ranges from 97.29% to 100% — no wellbore is close to being unusable, but two wellbores (5351, 5599) each carry 85 unrecorded calendar days, and wellbore 5693 carries 53 (which turns out to account for the entire field-wide gap above, since its span covers the full field date range).
- **Gaps:** 5 of 7 wellbores have at least one gap larger than a single day; the largest is 30 days (wellbore 5693, 2016-11-01 to 2016-12-01, right at the end of its recorded history). Notably, wellbores 5351 and 5599 each show an identical 12-day gap over the *same* two calendar dates (2012-01-02 to 2012-01-14) — a coincidence across two different wells that is worth investigating as a possible shared shutdown/maintenance event, not assumed to be one.
- **Field-life boundary:** 244 rows fall before the documented 2008-01-01 start of Volve production, all in Sep-Dec 2007, all on wellbores 5693 and 5769, all `WELL_TYPE = WI` / `FLOW_KIND = injection` with `ON_STREAM_HRS` NULL for every one of them. This is flagged for REVIEW only — it is plausible this reflects real early water-injection activity that predates what is commonly cited as "first oil," rather than a data error, but that is a domain question, not something this notebook decides.
- **Record presence vs. activity (Section 9.6):** wellbores 5693 and 5769 also carry the dataset's only `record_present_hours_missing` days (152 and 133 respectively) — rows that exist but have a NULL `ON_STREAM_HRS`. This overlaps suspiciously with the pre-2008 injection rows just described, which is a specific, testable link Section 10 can pick up rather than something concluded here.

```
VERDICT: PASS WITH REVIEW
```

For relational modelling, none of this blocks building the daily fact table on the confirmed `NPD_WELL_BORE_CODE + DATEPRD` grain (Section 4) — temporal gaps and pre-2008 records are legitimate rows, not malformed ones, and should be loaded as-is. What it does establish is that "no record for a date" and "a record exists but the well wasn't reporting/operating" are two structurally different situations in this dataset (53 field-wide gap-days vs. hundreds of inactive-or-hours-missing days per wellbore), and any later aggregation or reconciliation logic (e.g. Section 21) must not treat a missing calendar day and a recorded zero the same way.


## 10. On-stream-hours validation

**Main question:** is `ON_STREAM_HRS` internally consistent with the temporal and production/injection information recorded for each wellbore-day?

Six tests: basic validity (the one place a hard physical/calendar constraint is justified), a breakdown of NULL operating hours, zero-hours-vs-positive-volumes, positive-hours-vs-zero-volumes, the operating-hours distribution by wellbore, and an explicit follow-up on the Section 9 finding for wellbores 5693/5769.

**Audit-trail convention introduced here:** recurring findings that will be catalogued as formal issues in Section 23 are marked below with an anticipated ID — e.g. *(anticipated DQ-003, Section 23)*. The actual IDs are assigned when Section 23 is implemented; these are forward references so the notebook reads as a trail rather than independent EDA sections. The Section 9 finding this section follows up on is anticipated **DQ-001** (pre-field-life injection records); the NULL-hours pattern investigated below is anticipated **DQ-003**. The 12-day shared gap for wellbores 5351/5599 (Section 9.4) is deliberately **not** investigated here — it belongs to the issue register as anticipated **DQ-002**, not to on-stream-hours validation.


In [62]:
# --- TEMPORARY LOADING FOR SECTION 10 ---
# Section 1 (Load source data) has not been implemented yet in this notebook.
# This block loads only what Section 10 needs, and is clearly marked so it can
# be deleted once Section 1 provides `daily_df` for the whole notebook.
if "daily_df" not in globals():
    from pathlib import Path
    import pandas as pd

    PROJECT_ROOT = Path.cwd().parent
    WORKBOOK_PATH = PROJECT_ROOT / "data" / "raw" / "Volve production data.xlsx"
    DAILY_SHEET_NAME = "Daily Production Data"

    if not WORKBOOK_PATH.exists():
        raise FileNotFoundError(f"Source workbook not found at {WORKBOOK_PATH}")

    daily_df = pd.read_excel(WORKBOOK_PATH, sheet_name=DAILY_SHEET_NAME)
    print(f"[Section 10 temporary load] daily_df loaded: {daily_df.shape}")
else:
    print(f"Using daily_df already loaded earlier in the notebook: {daily_df.shape}")


Using daily_df already loaded earlier in the notebook: (15634, 24)


In [63]:
# Section 10 directly follows up Section 9's temporal findings. If Section 9
# already ran in this kernel session, its variables (dateprd_parsed,
# daily_with_dates, the pre-2008 field-life mask) are reused as-is; if not,
# the minimum needed is recomputed here so Section 10 still works in isolation.
if "dateprd_parsed" not in globals():
    dateprd_parsed = pd.to_datetime(daily_df["DATEPRD"], errors="coerce")
if "daily_with_dates" not in globals():
    daily_with_dates = daily_df.copy()
    daily_with_dates["DATEPRD_parsed"] = dateprd_parsed
if "EXPECTED_FIELD_START" not in globals():
    EXPECTED_FIELD_START = pd.Timestamp("2008-01-01")
if "EXPECTED_FIELD_END" not in globals():
    EXPECTED_FIELD_END = pd.Timestamp("2016-12-31")
if "before_field_life_mask" not in globals():
    before_field_life_mask = dateprd_parsed.notna() & (dateprd_parsed < EXPECTED_FIELD_START)

print(f"Pre-2008 records available for follow-up (Section 9): {int(before_field_life_mask.sum())}")


Pre-2008 records available for follow-up (Section 9): 244


In [64]:
# Confirm the columns this section needs before testing anything else.
required_columns = [
    "ON_STREAM_HRS", "NPD_WELL_BORE_CODE", "NPD_WELL_BORE_NAME", "WELL_TYPE", "FLOW_KIND",
    "BORE_OIL_VOL", "BORE_GAS_VOL", "BORE_WAT_VOL", "BORE_WI_VOL",
]
missing_columns = [c for c in required_columns if c not in daily_df.columns]

if missing_columns:
    raise ValueError(f"Required column(s) missing from daily_df: {missing_columns}")

hours = daily_df["ON_STREAM_HRS"]
print(f"Required columns present: {required_columns}")


Required columns present: ['ON_STREAM_HRS', 'NPD_WELL_BORE_CODE', 'NPD_WELL_BORE_NAME', 'WELL_TYPE', 'FLOW_KIND', 'BORE_OIL_VOL', 'BORE_GAS_VOL', 'BORE_WAT_VOL', 'BORE_WI_VOL']


### 10.1 Basic validity

A calendar day has 24 hours and cannot have negative operating time — this is one of the few places in this notebook where a hard physical/calendar rule justifies a structural **FAIL** rather than a REVIEW.


In [65]:
total_rows = len(daily_df)
n_non_null = int(hours.notna().sum())
n_null = int(hours.isna().sum())
n_zero = int((hours == 0).sum())
n_valid_range = int(((hours > 0) & (hours <= 24)).sum())
n_negative = int((hours < 0).sum())
n_above_24 = int((hours > 24).sum())

print(f"Total rows:                  {total_rows}")
print(f"Non-NULL ON_STREAM_HRS:      {n_non_null}")
print(f"NULL ON_STREAM_HRS:          {n_null}")
print(f"ON_STREAM_HRS = 0:           {n_zero}")
print(f"0 < ON_STREAM_HRS <= 24:     {n_valid_range}")
print(f"ON_STREAM_HRS < 0 (FAIL):    {n_negative}")
print(f"ON_STREAM_HRS > 24 (FAIL):   {n_above_24}")

if n_negative > 0 or n_above_24 > 0:
    basic_validity_status = "FAIL"
elif n_null > 0:
    basic_validity_status = "REVIEW"
else:
    basic_validity_status = "PASS"
print(f"\nBasic validity status: {basic_validity_status}")


Total rows:                  15634
Non-NULL ON_STREAM_HRS:      15349
NULL ON_STREAM_HRS:          285
ON_STREAM_HRS = 0:           1952
0 < ON_STREAM_HRS <= 24:     13377
ON_STREAM_HRS < 0 (FAIL):    0
ON_STREAM_HRS > 24 (FAIL):   20

Basic validity status: FAIL


**The 20 `ON_STREAM_HRS > 24` records are the FAIL-triggering condition above, so they are inspected directly rather than left as a bare count** *(anticipated DQ-004, Section 23)*:


In [66]:
over_24_rows = (
    daily_with_dates.loc[
        hours > 24,
        ["DATEPRD_parsed", "NPD_WELL_BORE_CODE", "NPD_WELL_BORE_NAME", "WELL_TYPE", "FLOW_KIND",
         "ON_STREAM_HRS", "BORE_OIL_VOL", "BORE_GAS_VOL", "BORE_WAT_VOL", "BORE_WI_VOL"],
    ]
    .sort_values("DATEPRD_parsed")
    .reset_index(drop=True)
)
print(f"Rows with ON_STREAM_HRS > 24: {len(over_24_rows)}")
print(f"Distinct calendar dates involved: {sorted(over_24_rows['DATEPRD_parsed'].dt.date.unique().tolist())}")
over_24_rows


Rows with ON_STREAM_HRS > 24: 20
Distinct calendar dates involved: [datetime.date(2008, 10, 26), datetime.date(2009, 10, 25), datetime.date(2010, 10, 31), datetime.date(2013, 10, 27), datetime.date(2014, 10, 26)]


,DATEPRD_parsed,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,WELL_TYPE,FLOW_KIND,ON_STREAM_HRS,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL
0,2008-10-26,5599,15/9-F-12,OP,production,25.00000,3836.99,547535.29,12.84,NaN
1,2008-10-26,5769,15/9-F-5,WI,injection,25.00000,NaN,NaN,NaN,6638.000000
2,2008-10-26,5351,15/9-F-14,OP,production,25.00000,4608.81,657672.80,1.54,NaN
3,2008-10-26,5693,15/9-F-4,WI,injection,25.00000,NaN,NaN,NaN,8783.000000
4,2009-10-25,5351,15/9-F-14,OP,production,24.50000,3384.02,472845.45,1072.62,NaN
5,2009-10-25,5599,15/9-F-12,OP,production,24.83333,4190.37,585516.23,279.62,NaN
6,2010-10-31,5769,15/9-F-5,WI,injection,25.00000,NaN,NaN,NaN,7220.000000
7,2010-10-31,5599,15/9-F-12,OP,production,25.00000,1677.68,248776.29,3676.04,NaN
8,2010-10-31,5351,15/9-F-14,OP,production,25.00000,2497.07,370280.31,3555.68,NaN
9,2013-10-27,7078,15/9-F-11,OP,production,24.30839,1219.94,188361.38,69.99,NaN


### 10.2 Investigate NULL operating hours *(anticipated DQ-003)*

Breaking the NULL population down by wellbore, `WELL_TYPE`, `FLOW_KIND`, year, and recorded production/injection activity — then testing the Section 9 finding directly: do all 244 pre-2008 records have NULL hours, and are all NULL-hours records explained by wellbores 5693/5769?


In [67]:
null_hours_df = daily_with_dates.loc[hours.isna()]

null_by_wellbore = (
    null_hours_df.groupby(["NPD_WELL_BORE_CODE", "NPD_WELL_BORE_NAME"])
    .size()
    .rename("null_hours_count")
    .reset_index()
)
null_by_wellbore


,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,null_hours_count
0,5693,15/9-F-4,152
1,5769,15/9-F-5,133


In [68]:
null_by_well_type = (
    null_hours_df["WELL_TYPE"].value_counts(dropna=False).rename_axis("WELL_TYPE").reset_index(name="null_hours_count")
)
null_by_well_type


,WELL_TYPE,null_hours_count
0,WI,285


In [69]:
null_by_flow_kind = (
    null_hours_df["FLOW_KIND"].value_counts(dropna=False).rename_axis("FLOW_KIND").reset_index(name="null_hours_count")
)
null_by_flow_kind


,FLOW_KIND,null_hours_count
0,injection,285


In [70]:
null_by_year = (
    null_hours_df["DATEPRD_parsed"].dt.year.value_counts().sort_index().rename_axis("year").reset_index(name="null_hours_count")
)
null_by_year


,year,null_hours_count
0,2007,244
1,2008,21
2,2016,20


In [71]:
# Recorded production/injection activity within the NULL-hours population -
# NaN volumes are not counted as positive activity, only genuine positive values are.
volume_cols = ["BORE_OIL_VOL", "BORE_GAS_VOL", "BORE_WAT_VOL", "BORE_WI_VOL"]
has_positive_volume = (null_hours_df[volume_cols] > 0).any(axis=1)

print(f"NULL-hours rows with at least one positive volume recorded: {int(has_positive_volume.sum())}")
print(f"NULL-hours rows with no positive volume recorded:           {int((~has_positive_volume).sum())}")


NULL-hours rows with at least one positive volume recorded: 0
NULL-hours rows with no positive volume recorded:           285


In [72]:
# Direct test of the Section 9 finding.
pre_2008_total = int(before_field_life_mask.sum())
pre_2008_and_null = int((before_field_life_mask & hours.isna()).sum())
null_total = n_null
null_and_pre_2008 = pre_2008_and_null

wellbores_with_null_hours = sorted(null_hours_df["NPD_WELL_BORE_CODE"].unique().tolist())

print(f"Pre-2008 records (Section 9):                  {pre_2008_total}")
print(f"Pre-2008 records with NULL ON_STREAM_HRS:      {pre_2008_and_null} / {pre_2008_total}")
print(f"NULL ON_STREAM_HRS records total:               {null_total}")
print(f"NULL ON_STREAM_HRS records that are pre-2008:   {null_and_pre_2008} / {null_total}")
print(f"\nWellbores with any NULL ON_STREAM_HRS record: {wellbores_with_null_hours}")


Pre-2008 records (Section 9):                  244
Pre-2008 records with NULL ON_STREAM_HRS:      244 / 244
NULL ON_STREAM_HRS records total:               285
NULL ON_STREAM_HRS records that are pre-2008:   244 / 285

Wellbores with any NULL ON_STREAM_HRS record: [5693, 5769]


### 10.3 Zero hours vs. positive volumes

Four separate tests. Each is **REVIEW**, not FAIL — a recorded zero operating-hour day alongside a positive volume could reflect a reporting convention or allocation timing, not necessarily an error.


In [73]:
zero_hours_mask = hours == 0

n_zero_hours_oil = int((zero_hours_mask & (daily_df["BORE_OIL_VOL"] > 0)).sum())
n_zero_hours_gas = int((zero_hours_mask & (daily_df["BORE_GAS_VOL"] > 0)).sum())
n_zero_hours_water = int((zero_hours_mask & (daily_df["BORE_WAT_VOL"] > 0)).sum())
n_zero_hours_wi = int((zero_hours_mask & (daily_df["BORE_WI_VOL"] > 0)).sum())

print("ON_STREAM_HRS = 0 combined with a positive volume (REVIEW):")
print(f"  + oil > 0:              {n_zero_hours_oil}")
print(f"  + gas > 0:              {n_zero_hours_gas}")
print(f"  + water > 0:            {n_zero_hours_water}")
print(f"  + water injection > 0:  {n_zero_hours_wi}")


ON_STREAM_HRS = 0 combined with a positive volume (REVIEW):
  + oil > 0:              1
  + gas > 0:              3
  + water > 0:            3
  + water injection > 0:  31


In [74]:
zero_hours_review_mask = zero_hours_mask & (
    (daily_df["BORE_OIL_VOL"] > 0)
    | (daily_df["BORE_GAS_VOL"] > 0)
    | (daily_df["BORE_WAT_VOL"] > 0)
    | (daily_df["BORE_WI_VOL"] > 0)
)
zero_hours_review_rows = (
    daily_with_dates.loc[
        zero_hours_review_mask,
        ["DATEPRD_parsed", "NPD_WELL_BORE_CODE", "NPD_WELL_BORE_NAME", "WELL_TYPE", "FLOW_KIND",
         "ON_STREAM_HRS", "BORE_OIL_VOL", "BORE_GAS_VOL", "BORE_WAT_VOL", "BORE_WI_VOL"],
    ]
    .sort_values("DATEPRD_parsed")
    .reset_index(drop=True)
)
print(f"Rows with ON_STREAM_HRS = 0 and at least one positive volume: {len(zero_hours_review_rows)}")
zero_hours_review_rows


Rows with ON_STREAM_HRS = 0 and at least one positive volume: 34


,DATEPRD_parsed,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,WELL_TYPE,FLOW_KIND,ON_STREAM_HRS,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL
0,2008-08-07,5693,15/9-F-4,WI,injection,0.0,NaN,NaN,NaN,0.088160
1,2011-06-18,5693,15/9-F-4,WI,injection,0.0,NaN,NaN,NaN,6263.692353
2,2011-10-16,5769,15/9-F-5,WI,injection,0.0,NaN,NaN,NaN,48.548819
3,2011-10-18,5693,15/9-F-4,WI,injection,0.0,NaN,NaN,NaN,0.008047
4,2011-11-30,5693,15/9-F-4,WI,injection,0.0,NaN,NaN,NaN,0.045601
5,2011-12-01,5693,15/9-F-4,WI,injection,0.0,NaN,NaN,NaN,0.012574
6,2011-12-04,5769,15/9-F-5,WI,injection,0.0,NaN,NaN,NaN,45.386962
7,2011-12-09,5693,15/9-F-4,WI,injection,0.0,NaN,NaN,NaN,0.020954
8,2011-12-25,5693,15/9-F-4,WI,injection,0.0,NaN,NaN,NaN,0.978442
9,2011-12-25,5769,15/9-F-5,WI,injection,0.0,NaN,NaN,NaN,54.799771


### 10.4 Positive hours vs. zero volumes

`ON_STREAM_HRS > 0` with oil, gas, water, and water injection all exactly `0` (not NULL — NULL is a different situation, tested in 10.2). Also REVIEW, broken down by `WELL_TYPE` and `FLOW_KIND` since a producer and an injector may have very different, equally legitimate, explanations.


In [75]:
positive_hours_mask = hours > 0
all_zero_volumes_mask = (
    (daily_df["BORE_OIL_VOL"] == 0)
    & (daily_df["BORE_GAS_VOL"] == 0)
    & (daily_df["BORE_WAT_VOL"] == 0)
    & (daily_df["BORE_WI_VOL"] == 0)
)
positive_hours_zero_volumes_mask = positive_hours_mask & all_zero_volumes_mask
n_positive_hours_zero_volumes = int(positive_hours_zero_volumes_mask.sum())

print(f"ON_STREAM_HRS > 0 with oil = gas = water = water injection = 0 (REVIEW): {n_positive_hours_zero_volumes}")


ON_STREAM_HRS > 0 with oil = gas = water = water injection = 0 (REVIEW): 0


In [76]:
positive_hours_zero_volumes_rows = daily_df.loc[positive_hours_zero_volumes_mask]

breakdown_by_well_type = (
    positive_hours_zero_volumes_rows["WELL_TYPE"].value_counts(dropna=False).rename_axis("WELL_TYPE").reset_index(name="count")
)
breakdown_by_well_type


,WELL_TYPE,count


In [77]:
breakdown_by_flow_kind = (
    positive_hours_zero_volumes_rows["FLOW_KIND"].value_counts(dropna=False).rename_axis("FLOW_KIND").reset_index(name="count")
)
breakdown_by_flow_kind


,FLOW_KIND,count


### 10.5 Operating-hours distribution by wellbore

Includes total reported `ON_STREAM_HRS` per wellbore — useful later when comparing this daily aggregation with the monthly worksheet's `On Stream` field (Section 21).


In [78]:
hours_distribution_rows = []
for code_value, group in daily_df.groupby("NPD_WELL_BORE_CODE"):
    name = group["NPD_WELL_BORE_NAME"].iloc[0]
    h = group["ON_STREAM_HRS"]
    hours_distribution_rows.append(
        {
            "npd_code": code_value,
            "wellbore_name": name,
            "count": int(h.notna().sum()),
            "null_count": int(h.isna().sum()),
            "zero_count": int((h == 0).sum()),
            "mean": h.mean(),
            "median": h.median(),
            "min": h.min(),
            "max": h.max(),
            "p25": h.quantile(0.25),
            "p75": h.quantile(0.75),
            "total_reported_hours": h.sum(),
        }
    )

hours_distribution = pd.DataFrame(hours_distribution_rows).sort_values("npd_code").reset_index(drop=True)
hours_distribution


,npd_code,wellbore_name,count,null_count,zero_count,mean,median,min,max,p25,p75,total_reported_hours
0,5351,15/9-F-14,3056,0,332,20.541124,24.0,0.0,25.0,24.000000,24.0,62773.67519
1,5599,15/9-F-12,3056,0,218,21.336410,24.0,0.0,25.0,24.000000,24.0,65204.06928
2,5693,15/9-F-4,3175,152,345,20.241626,24.0,0.0,25.0,23.583330,24.0,64267.16117
3,5769,15/9-F-5,3173,133,497,19.171085,24.0,0.0,25.0,20.833330,24.0,60829.85308
4,7078,15/9-F-11,1165,0,43,22.322932,24.0,0.0,25.0,24.000000,24.0,26006.21614
5,7289,15/9-F-15 D,978,0,209,18.225800,24.0,0.0,24.0,15.612497,24.0,17824.83278
6,7405,15/9-F-1 C,746,0,308,13.382752,24.0,0.0,25.0,0.000000,24.0,9983.53315


In [79]:
field_total_reported_hours = hours.sum()
print(f"Field-wide total reported ON_STREAM_HRS (sum across all rows): {field_total_reported_hours:,.2f}")


Field-wide total reported ON_STREAM_HRS (sum across all rows): 306,889.34


### 10.6 Explicit Section 9 follow-up *(anticipated DQ-001)*

Wellbores 5693 and 5769, restricted to the pre-2008 / NULL-hours period. The question here is **not** "are these records wrong" — it is "what exactly characterizes these records." A compact per-well/type/flow-kind characterization is shown first, followed by a small sample of the raw rows.


In [80]:
anomalous_wellbores = [5693, 5769]
anomalous_period_mask = daily_with_dates["NPD_WELL_BORE_CODE"].isin(anomalous_wellbores) & (
    before_field_life_mask | hours.isna()
)
anomalous_period_rows = (
    daily_with_dates.loc[
        anomalous_period_mask,
        ["DATEPRD_parsed", "NPD_WELL_BORE_CODE", "NPD_WELL_BORE_NAME", "WELL_TYPE", "FLOW_KIND",
         "ON_STREAM_HRS", "BORE_OIL_VOL", "BORE_GAS_VOL", "BORE_WAT_VOL", "BORE_WI_VOL"],
    ]
    .sort_values(["NPD_WELL_BORE_CODE", "DATEPRD_parsed"])
    .reset_index(drop=True)
)
print(f"Rows for wellbores {anomalous_wellbores} that are pre-2008 and/or have NULL ON_STREAM_HRS: {len(anomalous_period_rows)}")


Rows for wellbores [5693, 5769] that are pre-2008 and/or have NULL ON_STREAM_HRS: 285


In [81]:
anomalous_summary = (
    anomalous_period_rows.groupby(["NPD_WELL_BORE_CODE", "WELL_TYPE", "FLOW_KIND"])
    .agg(
        n_rows=("ON_STREAM_HRS", "size"),
        first_date=("DATEPRD_parsed", "min"),
        last_date=("DATEPRD_parsed", "max"),
        on_stream_hrs_null_count=("ON_STREAM_HRS", lambda s: int(s.isna().sum())),
        oil_vol_nonzero_count=("BORE_OIL_VOL", lambda s: int((s.fillna(0) != 0).sum())),
        gas_vol_nonzero_count=("BORE_GAS_VOL", lambda s: int((s.fillna(0) != 0).sum())),
        wat_vol_nonzero_count=("BORE_WAT_VOL", lambda s: int((s.fillna(0) != 0).sum())),
        wi_vol_nonzero_count=("BORE_WI_VOL", lambda s: int((s.fillna(0) != 0).sum())),
    )
    .reset_index()
)
anomalous_summary


,NPD_WELL_BORE_CODE,WELL_TYPE,FLOW_KIND,n_rows,first_date,last_date,on_stream_hrs_null_count,oil_vol_nonzero_count,gas_vol_nonzero_count,wat_vol_nonzero_count,wi_vol_nonzero_count
0,5693,WI,injection,152,2007-09-01,2016-12-01,152,0,0,0,0
1,5769,WI,injection,133,2007-09-01,2008-07-21,133,0,0,0,0


In [82]:
# Small raw sample - first 10 records per wellbore, for a direct look.
anomalous_period_rows.groupby("NPD_WELL_BORE_CODE").head(10)


,DATEPRD_parsed,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,WELL_TYPE,FLOW_KIND,ON_STREAM_HRS,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL
0,2007-09-01,5693,15/9-F-4,WI,injection,NaN,NaN,NaN,NaN,NaN
1,2007-09-02,5693,15/9-F-4,WI,injection,NaN,NaN,NaN,NaN,NaN
2,2007-09-03,5693,15/9-F-4,WI,injection,NaN,NaN,NaN,NaN,NaN
3,2007-09-04,5693,15/9-F-4,WI,injection,NaN,NaN,NaN,NaN,NaN
4,2007-09-05,5693,15/9-F-4,WI,injection,NaN,NaN,NaN,NaN,NaN
5,2007-09-06,5693,15/9-F-4,WI,injection,NaN,NaN,NaN,NaN,NaN
6,2007-09-07,5693,15/9-F-4,WI,injection,NaN,NaN,NaN,NaN,NaN
7,2007-09-08,5693,15/9-F-4,WI,injection,NaN,NaN,NaN,NaN,NaN
8,2007-09-09,5693,15/9-F-4,WI,injection,NaN,NaN,NaN,NaN,NaN
9,2007-09-10,5693,15/9-F-4,WI,injection,NaN,NaN,NaN,NaN,NaN


### 10.7 DST-hypothesis verification *(DQ-004)*

The DST ("daylight-saving fall-back day is 25 hours") explanation for the 20 `ON_STREAM_HRS > 24` records is a hypothesis, not a conclusion — it must be tested before it informs any constraint or business rule. Five specific checks, as requested:

1. Do all 20 records equal exactly 25 hours?
2. Do they occur on the expected autumn DST transition date for their year?
3. Is the pattern consistent across the wellbores that were actually reporting on those dates?
4. Are there zero qualifying values (between 24 and 25, or above 25) on any *other* date?
5. Does this correspond to the source's documented reporting convention and timezone?


In [83]:
# Check 1: exactly 25.0, or something else?
n_exactly_25 = int((over_24_rows["ON_STREAM_HRS"] == 25).sum())
n_between_24_and_25 = len(over_24_rows) - n_exactly_25

print(f"Records with ON_STREAM_HRS exactly 25.0:              {n_exactly_25} / {len(over_24_rows)}")
print(f"Records with ON_STREAM_HRS strictly between 24 and 25: {n_between_24_and_25} / {len(over_24_rows)}")
print()
over_24_rows[["DATEPRD_parsed", "NPD_WELL_BORE_CODE", "NPD_WELL_BORE_NAME", "ON_STREAM_HRS"]].sort_values("ON_STREAM_HRS")


Records with ON_STREAM_HRS exactly 25.0:              13 / 20
Records with ON_STREAM_HRS strictly between 24 and 25: 7 / 20



,DATEPRD_parsed,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,ON_STREAM_HRS
13,2013-10-27,5693,15/9-F-4,24.14167
11,2013-10-27,5769,15/9-F-5,24.14167
10,2013-10-27,5599,15/9-F-12,24.14167
12,2013-10-27,5351,15/9-F-14,24.30833
9,2013-10-27,7078,15/9-F-11,24.30839
4,2009-10-25,5351,15/9-F-14,24.50000
5,2009-10-25,5599,15/9-F-12,24.83333
17,2014-10-26,7078,15/9-F-11,25.00000
16,2014-10-26,5599,15/9-F-12,25.00000
15,2014-10-26,7405,15/9-F-1 C,25.00000


In [84]:
# Check 2: do the observed dates match the expected EU/Norway DST
# "fall back" date (last Sunday of October) for their respective year?
def last_sunday_of_october(year: int) -> pd.Timestamp:
    d = pd.Timestamp(year=year, month=10, day=31)
    offset_days = (d.weekday() - 6) % 7  # weekday(): Monday=0 ... Sunday=6
    return d - pd.Timedelta(days=offset_days)

observed_dates = sorted(over_24_rows["DATEPRD_parsed"].dt.normalize().unique())
years_present = sorted({pd.Timestamp(d).year for d in observed_dates})
expected_dst_dates = {year: last_sunday_of_october(year) for year in years_present}

print("Expected EU/Norway DST fall-back date (last Sunday of October) by year:")
for year, expected in expected_dst_dates.items():
    print(f"  {year}: {expected.date()}")

print("\nObserved dates with ON_STREAM_HRS > 24:")
for d in observed_dates:
    print(f"  {pd.Timestamp(d).date()}")

all_dates_match_dst = all(pd.Timestamp(d) in expected_dst_dates.values() for d in observed_dates)
print(f"\nEvery observed over-24-hour date matches the expected DST fall-back date for its year: {all_dates_match_dst}")


Expected EU/Norway DST fall-back date (last Sunday of October) by year:
  2008: 2008-10-26
  2009: 2009-10-25
  2010: 2010-10-31
  2013: 2013-10-27
  2014: 2014-10-26

Observed dates with ON_STREAM_HRS > 24:
  2008-10-26
  2009-10-25
  2010-10-31
  2013-10-27
  2014-10-26

Every observed over-24-hour date matches the expected DST fall-back date for its year: True


In [85]:
# Check 3: is the pattern consistent across the wellbores that were
# actually reporting on each of these dates, or only some of them?
wellbore_consistency_rows = []
for d in observed_dates:
    day_rows = daily_with_dates.loc[daily_with_dates["DATEPRD_parsed"] == d]
    for _, row in day_rows.iterrows():
        wellbore_consistency_rows.append(
            {
                "date": pd.Timestamp(d).date(),
                "npd_code": row["NPD_WELL_BORE_CODE"],
                "wellbore_name": row["NPD_WELL_BORE_NAME"],
                "on_stream_hrs": row["ON_STREAM_HRS"],
                "exceeds_24": bool(pd.notna(row["ON_STREAM_HRS"]) and row["ON_STREAM_HRS"] > 24),
            }
        )

wellbore_consistency = pd.DataFrame(wellbore_consistency_rows).sort_values(["date", "npd_code"]).reset_index(drop=True)
wellbore_consistency


,date,npd_code,wellbore_name,on_stream_hrs,exceeds_24
0,2008-10-26,5351,15/9-F-14,25.00000,True
1,2008-10-26,5599,15/9-F-12,25.00000,True
2,2008-10-26,5693,15/9-F-4,25.00000,True
3,2008-10-26,5769,15/9-F-5,25.00000,True
4,2009-10-25,5351,15/9-F-14,24.50000,True
5,2009-10-25,5599,15/9-F-12,24.83333,True
6,2009-10-25,5693,15/9-F-4,23.83333,False
7,2009-10-25,5769,15/9-F-5,24.00000,False
8,2010-10-31,5351,15/9-F-14,25.00000,True
9,2010-10-31,5599,15/9-F-12,25.00000,True


In [86]:
consistency_summary = (
    wellbore_consistency.groupby("date")
    .agg(wellbores_reporting=("npd_code", "nunique"), wellbores_exceeding_24=("exceeds_24", "sum"))
    .reset_index()
)
consistency_summary


,date,wellbores_reporting,wellbores_exceeding_24
0,2008-10-26,4,4
1,2009-10-25,4,2
2,2010-10-31,4,3
3,2013-10-27,5,5
4,2014-10-26,7,6


In [87]:
# Check 4: are there ANY qualifying values (>24) on a date other than
# the 5 identified DST dates?
non_dst_over_24 = daily_with_dates.loc[
    (hours > 24) & (~daily_with_dates["DATEPRD_parsed"].dt.normalize().isin(observed_dates))
]
print(f"Rows with ON_STREAM_HRS > 24 on a date OTHER than the 5 identified DST dates: {len(non_dst_over_24)}")


Rows with ON_STREAM_HRS > 24 on a date OTHER than the 5 identified DST dates: 0


**Check 5 — reporting convention and timezone:** cannot be tested from the data itself. Confirming that Volve's `ON_STREAM_HRS` field is genuinely a wall-clock-hours measure recorded in a DST-observing timezone (rather than, say, a fixed allocation basis unrelated to clock time) requires Equinor's Volve dataset documentation, which is outside this notebook's inputs. This check remains open.


In [88]:
print("DST HYPOTHESIS VERIFICATION SUMMARY (DQ-004)")
print("-" * 60)
print(f"1. All 20 records exactly 25.0 hours:            {n_exactly_25}/{len(over_24_rows)} -> {'CONFIRMED' if n_exactly_25 == len(over_24_rows) else 'PARTIALLY SUPPORTED'}")
print(f"2. All dates match expected DST fall-back date:  {'CONFIRMED' if all_dates_match_dst else 'NOT CONFIRMED'}")
print(f"3. Consistent across reporting wellbores:        see consistency_summary above (not all reporting wellbores exceed 24h on every DST date)")
print(f"4. No qualifying values on other dates:           {'CONFIRMED' if len(non_dst_over_24) == 0 else 'NOT CONFIRMED'} ({len(non_dst_over_24)} found)")
print(f"5. Matches source reporting convention/timezone: NOT TESTABLE FROM DATA ALONE")
print()
print("DQ-004 status: OPEN. Checks 2 and 4 are fully confirmed by the data (the date")
print("pattern is exact and the phenomenon is fully isolated to those 5 dates). Check 1")
print("is only partially clean and check 3 shows the effect is not uniform across every")
print("reporting wellbore, which is evidence against a simple universal clock artifact.")
print("Check 5 cannot be resolved without source documentation. Treat DQ-004 as accepted-")
print("pending-confirmation, not resolved - a generic ON_STREAM_HRS <= 24 rule correctly")
print("flagged these rows for review; it should not be silently widened to 25 without that")
print("confirmation.")


DST HYPOTHESIS VERIFICATION SUMMARY (DQ-004)
------------------------------------------------------------
1. All 20 records exactly 25.0 hours:            13/20 -> PARTIALLY SUPPORTED
2. All dates match expected DST fall-back date:  CONFIRMED
3. Consistent across reporting wellbores:        see consistency_summary above (not all reporting wellbores exceed 24h on every DST date)
4. No qualifying values on other dates:           CONFIRMED (0 found)
5. Matches source reporting convention/timezone: NOT TESTABLE FROM DATA ALONE

DQ-004 status: OPEN. Checks 2 and 4 are fully confirmed by the data (the date
pattern is exact and the phenomenon is fully isolated to those 5 dates). Check 1
is only partially clean and check 3 shows the effect is not uniform across every
reporting wellbore, which is evidence against a simple universal clock artifact.
Check 5 cannot be resolved without source documentation. Treat DQ-004 as accepted-
pending-confirmation, not resolved - a generic ON_STREAM_HRS <= 24

In [89]:
# Final on-stream-hours verdict. PASS / PASS WITH REVIEW / FAIL, matching the
# same non-binary style introduced for temporal data in Section 9.
zero_hours_positive_production_mask = zero_hours_mask & (
    (daily_df["BORE_OIL_VOL"] > 0) | (daily_df["BORE_GAS_VOL"] > 0) | (daily_df["BORE_WAT_VOL"] > 0)
)
n_zero_hours_positive_production = int(zero_hours_positive_production_mask.sum())
n_zero_hours_positive_injection = n_zero_hours_wi

if n_negative > 0 or n_above_24 > 0:
    hours_verdict = "FAIL"
elif (
    n_null > 0
    or n_zero_hours_positive_production > 0
    or n_zero_hours_positive_injection > 0
    or n_positive_hours_zero_volumes > 0
):
    hours_verdict = "PASS WITH REVIEW"
else:
    hours_verdict = "PASS"

print("=" * 60)
print("ON-STREAM HOURS VALIDATION")
print("=" * 60)
print(f"Rows:                              {total_rows:,}")
print()
print(f"Valid range (0,24]:                {n_valid_range:,}")
print(f"NULL:                              {n_null:,}")
print(f"Negative:                          {n_negative}")
print(f"Above 24:                          {n_above_24}")
print()
print(f"Zero hours + positive production:  {n_zero_hours_positive_production}")
print(f"Zero hours + positive injection:   {n_zero_hours_positive_injection}")
print(f"Positive hours + zero volumes:     {n_positive_hours_zero_volumes}")
print()
print("Section 9 pre-2008 population:")
print(f"{pre_2008_total} records")
print()
print("Pre-2008 with NULL hours:")
print(f"{pre_2008_and_null} / {pre_2008_total}")
print()
print(f"VERDICT: {hours_verdict}")


ON-STREAM HOURS VALIDATION
Rows:                              15,634

Valid range (0,24]:                13,377
NULL:                              285
Negative:                          0
Above 24:                          20

Zero hours + positive production:  3
Zero hours + positive injection:   31
Positive hours + zero volumes:     0

Section 9 pre-2008 population:
244 records

Pre-2008 with NULL hours:
244 / 244

VERDICT: FAIL


**Interpretation for later relational modelling:**

The verdict is **FAIL** — the tested generic rule `ON_STREAM_HRS <= 24` is violated by 20 rows. That is a fact about this rule, not a verdict on the dataset: Section 23 can still classify DQ-004 as accepted source behaviour if the DST explanation holds up, and the overall dataset can still end up "suitable, with documented rules" even though this specific structural check failed.

**DQ-004 is left OPEN, not resolved**, because the five-point verification produced a genuinely mixed result rather than a clean confirmation:

1. **Magnitude — partially supported.** Only 13/20 records are exactly 25.0 hours; the other 7 sit between 24.14 and 24.83. A pure clock artifact would push every affected record to exactly +1 hour, so this alone does not confirm DST.
2. **Date pattern — confirmed.** All 5 dates (2008-10-26, 2009-10-25, 2010-10-31, 2013-10-27, 2014-10-26) exactly match the computed last-Sunday-of-October EU/Norway DST fall-back date for their respective year. This is a precise, non-coincidental match.
3. **Wellbore consistency — not confirmed, and this is the important negative result.** The effect is not uniform across wellbores reporting on a given DST date. On 2009-10-25, wellbore 5693 recorded 23.83 hours (not >24) while 5351 recorded 24.50. On 2010-10-31, wellbore 5693 recorded exactly **0.0** hours while 5351, 5599, and 5769 all recorded 25.0. A universal timezone/clock artifact should affect every actively-reporting well on that calendar day the same way; this data does not show that.
4. **Isolation — confirmed.** Zero rows anywhere in the dataset exceed 24 hours on any date other than these 5. Whatever is happening is exclusively tied to these 5 calendar dates.
5. **Source convention/timezone — not testable from the data in hand.** This requires Equinor's Volve reporting documentation, which is outside this notebook's inputs.

Taken together: the date pattern (2) and isolation (4) are strong, precise evidence that something DST-related is happening on exactly these calendar dates. But the non-uniformity across wellbores (3) means it is not simply "every well gets +1 hour that day" — it may instead be that only some wells' source systems apply a DST adjustment, or that `ON_STREAM_HRS` is itself a computed/allocated value (not a raw clock reading) that only sometimes absorbs the extra hour. **No upper-bound constraint is recommended here.** Widening a `CHECK` constraint from 24 to 25 before understanding *why* the effect is inconsistent across wellbores would be adjusting the rule to fit the data rather than understanding the data — exactly the workaround this investigation was meant to avoid. The correct next step is domain/source-documentation confirmation (check 5), not a schema change.

**Data-quality audit trail so far:**

| ID | Finding | Status |
|---|---|---|
| DQ-001 | Pre-field-life injection records (wellbores 5693/5769, pre-2008) | Open — anticipated, not yet formally registered (Section 23) |
| DQ-002 | Shared 12-day reporting gap (wellbores 5351/5599, 2012-01-02 to 2012-01-14) | Open — untouched since Section 9, deliberately not investigated in Section 10 |
| DQ-003 | NULL `ON_STREAM_HRS` (285 rows, wellbores 5693/5769 only, 244 of which are also DQ-001) | Open — fully characterized in Section 10.2 |
| DQ-004 | `ON_STREAM_HRS > 24` on 5 DST-pattern dates (20 rows) | **Open** — DST hypothesis partially supported, not confirmed; see five-point verification above |

For relational modelling: no schema decision follows from DQ-004 yet. The daily fact table should still be built on the confirmed Section 4 grain with `ON_STREAM_HRS` loaded as-is (no constraint enforcing an upper bound until DQ-004 is resolved), and all four open issues above should carry forward into the Section 23 issue register with these same IDs so later sections can reference them directly rather than re-describing them.


## 11. Missing-value assessment

This goes beyond a global `isna()` table. Sections 9 and 10 already showed missingness has operational structure (NULL `ON_STREAM_HRS` is concentrated on exactly 2 wellbores, entirely `WELL_TYPE = WI`). Four levels are examined: global missingness, missingness by wellbore, missingness through time, and missingness by operating context (`WELL_TYPE`, `FLOW_KIND`, `ON_STREAM_HRS` status).

A preliminary classification is introduced to avoid treating every NULL the same way:

- **STRUCTURAL** — identifiers, dates, etc. Missing values here would threaten relational integrity.
- **CONTEXTUAL** — a field may be NULL because it does not apply to that operational mode (e.g. an injection-only field on a producer row).
- **MEASUREMENT** — a measurement was expected but is absent (sensor/reporting gap).
- **UNKNOWN** — meaning cannot yet be determined from evidence available in this section.

This classification is an **investigation aid, not a permanent decision** — everything here is "preliminary" and subject to revision once Section 12 (categorical context) and Sections 13-17 (per-measurement validation) look closer.

**`NULL ≠ 0` is preserved throughout.** `BORE_WI_VOL = 0` (a recorded zero injection volume) and `BORE_WI_VOL = NULL` (not measured / not applicable / not reported) are different facts with different implications for SQL analysis and schema design — neither is silently collapsed into the other anywhere in this section.


In [90]:
# --- TEMPORARY LOADING FOR SECTION 11 ---
# Section 1 (Load source data) has not been implemented yet in this notebook.
# This block loads only what Section 11 needs, and is clearly marked so it can
# be deleted once Section 1 provides `daily_df` for the whole notebook.
if "daily_df" not in globals():
    from pathlib import Path
    import pandas as pd

    PROJECT_ROOT = Path.cwd().parent
    WORKBOOK_PATH = PROJECT_ROOT / "data" / "raw" / "Volve production data.xlsx"
    DAILY_SHEET_NAME = "Daily Production Data"

    if not WORKBOOK_PATH.exists():
        raise FileNotFoundError(f"Source workbook not found at {WORKBOOK_PATH}")

    daily_df = pd.read_excel(WORKBOOK_PATH, sheet_name=DAILY_SHEET_NAME)
    print(f"[Section 11 temporary load] daily_df loaded: {daily_df.shape}")
else:
    print(f"Using daily_df already loaded earlier in the notebook: {daily_df.shape}")


Using daily_df already loaded earlier in the notebook: (15634, 24)


In [91]:
# Reuse Section 9/10 derived state if available, recompute the minimum if not.
if "dateprd_parsed" not in globals():
    dateprd_parsed = pd.to_datetime(daily_df["DATEPRD"], errors="coerce")
if "daily_with_dates" not in globals():
    daily_with_dates = daily_df.copy()
    daily_with_dates["DATEPRD_parsed"] = dateprd_parsed
if "hours" not in globals():
    hours = daily_df["ON_STREAM_HRS"]

year_series = daily_with_dates["DATEPRD_parsed"].dt.year
print("Ready: daily_with_dates, dateprd_parsed, hours, year_series")


Ready: daily_with_dates, dateprd_parsed, hours, year_series


### 11.1 Global missingness


In [92]:
global_missingness = (
    pd.DataFrame(
        {
            "missing_count": daily_df.isna().sum(),
            "missing_pct": (daily_df.isna().mean() * 100).round(2),
        }
    )
    .sort_values("missing_pct", ascending=False)
)
global_missingness


,missing_count,missing_pct
BORE_WI_VOL,9928,63.50
AVG_ANNULUS_PRESS,7744,49.53
AVG_CHOKE_SIZE_P,6715,42.95
AVG_DOWNHOLE_PRESSURE,6654,42.56
AVG_DOWNHOLE_TEMPERATURE,6654,42.56
AVG_DP_TUBING,6654,42.56
AVG_WHT_P,6488,41.50
AVG_WHP_P,6479,41.44
BORE_WAT_VOL,6473,41.40
BORE_GAS_VOL,6473,41.40


In [93]:
columns_with_missing = [c for c in daily_df.columns if daily_df[c].isna().any()]
print(f"Columns with at least one missing value: {len(columns_with_missing)} / {daily_df.shape[1]}")
print(columns_with_missing)


Columns with at least one missing value: 14 / 24
['ON_STREAM_HRS', 'AVG_DOWNHOLE_PRESSURE', 'AVG_DOWNHOLE_TEMPERATURE', 'AVG_DP_TUBING', 'AVG_ANNULUS_PRESS', 'AVG_CHOKE_SIZE_P', 'AVG_CHOKE_UOM', 'AVG_WHP_P', 'AVG_WHT_P', 'DP_CHOKE_SIZE', 'BORE_OIL_VOL', 'BORE_GAS_VOL', 'BORE_WAT_VOL', 'BORE_WI_VOL']


### 11.2 Missingness by wellbore

Which measurements are missing for which wells? Rows are columns, columns are wellbores (code and name), values are missing %.


In [94]:
# NOTE on errors="ignore" below: pandas >= 2.2 already excludes the
# groupby key column from the sub-frame passed into .apply(), so
# .drop(columns=...) on it would raise KeyError on this pandas version
# (3.0.5). errors="ignore" makes this cell work across pandas versions
# rather than silently swallowing an unrelated missing-column problem -
# the dropped column is always exactly the groupby key, never a data column.
wellbore_missingness_pct = daily_df.groupby("NPD_WELL_BORE_CODE").apply(lambda g: (g.isna().mean() * 100).round(2))
code_to_name = daily_df.drop_duplicates("NPD_WELL_BORE_CODE").set_index("NPD_WELL_BORE_CODE")["NPD_WELL_BORE_NAME"]
wellbore_missingness_pct = wellbore_missingness_pct.drop(columns=["NPD_WELL_BORE_CODE"], errors="ignore").T
wellbore_missingness_pct.columns = [f"{code_val} ({code_to_name[code_val]})" for code_val in wellbore_missingness_pct.columns]
wellbore_missingness_pct


,5351 (15/9-F-14),5599 (15/9-F-12),5693 (15/9-F-4),5769 (15/9-F-5),7078 (15/9-F-11),7289 (15/9-F-15 D),7405 (15/9-F-1 C)
DATEPRD,0.00,0.00,0.00,0.00,0.00,0.0,0.00
WELL_BORE_CODE,0.00,0.00,0.00,0.00,0.00,0.0,0.00
NPD_WELL_BORE_NAME,0.00,0.00,0.00,0.00,0.00,0.0,0.00
NPD_FIELD_CODE,0.00,0.00,0.00,0.00,0.00,0.0,0.00
NPD_FIELD_NAME,0.00,0.00,0.00,0.00,0.00,0.0,0.00
NPD_FACILITY_CODE,0.00,0.00,0.00,0.00,0.00,0.0,0.00
NPD_FACILITY_NAME,0.00,0.00,0.00,0.00,0.00,0.0,0.00
ON_STREAM_HRS,0.00,0.00,4.57,4.02,0.00,0.0,0.00
AVG_DOWNHOLE_PRESSURE,0.20,0.20,100.00,100.00,0.52,0.0,0.40
AVG_DOWNHOLE_TEMPERATURE,0.20,0.20,100.00,100.00,0.52,0.0,0.40


### 11.3 Missingness through time

Are NULLs concentrated in particular periods, or spread evenly? Restricted to the columns that have any missingness at all (rows are years, columns are those columns, values are missing %).


In [95]:
# errors="ignore" here is the same pandas-version accommodation explained
# above (11.2) - "year" is always the groupby key, dropped defensively.
missingness_by_year = (
    daily_df[columns_with_missing]
    .assign(year=year_series)
    .groupby("year")
    .apply(lambda g: (g.drop(columns="year", errors="ignore").isna().mean() * 100).round(2))
)
missingness_by_year


,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,AVG_DOWNHOLE_TEMPERATURE,AVG_DP_TUBING,AVG_ANNULUS_PRESS,AVG_CHOKE_SIZE_P,AVG_CHOKE_UOM,AVG_WHP_P,AVG_WHT_P,DP_CHOKE_SIZE,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL
year,,,,,,,,,,,,,,
2007,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
2008,1.53,53.20,53.20,53.20,63.30,64.97,53.20,53.20,53.20,1.74,53.20,53.20,53.20,73.18
2009,0.00,51.03,51.03,51.03,73.11,52.06,50.21,50.21,50.21,0.00,50.21,50.21,50.21,53.65
2010,0.00,50.48,50.48,50.48,54.91,52.01,50.48,50.48,50.48,0.00,50.48,50.48,50.48,52.63
2011,0.00,52.07,52.07,52.07,52.07,54.14,52.07,52.07,52.07,0.00,52.07,52.07,52.07,51.78
2012,0.00,52.29,52.29,52.29,52.29,52.29,52.29,52.29,52.29,0.00,52.29,52.29,52.29,47.71
2013,0.00,45.54,45.54,45.54,45.54,45.30,45.17,45.54,45.54,0.37,45.17,45.17,45.17,54.83
2014,0.00,30.04,30.04,30.04,40.29,29.92,29.92,29.92,29.92,0.00,29.92,29.92,29.92,70.08
2015,0.00,28.57,28.57,28.57,42.86,28.57,28.57,28.57,28.57,0.00,28.57,28.57,28.57,71.43


### 11.4 Missingness by operating context

`WELL_TYPE`, `FLOW_KIND`, and `ON_STREAM_HRS` status. This is the level most likely to explain the patterns above with a legitimate operational reason rather than a data-quality defect — Section 12 will investigate the categorical context (`WELL_TYPE`/`FLOW_KIND`) in more depth using this as a starting point.


In [96]:
# errors="ignore" here is the same pandas-version accommodation explained
# above (11.2) - "WELL_TYPE" is always the groupby key, dropped defensively.
missingness_by_well_type = (
    daily_df[columns_with_missing]
    .assign(WELL_TYPE=daily_df["WELL_TYPE"])
    .groupby("WELL_TYPE")
    .apply(lambda g: (g.drop(columns="WELL_TYPE", errors="ignore").isna().mean() * 100).round(2))
)
missingness_by_well_type


,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,AVG_DOWNHOLE_TEMPERATURE,AVG_DP_TUBING,AVG_ANNULUS_PRESS,AVG_CHOKE_SIZE_P,AVG_CHOKE_UOM,AVG_WHP_P,AVG_WHT_P,DP_CHOKE_SIZE,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL
WELL_TYPE,,,,,,,,,,,,,,
OP,0.00,1.80,1.80,1.80,13.89,2.65,0.00,0.07,0.15,0.07,0.00,0.00,0.00,100.00
WI,4.39,99.97,99.97,99.97,99.74,99.72,99.72,99.72,99.74,4.44,99.72,99.72,99.72,12.09


In [97]:
# errors="ignore" here is the same pandas-version accommodation explained
# above (11.2) - "FLOW_KIND" is always the groupby key, dropped defensively.
missingness_by_flow_kind = (
    daily_df[columns_with_missing]
    .assign(FLOW_KIND=daily_df["FLOW_KIND"])
    .groupby("FLOW_KIND")
    .apply(lambda g: (g.drop(columns="FLOW_KIND", errors="ignore").isna().mean() * 100).round(2))
)
missingness_by_flow_kind


,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,AVG_DOWNHOLE_TEMPERATURE,AVG_DP_TUBING,AVG_ANNULUS_PRESS,AVG_CHOKE_SIZE_P,AVG_CHOKE_UOM,AVG_WHP_P,AVG_WHT_P,DP_CHOKE_SIZE,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL
FLOW_KIND,,,,,,,,,,,,,,
injection,4.4,100.00,100.00,100.00,100.00,100.00,100.0,100.00,100.00,4.45,100.0,100.0,100.0,12.08
production,0.0,1.98,1.98,1.98,13.87,2.64,0.0,0.07,0.16,0.07,0.0,0.0,0.0,99.84


In [98]:
# ON_STREAM_HRS status buckets, consistent with Section 9's presence
# classification: active (>0), inactive (==0), hours_missing (NULL).
# errors="ignore" here is the same pandas-version accommodation explained
# above (11.2) - "hours_status" is always the groupby key, dropped defensively.
hours_status = pd.Series("active", index=daily_df.index)
hours_status[hours == 0] = "inactive"
hours_status[hours.isna()] = "hours_missing"

missingness_by_hours_status = (
    daily_df[columns_with_missing]
    .assign(hours_status=hours_status)
    .groupby("hours_status")
    .apply(lambda g: (g.drop(columns="hours_status", errors="ignore").isna().mean() * 100).round(2))
)
missingness_by_hours_status


,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,AVG_DOWNHOLE_TEMPERATURE,AVG_DP_TUBING,AVG_ANNULUS_PRESS,AVG_CHOKE_SIZE_P,AVG_CHOKE_UOM,AVG_WHP_P,AVG_WHT_P,DP_CHOKE_SIZE,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL
hours_status,,,,,,,,,,,,,,
active,0.0,41.21,41.21,41.21,47.23,40.14,40.14,40.16,40.16,0.03,40.14,40.14,40.14,59.80
hours_missing,100.0,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
inactive,0.0,43.44,43.44,43.44,57.94,53.94,41.55,41.70,42.16,0.26,41.55,41.55,41.55,83.56


### Preliminary missingness classification

Data-driven, not asserted: each column's classification is derived from the evidence gathered above (0% missing, association with `WELL_TYPE`, or neither), then a small number of known structural/measurement fields are set explicitly. **Preliminary — an investigation aid, to be revisited in Sections 12-17.**


In [99]:
structural_columns = {
    "DATEPRD", "WELL_BORE_CODE", "NPD_WELL_BORE_CODE", "NPD_WELL_BORE_NAME",
    "NPD_FIELD_CODE", "NPD_FIELD_NAME", "NPD_FACILITY_CODE", "NPD_FACILITY_NAME",
    "WELL_TYPE", "FLOW_KIND",
}

classification_rows = []
for col in daily_df.columns:
    missing_pct = round(daily_df[col].isna().mean() * 100, 2)

    if col in structural_columns:
        category = "STRUCTURAL"
        rationale = "Identifier/date/operating-mode field that other context in this section depends on."
    elif col == "ON_STREAM_HRS":
        category = "MEASUREMENT"
        rationale = "Operating-time measurement; missingness fully characterized as DQ-003 (Section 10.2)."
    elif missing_pct == 0:
        category = "STRUCTURAL"
        rationale = "No missing values observed."
    else:
        wt_spread = (
            missingness_by_well_type[col].max() - missingness_by_well_type[col].min()
            if col in missingness_by_well_type.columns else 0
        )
        fk_spread = (
            missingness_by_flow_kind[col].max() - missingness_by_flow_kind[col].min()
            if col in missingness_by_flow_kind.columns else 0
        )
        if wt_spread > 50 or fk_spread > 50:
            category = "CONTEXTUAL"
            rationale = f"Missingness varies sharply by WELL_TYPE/FLOW_KIND (spread {max(wt_spread, fk_spread):.1f}pp) - likely tied to operating mode. Preliminary."
        else:
            category = "MEASUREMENT"
            rationale = f"Missingness present without a sharp operating-mode split (spread {max(wt_spread, fk_spread):.1f}pp) - looks like a sensor/reporting gap. Preliminary."

    classification_rows.append(
        {"column": col, "missing_pct": missing_pct, "preliminary_classification": category, "rationale": rationale}
    )

missingness_classification = pd.DataFrame(classification_rows).sort_values("missing_pct", ascending=False).reset_index(drop=True)
missingness_classification


,column,missing_pct,preliminary_classification,rationale
0,BORE_WI_VOL,63.50,CONTEXTUAL,Missingness varies sharply by WELL_TYPE/FLOW_K...
1,AVG_ANNULUS_PRESS,49.53,CONTEXTUAL,Missingness varies sharply by WELL_TYPE/FLOW_K...
2,AVG_CHOKE_SIZE_P,42.95,CONTEXTUAL,Missingness varies sharply by WELL_TYPE/FLOW_K...
3,AVG_DOWNHOLE_PRESSURE,42.56,CONTEXTUAL,Missingness varies sharply by WELL_TYPE/FLOW_K...
4,AVG_DOWNHOLE_TEMPERATURE,42.56,CONTEXTUAL,Missingness varies sharply by WELL_TYPE/FLOW_K...
5,AVG_DP_TUBING,42.56,CONTEXTUAL,Missingness varies sharply by WELL_TYPE/FLOW_K...
6,AVG_WHT_P,41.50,CONTEXTUAL,Missingness varies sharply by WELL_TYPE/FLOW_K...
7,AVG_WHP_P,41.44,CONTEXTUAL,Missingness varies sharply by WELL_TYPE/FLOW_K...
8,BORE_WAT_VOL,41.40,CONTEXTUAL,Missingness varies sharply by WELL_TYPE/FLOW_K...
9,BORE_GAS_VOL,41.40,CONTEXTUAL,Missingness varies sharply by WELL_TYPE/FLOW_K...


### NULL ≠ 0, demonstrated concretely

For the four volume columns: NULL, zero, positive, and negative counts, kept fully separate.


In [100]:
null_vs_zero_rows = []
for col in ["BORE_OIL_VOL", "BORE_GAS_VOL", "BORE_WAT_VOL", "BORE_WI_VOL"]:
    s = daily_df[col]
    null_vs_zero_rows.append(
        {
            "column": col,
            "null_count": int(s.isna().sum()),
            "zero_count": int((s == 0).sum()),
            "positive_count": int((s > 0).sum()),
            "negative_count": int((s < 0).sum()),
        }
    )

null_vs_zero = pd.DataFrame(null_vs_zero_rows)
null_vs_zero


,column,null_count,zero_count,positive_count,negative_count
0,BORE_OIL_VOL,6473,1153,8008,0
1,BORE_GAS_VOL,6473,1150,8011,0
2,BORE_WAT_VOL,6473,1526,7631,4
3,BORE_WI_VOL,9928,302,5404,0


In [101]:
# BORE_WI_VOL specifically, split by WELL_TYPE - makes the point concretely:
# NULL and zero mean different things, and which one is common depends on context.
wi = daily_df["BORE_WI_VOL"]
wi_status = pd.Series("other", index=daily_df.index)
wi_status[wi.isna()] = "NULL"
wi_status[wi == 0] = "zero"
wi_status[wi > 0] = "positive"
wi_status[wi < 0] = "negative"

wi_null_vs_zero_by_type = (
    daily_df.assign(wi_status=wi_status).groupby(["WELL_TYPE", "wi_status"]).size().unstack(fill_value=0)
)
wi_null_vs_zero_by_type


wi_status,NULL,positive,zero
WELL_TYPE,,,
OP,9143,0,0
WI,785,5404,302


### Section 11 summary

More informative than a bare missing %: a pattern description and status per column, derived from the evidence above. `ON_STREAM_HRS` carries its Section 10 issue ID directly rather than a generic status.


In [102]:
def describe_pattern(col: str) -> tuple[str, str]:
    s = daily_df[col]
    if s.isna().sum() == 0:
        return "none", "PASS"
    if col == "ON_STREAM_HRS":
        return "5693/5769 + early records", "DQ-003"

    missing_mask = s.isna()
    total_missing = int(missing_mask.sum())

    wt_spread = (
        missingness_by_well_type[col].max() - missingness_by_well_type[col].min()
        if col in missingness_by_well_type.columns else 0
    )
    fk_spread = (
        missingness_by_flow_kind[col].max() - missingness_by_flow_kind[col].min()
        if col in missingness_by_flow_kind.columns else 0
    )

    per_wellbore_missing_counts = daily_df.loc[missing_mask, "NPD_WELL_BORE_CODE"].value_counts()
    top2_row_share = per_wellbore_missing_counts.head(2).sum() / total_missing if total_missing else 0

    per_year_missing_counts = year_series[missing_mask].value_counts()
    top_year_share = per_year_missing_counts.head(1).sum() / total_missing if total_missing else 0

    if wt_spread > 50 or fk_spread > 50:
        return "associated with operating mode (WELL_TYPE/FLOW_KIND)", "REVIEW"
    elif top2_row_share > 0.7 and top_year_share > 0.5:
        return "concentrated by well/time", "REVIEW"
    elif top2_row_share > 0.7:
        return "concentrated by well", "REVIEW"
    elif top_year_share > 0.5:
        return "concentrated by time period", "REVIEW"
    else:
        return "spread broadly, no single dominant driver", "REVIEW"


summary_rows = []
for col in daily_df.columns:
    missing_pct = round(daily_df[col].isna().mean() * 100, 2)
    pattern, status = describe_pattern(col)
    summary_rows.append({"column": col, "missing_pct": missing_pct, "pattern": pattern, "status": status})

missingness_summary_table = pd.DataFrame(summary_rows).sort_values("missing_pct", ascending=False).reset_index(drop=True)
missingness_summary_table


,column,missing_pct,pattern,status
0,BORE_WI_VOL,63.50,associated with operating mode (WELL_TYPE/FLOW...,REVIEW
1,AVG_ANNULUS_PRESS,49.53,associated with operating mode (WELL_TYPE/FLOW...,REVIEW
2,AVG_CHOKE_SIZE_P,42.95,associated with operating mode (WELL_TYPE/FLOW...,REVIEW
3,AVG_DOWNHOLE_PRESSURE,42.56,associated with operating mode (WELL_TYPE/FLOW...,REVIEW
4,AVG_DOWNHOLE_TEMPERATURE,42.56,associated with operating mode (WELL_TYPE/FLOW...,REVIEW
5,AVG_DP_TUBING,42.56,associated with operating mode (WELL_TYPE/FLOW...,REVIEW
6,AVG_WHT_P,41.50,associated with operating mode (WELL_TYPE/FLOW...,REVIEW
7,AVG_WHP_P,41.44,associated with operating mode (WELL_TYPE/FLOW...,REVIEW
8,BORE_WAT_VOL,41.40,associated with operating mode (WELL_TYPE/FLOW...,REVIEW
9,BORE_GAS_VOL,41.40,associated with operating mode (WELL_TYPE/FLOW...,REVIEW


In [103]:
status_counts = missingness_summary_table["status"].value_counts()
print("SECTION 11 - MISSING-VALUE ASSESSMENT SUMMARY")
print("=" * 60)
for status, n in status_counts.items():
    print(f"{status}: {n} column(s)")


SECTION 11 - MISSING-VALUE ASSESSMENT SUMMARY
REVIEW: 13 column(s)
PASS: 10 column(s)
DQ-003: 1 column(s)


**Interpretation for later relational modelling:**

Of 24 daily columns, 14 have any missingness. The data-driven classification (not asserted, derived from the actual WELL_TYPE/FLOW_KIND spread computed in 11.4) puts:

- **10 columns as STRUCTURAL**, all at 0% missing (`PASS`): every identifier, `DATEPRD`, and the two operating-mode fields `WELL_TYPE`/`FLOW_KIND`. Integrity here was already confirmed in Sections 4-8.
- **12 columns as CONTEXTUAL**, all `REVIEW`, all driven by a WELL_TYPE/FLOW_KIND spread far above the 50pp threshold. These split cleanly into two opposite directions: 11 of them (`AVG_DOWNHOLE_PRESSURE`, `AVG_DOWNHOLE_TEMPERATURE`, `AVG_DP_TUBING`, `AVG_ANNULUS_PRESS`, `AVG_CHOKE_SIZE_P`, `AVG_CHOKE_UOM`, `AVG_WHP_P`, `AVG_WHT_P`, `BORE_OIL_VOL`, `BORE_GAS_VOL`, `BORE_WAT_VOL`) are ~0-14% missing on `WELL_TYPE = OP` rows but 99.7-100% missing on `WELL_TYPE = WI` rows — i.e. producer-associated, matching the "associated with producers" pattern this section set out to test. `BORE_WI_VOL` is the mirror opposite: 100% missing on OP rows (9,143/9,143) but only ~12% missing on WI rows — injector-associated. This is a coherent, physically sensible split (producers get downhole/wellhead sensor readings and oil/gas/water volumes; injectors get an injection volume instead of those), not 12 independent coincidences.
- **2 columns as MEASUREMENT**: `DP_CHOKE_SIZE` (1.88%, pattern "concentrated by well/time", not operating-mode-driven) and `ON_STREAM_HRS` (1.82%, carrying its own `DQ-003` status rather than a generic one).
- **0 columns landed in UNKNOWN** — the WELL_TYPE/FLOW_KIND split explained every column with missingness cleanly enough to avoid that bucket this round, though it remains available for future datasets where the split isn't this clean.

**A finding beyond what was asked for:** the `hours_missing` bucket in 11.4 (the 285 rows with NULL `ON_STREAM_HRS`) shows **100% missingness across every other measurement column too** — not just the four volume columns already checked in Section 10.6, but every downhole/wellhead/choke sensor reading as well. These 285 rows (wellbores 5693/5769 only, per Section 10) are essentially blank across the board, which strengthens rather than changes DQ-003's existing characterization.

**NULL ≠ 0 is demonstrated concretely, not just asserted:** `BORE_WAT_VOL` alone carries 6,473 NULL, 1,526 zero, 7,631 positive, *and* 4 negative values — four distinct, non-interchangeable populations. The `BORE_WI_VOL` × `WELL_TYPE` breakdown is the clearest illustration of why context matters: on `OP` rows it is 9,143 NULL / 0 zero / 0 positive (never populated, cleanly contextual), while on `WI` rows it is 785 NULL / 302 zero / 5,404 positive (all three states genuinely occur). Collapsing NULL and 0 together anywhere in this dataset would erase that distinction and misrepresent both a "not applicable" well and a "recorded zero injection" well as the same thing.

**One number worth flagging for Section 14, not resolved here:** the WI-row NULL count for `BORE_WI_VOL` here is 785 (grouped by `WELL_TYPE = WI`, 6,491 rows), which differs slightly from the 782-injection-row figure noted during source exploration (grouped by `FLOW_KIND = injection`, 6,473 rows) — the two groupings aren't identical populations (18 rows are `WELL_TYPE = WI` but `FLOW_KIND = production`). Section 14 (injection-volume validation) is the right place to reconcile which grouping is authoritative.

For relational modelling: the CONTEXTUAL columns are strong candidates for staying nullable in the fact table exactly as-is (NULL correctly represents "not applicable to this operating mode," not missing data), while the classification here remains preliminary until Section 12 formally investigates `WELL_TYPE`/`FLOW_KIND` as the mechanism, and Sections 13-17 validate each measurement individually. Handing off to **Section 12: categorical validation**, which is where this section's operating-mode hypothesis gets tested directly.


## 12. Categorical-value validation

**Central question:** do `WELL_TYPE` and `FLOW_KIND` represent stable wellbore attributes, daily operational states, reporting classifications, or some combination of these?

This section explains several findings from Sections 9-11 rather than introducing new ones: the `WELL_TYPE` instability first noticed in Section 9 (wellbores 5693/5769 showing both `OP` and `WI`), and the 12 CONTEXTUAL columns Section 11 inferred are driven by operating mode. Nine checks: category inventory, `WELL_TYPE` by wellbore, `WELL_TYPE` transitions, `FLOW_KIND` stability, the `WELL_TYPE`×`FLOW_KIND` contingency table, category × physical activity, missingness by category (reconnecting to Section 11), `AVG_CHOKE_UOM` unit consistency, and a final attribute classification (STATIC / TEMPORAL / UNCLEAR) that feeds directly into Section 24's dimension design.


In [104]:
# --- TEMPORARY LOADING FOR SECTION 12 ---
# Section 1 (Load source data) has not been implemented yet in this notebook.
# This block loads only what Section 12 needs, and is clearly marked so it can
# be deleted once Section 1 provides `daily_df` for the whole notebook.
if "daily_df" not in globals():
    from pathlib import Path
    import pandas as pd

    PROJECT_ROOT = Path.cwd().parent
    WORKBOOK_PATH = PROJECT_ROOT / "data" / "raw" / "Volve production data.xlsx"
    DAILY_SHEET_NAME = "Daily Production Data"

    if not WORKBOOK_PATH.exists():
        raise FileNotFoundError(f"Source workbook not found at {WORKBOOK_PATH}")

    daily_df = pd.read_excel(WORKBOOK_PATH, sheet_name=DAILY_SHEET_NAME)
    print(f"[Section 12 temporary load] daily_df loaded: {daily_df.shape}")
else:
    print(f"Using daily_df already loaded earlier in the notebook: {daily_df.shape}")


Using daily_df already loaded earlier in the notebook: (15634, 24)


In [105]:
# Reuse Section 9-11 derived state if available, recompute the minimum if not.
from collections import Counter

if "dateprd_parsed" not in globals():
    dateprd_parsed = pd.to_datetime(daily_df["DATEPRD"], errors="coerce")
if "daily_with_dates" not in globals():
    daily_with_dates = daily_df.copy()
    daily_with_dates["DATEPRD_parsed"] = dateprd_parsed

print("Ready: daily_with_dates, dateprd_parsed")


Ready: daily_with_dates, dateprd_parsed


In [106]:
required_columns = [
    "NPD_WELL_BORE_CODE", "NPD_WELL_BORE_NAME", "WELL_TYPE", "FLOW_KIND", "AVG_CHOKE_UOM", "DATEPRD",
    "BORE_OIL_VOL", "BORE_GAS_VOL", "BORE_WAT_VOL", "BORE_WI_VOL",
]
missing_columns = [c for c in required_columns if c not in daily_df.columns]
if missing_columns:
    raise ValueError(f"Required column(s) missing from daily_df: {missing_columns}")
print(f"Required columns present: {required_columns}")


Required columns present: ['NPD_WELL_BORE_CODE', 'NPD_WELL_BORE_NAME', 'WELL_TYPE', 'FLOW_KIND', 'AVG_CHOKE_UOM', 'DATEPRD', 'BORE_OIL_VOL', 'BORE_GAS_VOL', 'BORE_WAT_VOL', 'BORE_WI_VOL']


### 12.1 Category inventory

Raw distinct values, counts, percentages, and NULL counts for `WELL_TYPE`, `FLOW_KIND`, `AVG_CHOKE_UOM` — plus a check for whether trimming whitespace or case-folding would collapse apparently different categories together. Nothing is actually modified; this only tests whether normalization *would* change anything.


In [107]:
def category_inventory(col: str) -> pd.DataFrame:
    raw_counts = daily_df[col].value_counts(dropna=False)
    pct = (raw_counts / len(daily_df) * 100).round(2)
    inventory = pd.DataFrame({"count": raw_counts, "pct": pct})
    inventory.index.name = col
    return inventory.reset_index()


for col in ["WELL_TYPE", "FLOW_KIND", "AVG_CHOKE_UOM"]:
    print(f"--- {col} ---")
    print(category_inventory(col).to_string(index=False))
    print()


--- WELL_TYPE ---
WELL_TYPE  count   pct
       OP   9143 58.48
       WI   6491 41.52

--- FLOW_KIND ---
 FLOW_KIND  count  pct
production   9161 58.6
 injection   6473 41.4

--- AVG_CHOKE_UOM ---
AVG_CHOKE_UOM  count  pct
            %   9161 58.6
          NaN   6473 41.4



In [108]:
normalization_check_rows = []
for col in ["WELL_TYPE", "FLOW_KIND", "AVG_CHOKE_UOM"]:
    values = daily_df[col].dropna().astype(str)
    raw_nunique = values.nunique()
    trimmed_nunique = values.str.strip().nunique()
    casefolded_nunique = values.str.strip().str.casefold().nunique()
    normalization_check_rows.append(
        {
            "column": col,
            "raw_nunique": raw_nunique,
            "trimmed_nunique": trimmed_nunique,
            "trimmed_casefolded_nunique": casefolded_nunique,
            "would_normalization_collapse_categories": (trimmed_nunique < raw_nunique) or (casefolded_nunique < trimmed_nunique),
        }
    )

normalization_check = pd.DataFrame(normalization_check_rows)
normalization_check


,column,raw_nunique,trimmed_nunique,trimmed_casefolded_nunique,would_normalization_collapse_categories
0,WELL_TYPE,2,2,2,False
1,FLOW_KIND,2,2,2,False
2,AVG_CHOKE_UOM,1,1,1,False


### 12.2 `WELL_TYPE` by wellbore, and 12.3 `WELL_TYPE` transitions

One helper computes both the per-wellbore detail/summary (12.2) and the chronological transition analysis (12.3) for a categorical column, so the identical method can be reused for `FLOW_KIND` in 12.4 without duplicating (and risking divergent bugs in) the same ~30 lines of logic twice.


In [109]:
def categorical_stability_report(col: str):
    detail = (
        daily_with_dates.groupby(["NPD_WELL_BORE_CODE", col])
        .agg(
            wellbore_name=("NPD_WELL_BORE_NAME", "first"),
            count=(col, "size"),
            first_date=("DATEPRD_parsed", "min"),
            last_date=("DATEPRD_parsed", "max"),
        )
        .reset_index()
        .sort_values(["NPD_WELL_BORE_CODE", "first_date"])
        .reset_index(drop=True)
    )

    summary = (
        detail.groupby("NPD_WELL_BORE_CODE")
        .agg(
            wellbore_name=("wellbore_name", "first"),
            distinct_count=(col, "nunique"),
            values=(col, lambda s: sorted(s.unique().tolist())),
        )
        .reset_index()
    )
    summary["status"] = summary["distinct_count"].apply(lambda n: "REVIEW" if n > 1 else "PASS")

    transition_counter = Counter()
    transition_rows = []
    for code_value, group in daily_with_dates.groupby("NPD_WELL_BORE_CODE"):
        g = group.dropna(subset=["DATEPRD_parsed"]).sort_values("DATEPRD_parsed")
        prev_vals = g[col].shift(1)
        for p, c, d, name in zip(prev_vals, g[col], g["DATEPRD_parsed"], g["NPD_WELL_BORE_NAME"]):
            if pd.isna(p):
                continue  # first record for this wellbore - no previous state to compare
            transition_counter[f"{p}->{c}"] += 1
            if p != c:
                transition_rows.append(
                    {"npd_code": code_value, "wellbore_name": name, "previous": p, "current": c, "transition_date": d.date()}
                )

    tally = pd.DataFrame([{"transition": k, "count": v} for k, v in sorted(transition_counter.items())])
    review = (
        pd.DataFrame(transition_rows).sort_values(["npd_code", "transition_date"]).reset_index(drop=True)
        if transition_rows
        else pd.DataFrame(columns=["npd_code", "wellbore_name", "previous", "current", "transition_date"])
    )
    return detail, summary, tally, review


well_type_detail, well_type_summary, well_type_tally, well_type_review = categorical_stability_report("WELL_TYPE")
well_type_detail


,NPD_WELL_BORE_CODE,WELL_TYPE,wellbore_name,count,first_date,last_date
0,5351,OP,15/9-F-14,3056,2008-02-12,2016-09-17
1,5599,OP,15/9-F-12,3056,2008-02-12,2016-09-17
2,5693,WI,15/9-F-4,3327,2007-09-01,2016-12-01
3,5769,WI,15/9-F-5,3162,2007-09-01,2016-09-18
4,5769,OP,15/9-F-5,144,2016-04-12,2016-09-17
5,7078,OP,15/9-F-11,1165,2013-07-08,2016-09-17
6,7289,OP,15/9-F-15 D,978,2014-01-12,2016-09-17
7,7405,WI,15/9-F-1 C,2,2014-04-07,2014-07-07
8,7405,OP,15/9-F-1 C,744,2014-04-08,2016-04-21


In [110]:
# 12.2 summary: distinct WELL_TYPE count per wellbore, flagged REVIEW if > 1.
well_type_summary


,NPD_WELL_BORE_CODE,wellbore_name,distinct_count,values,status
0,5351,15/9-F-14,1,[OP],PASS
1,5599,15/9-F-12,1,[OP],PASS
2,5693,15/9-F-4,1,[WI],PASS
3,5769,15/9-F-5,2,"[OP, WI]",REVIEW
4,7078,15/9-F-11,1,[OP],PASS
5,7289,15/9-F-15 D,1,[OP],PASS
6,7405,15/9-F-1 C,2,"[OP, WI]",REVIEW


### 12.3 `WELL_TYPE` transitions

Consecutive-pair tally (including non-changes, as a baseline) followed by a review table containing **only actual changes** — this is what tells us whether a second `WELL_TYPE` value is a one-row anomaly or a meaningful period.


In [111]:
well_type_tally

,transition,count
0,OP->OP,9133
1,OP->WI,5
2,WI->OP,6
3,WI->WI,6483


In [112]:
print(f"Actual WELL_TYPE changes (previous != current): {len(well_type_review)}")
well_type_review


Actual WELL_TYPE changes (previous != current): 11


,npd_code,wellbore_name,previous,current,transition_date
0,5769,15/9-F-5,WI,OP,2016-04-12
1,5769,15/9-F-5,OP,WI,2016-05-04
2,5769,15/9-F-5,WI,OP,2016-05-05
3,5769,15/9-F-5,OP,WI,2016-07-26
4,5769,15/9-F-5,WI,OP,2016-08-02
5,5769,15/9-F-5,OP,WI,2016-09-06
6,5769,15/9-F-5,WI,OP,2016-09-13
7,5769,15/9-F-5,OP,WI,2016-09-18
8,7405,15/9-F-1 C,WI,OP,2014-04-08
9,7405,15/9-F-1 C,OP,WI,2014-07-07


### 12.4 `FLOW_KIND` stability

The same method as 12.2/12.3, applied to `FLOW_KIND`.


In [113]:
flow_kind_detail, flow_kind_summary, flow_kind_tally, flow_kind_review = categorical_stability_report("FLOW_KIND")
flow_kind_detail


,NPD_WELL_BORE_CODE,FLOW_KIND,wellbore_name,count,first_date,last_date
0,5351,production,15/9-F-14,3056,2008-02-12,2016-09-17
1,5599,production,15/9-F-12,3056,2008-02-12,2016-09-17
2,5693,injection,15/9-F-4,3327,2007-09-01,2016-12-01
3,5769,injection,15/9-F-5,3146,2007-09-01,2016-09-18
4,5769,production,15/9-F-5,160,2016-04-11,2016-09-17
5,7078,production,15/9-F-11,1165,2013-07-08,2016-09-17
6,7289,production,15/9-F-15 D,978,2014-01-12,2016-09-17
7,7405,production,15/9-F-1 C,746,2014-04-07,2016-04-21


In [114]:
flow_kind_summary

,NPD_WELL_BORE_CODE,wellbore_name,distinct_count,values,status
0,5351,15/9-F-14,1,[production],PASS
1,5599,15/9-F-12,1,[production],PASS
2,5693,15/9-F-4,1,[injection],PASS
3,5769,15/9-F-5,2,"[injection, production]",REVIEW
4,7078,15/9-F-11,1,[production],PASS
5,7289,15/9-F-15 D,1,[production],PASS
6,7405,15/9-F-1 C,1,[production],PASS


In [115]:
flow_kind_tally

,transition,count
0,injection->injection,6470
1,injection->production,1
2,production->injection,1
3,production->production,9155


In [116]:
print(f"Actual FLOW_KIND changes (previous != current): {len(flow_kind_review)}")
flow_kind_review


Actual FLOW_KIND changes (previous != current): 2


,npd_code,wellbore_name,previous,current,transition_date
0,5769,15/9-F-5,injection,production,2016-04-11
1,5769,15/9-F-5,production,injection,2016-09-18


### 12.5 `WELL_TYPE` × `FLOW_KIND`

Counts and percentages, to discover what these two columns actually encode relative to each other.


In [117]:
contingency_counts = pd.crosstab(daily_df["WELL_TYPE"], daily_df["FLOW_KIND"], margins=True, margins_name="Total")
contingency_counts


FLOW_KIND,injection,production,Total
WELL_TYPE,,,
OP,0,9143,9143
WI,6473,18,6491
Total,6473,9161,15634


In [118]:
contingency_pct = (pd.crosstab(daily_df["WELL_TYPE"], daily_df["FLOW_KIND"], normalize="all") * 100).round(2)
contingency_pct


FLOW_KIND,injection,production
WELL_TYPE,,
OP,0.0,58.48
WI,41.4,0.12


### 12.6 Category × physical activity

For every `WELL_TYPE` × `FLOW_KIND` combination: row count, rows with each volume positive, rows with all four volumes exactly zero, and rows with all four volumes NULL.


In [119]:
volume_cols = ["BORE_OIL_VOL", "BORE_GAS_VOL", "BORE_WAT_VOL", "BORE_WI_VOL"]


def activity_row(g: pd.DataFrame) -> pd.Series:
    return pd.Series(
        {
            "row_count": len(g),
            "oil_gt_0": int((g["BORE_OIL_VOL"] > 0).sum()),
            "gas_gt_0": int((g["BORE_GAS_VOL"] > 0).sum()),
            "water_gt_0": int((g["BORE_WAT_VOL"] > 0).sum()),
            "water_injection_gt_0": int((g["BORE_WI_VOL"] > 0).sum()),
            "all_volumes_zero": int((g[volume_cols] == 0).all(axis=1).sum()),
            "all_volumes_null": int(g[volume_cols].isna().all(axis=1).sum()),
        }
    )


category_activity = (
    daily_df.groupby(["WELL_TYPE", "FLOW_KIND"], dropna=False).apply(activity_row).reset_index()
)
category_activity


,WELL_TYPE,FLOW_KIND,row_count,oil_gt_0,gas_gt_0,water_gt_0,water_injection_gt_0,all_volumes_zero,all_volumes_null
0,OP,production,9143,7999,8002,7622,0,0,0
1,WI,injection,6473,0,0,0,5404,0,782
2,WI,production,18,9,9,9,0,7,0


### 12.7 Missingness by category

Reconnecting to Section 11: do the 12 CONTEXTUAL columns identified there actually follow `WELL_TYPE`×`FLOW_KIND` combinations? Quantified, not just described.


In [120]:
if "missingness_classification" in globals():
    contextual_columns = missingness_classification.loc[
        missingness_classification["preliminary_classification"] == "CONTEXTUAL", "column"
    ].tolist()
    print("Using CONTEXTUAL column list computed live in Section 11.")
else:
    # Fallback matching Section 11's result, only used if Section 11 has not run
    # in this kernel session - documented here rather than silently recomputed.
    contextual_columns = [
        "AVG_DOWNHOLE_PRESSURE", "AVG_DOWNHOLE_TEMPERATURE", "AVG_DP_TUBING", "AVG_ANNULUS_PRESS",
        "AVG_CHOKE_SIZE_P", "AVG_CHOKE_UOM", "AVG_WHP_P", "AVG_WHT_P",
        "BORE_OIL_VOL", "BORE_GAS_VOL", "BORE_WAT_VOL", "BORE_WI_VOL",
    ]
    print("Section 11's missingness_classification not found in this session - using its recorded result as a fallback.")

print(f"CONTEXTUAL columns carried forward: {len(contextual_columns)}")
print(contextual_columns)


Using CONTEXTUAL column list computed live in Section 11.
CONTEXTUAL columns carried forward: 12
['BORE_WI_VOL', 'AVG_ANNULUS_PRESS', 'AVG_CHOKE_SIZE_P', 'AVG_DOWNHOLE_PRESSURE', 'AVG_DOWNHOLE_TEMPERATURE', 'AVG_DP_TUBING', 'AVG_WHT_P', 'AVG_WHP_P', 'BORE_WAT_VOL', 'BORE_GAS_VOL', 'BORE_OIL_VOL', 'AVG_CHOKE_UOM']


In [121]:
# errors="ignore" here is the same pandas-version accommodation documented in
# Section 11.2 - WELL_TYPE/FLOW_KIND are always the groupby keys, dropped defensively.
missingness_by_combo = (
    daily_df[contextual_columns]
    .assign(WELL_TYPE=daily_df["WELL_TYPE"], FLOW_KIND=daily_df["FLOW_KIND"])
    .groupby(["WELL_TYPE", "FLOW_KIND"], dropna=False)
    .apply(lambda g: (g.drop(columns=["WELL_TYPE", "FLOW_KIND"], errors="ignore").isna().mean() * 100).round(2))
)
missingness_by_combo


BORE_WI_VOL  AVG_ANNULUS_PRESS  AVG_CHOKE_SIZE_P  \
WELL_TYPE FLOW_KIND                                                      
OP        production       100.00              13.89              2.65   
WI        injection         12.08             100.00            100.00   
          production        16.67               5.56              0.00   

                      AVG_DOWNHOLE_PRESSURE  AVG_DOWNHOLE_TEMPERATURE  \
WELL_TYPE FLOW_KIND                                                     
OP        production                   1.80                      1.80   
WI        injection                  100.00                    100.00   
          production                  88.89                     88.89   

                      AVG_DP_TUBING  AVG_WHT_P  AVG_WHP_P  BORE_WAT_VOL  \
WELL_TYPE FLOW_KIND                                                       
OP        production           1.80       0.15       0.07           0.0   
WI        injection          100.00     100.00     100.00         100.0   
          production          88.89       5.56       0.00           0.0   

                      BORE_GAS_VOL  BORE_OIL_VOL  AVG_CHOKE_UOM  
WELL_TYPE FLOW_KIND                                              
OP        production           0.0           0.0            0.0  
WI        injection          100.0         100.0          100.0  
          production           0.0           0.0            0.0

In [122]:
# Quantify how cleanly each CONTEXTUAL column's missingness is separated by
# combo, rather than just describing it: "cleanly separated" means every
# WELL_TYPE x FLOW_KIND combo is either under 1% or over 99% missing for
# that column - i.e. essentially all-present or all-absent per combo.
combo_cleanliness_rows = []
for col in contextual_columns:
    vals = missingness_by_combo[col]
    is_clean_split = bool(((vals < 1) | (vals > 99)).all())
    combo_cleanliness_rows.append(
        {
            "column": col,
            "min_pct_across_combos": vals.min(),
            "max_pct_across_combos": vals.max(),
            "cleanly_separated_by_combo": is_clean_split,
        }
    )

combo_cleanliness = pd.DataFrame(combo_cleanliness_rows)
combo_cleanliness


,column,min_pct_across_combos,max_pct_across_combos,cleanly_separated_by_combo
0,BORE_WI_VOL,12.08,100.0,False
1,AVG_ANNULUS_PRESS,5.56,100.0,False
2,AVG_CHOKE_SIZE_P,0.00,100.0,False
3,AVG_DOWNHOLE_PRESSURE,1.80,100.0,False
4,AVG_DOWNHOLE_TEMPERATURE,1.80,100.0,False
5,AVG_DP_TUBING,1.80,100.0,False
6,AVG_WHT_P,0.15,100.0,False
7,AVG_WHP_P,0.00,100.0,True
8,BORE_WAT_VOL,0.00,100.0,True
9,BORE_GAS_VOL,0.00,100.0,True


### 12.8 Unit consistency (`AVG_CHOKE_UOM`)


In [123]:
choke_uom_non_null = daily_df["AVG_CHOKE_UOM"].dropna().astype(str)

print(f"Distinct raw AVG_CHOKE_UOM values: {sorted(choke_uom_non_null.unique().tolist())}")
print(f"NULL AVG_CHOKE_UOM count: {int(daily_df['AVG_CHOKE_UOM'].isna().sum())}")
print(f"Raw nunique:                 {choke_uom_non_null.nunique()}")
print(f"Trimmed nunique:             {choke_uom_non_null.str.strip().nunique()}")
print(f"Trimmed+casefolded nunique:  {choke_uom_non_null.str.strip().str.casefold().nunique()}")


Distinct raw AVG_CHOKE_UOM values: ['%']
NULL AVG_CHOKE_UOM count: 6473
Raw nunique:                 1
Trimmed nunique:             1
Trimmed+casefolded nunique:  1


In [124]:
uom_by_wellbore = daily_df.groupby("NPD_WELL_BORE_CODE")["AVG_CHOKE_UOM"].agg(lambda s: sorted(s.dropna().unique().tolist()))
print("Distinct AVG_CHOKE_UOM value(s) observed per wellbore:")
uom_by_wellbore


Distinct AVG_CHOKE_UOM value(s) observed per wellbore:


NPD_WELL_BORE_CODE
5351    [%]
5599    [%]
5693     []
5769    [%]
7078    [%]
7289    [%]
7405    [%]
Name: AVG_CHOKE_UOM, dtype: object

In [125]:
choke_uom_is_constant_where_present = bool(choke_uom_non_null.nunique() <= 1)
print(f"AVG_CHOKE_UOM takes exactly one non-null value wherever it is present: {choke_uom_is_constant_where_present}")


AVG_CHOKE_UOM takes exactly one non-null value wherever it is present: True


### 12.9 Attribute classification

Based on the transition evidence above (12.3, 12.4) and the unit-consistency evidence (12.8) — not asserted from column names.


In [126]:
well_type_classification = "TEMPORAL" if len(well_type_review) > 0 else "STATIC"
flow_kind_classification = "TEMPORAL" if len(flow_kind_review) > 0 else "STATIC"
choke_uom_classification = "STATIC" if choke_uom_is_constant_where_present else "UNCLEAR"

attribute_classification = pd.DataFrame(
    [
        {
            "attribute": "WELL_TYPE",
            "classification": well_type_classification,
            "evidence": f"{len(well_type_review)} actual transition(s) observed across wellbores (12.3)",
        },
        {
            "attribute": "FLOW_KIND",
            "classification": flow_kind_classification,
            "evidence": f"{len(flow_kind_review)} actual transition(s) observed across wellbores (12.4)",
        },
        {
            "attribute": "AVG_CHOKE_UOM",
            "classification": choke_uom_classification,
            "evidence": f"{choke_uom_non_null.nunique()} distinct non-null unit value(s) observed (12.8)",
        },
    ]
)
attribute_classification


,attribute,classification,evidence
0,WELL_TYPE,TEMPORAL,11 actual transition(s) observed across wellbo...
1,FLOW_KIND,TEMPORAL,2 actual transition(s) observed across wellbor...
2,AVG_CHOKE_UOM,STATIC,1 distinct non-null unit value(s) observed (12.8)


**This notebook is now distinguishing four kinds of attribute**, which matters directly for Section 24's schema design:

- **Stable identity** — `NPD_WELL_BORE_CODE`, `NPD_WELL_BORE_NAME`, `WELL_BORE_CODE` (confirmed Section 6).
- **Stable context** — field, facility (confirmed Section 8; single value for the whole dataset).
- **Potential temporal state** — `WELL_TYPE`, `FLOW_KIND` (tested here; see 12.9's classification for whether that potential is realized).
- **Measurements** — pressures, temperatures, volumes, hours (validated individually in Sections 13-17).

If `WELL_TYPE` classifies as TEMPORAL, putting a single `WELL_TYPE` value on a future wellbore dimension row would destroy real information — it would need to live on the daily fact table (where it already does) or a type-2 history table, not a flat one-row-per-wellbore dimension.


**Interpretation for later relational modelling:**

**Central question answered: `WELL_TYPE` and `FLOW_KIND` are TEMPORAL, not stable wellbore attributes.** Both classify as TEMPORAL in 12.9, confirmed by real transition evidence, not column-name assumption.

**12.2/12.3 — exactly 2 of 7 wellbores show `WELL_TYPE` variation** (5769, 7405), matching Section 9's original finding precisely. Neither is a one-row anomaly:
- **5769** oscillates through a genuine multi-month period in 2016 (11 transitions total: OP↔WI repeatedly between 2016-04-12 and 2016-09-18) — consistent with a real operational conversion/testing period, not noise.
- **7405** flips WI→OP once, in its first 2 days on record (2014-04-07 to 2014-04-08) — a brief early-life classification settling, not a recurring pattern.

**12.4 — only 1 of 7 wellbores (5769) shows `FLOW_KIND` variation**, and only 2 transitions (injection→production 2016-04-11, production→injection 2016-09-18) — a single coarse ~5-month window. `WELL_TYPE` oscillates *inside* that window 11 times while `FLOW_KIND` stays flat at "production" throughout it. The two columns are tracking related but different things: `FLOW_KIND` looks like the coarser operational classification for a period, `WELL_TYPE` a finer-grained (and noisier) status within it.

**12.5/12.6 — the two columns are ~99.9% redundant, and the 0.1% exception is now fully explained.** `WELL_TYPE = OP` always means `FLOW_KIND = production` (9,143/9,143). `WELL_TYPE = WI` means `FLOW_KIND = injection` in 6,473/6,491 rows; the remaining 18 "WI + production" rows split exactly as expected — 2 from 7405's early-life quirk, 16 from 5769's 2016 transition window (146 production-flow_kind days had OP well_type, 160-144=16 had WI well_type). Physical activity confirms this combo is genuinely mixed, not a data artifact: of the 18 rows, 9 show real oil/gas/water production, 7 show all-zero volumes, and the OP/production and WI/injection combos are otherwise clean (0 all-null rows in OP/production; the 782 all-null rows in WI/injection exactly match the already-known DQ-001/DQ-003 population from Sections 9-10).

**12.7 — Section 11's CONTEXTUAL hypothesis mostly holds, but the finer 3-way split reveals something Section 11's 2-way (OP vs WI) view couldn't show.** 6 of the 12 CONTEXTUAL columns (`AVG_WHP_P`, `AVG_WHT_P`, `BORE_OIL_VOL`, `BORE_GAS_VOL`, `BORE_WAT_VOL`, `AVG_CHOKE_UOM`) are **cleanly separated** even under the 3-way `WELL_TYPE`×`FLOW_KIND` split — 0% missing in *both* production combos (OP/production and WI/production), 100% in WI/injection. But the other 6 (`BORE_WI_VOL`, `AVG_ANNULUS_PRESS`, `AVG_CHOKE_SIZE_P`, `AVG_DOWNHOLE_PRESSURE`, `AVG_DOWNHOLE_TEMPERATURE`, `AVG_DP_TUBING`) are **not** cleanly separated: within the WI/production transition rows specifically, they sit at intermediate missingness (5.56%-88.89%, not 0% or 100%). This is a real physical signal, not noise — it suggests that during 5769/7405's mode transitions, wellhead-level metering (WHP/WHT) and surface production volumes came back online immediately, while downhole/annulus instrumentation and injection-volume metering lagged behind. That is exactly the kind of nuance Section 11's binary view could not see.

**12.8/12.9 — `AVG_CHOKE_UOM` is STATIC** (exactly one non-null value, `"%"`, wherever present; 0 whitespace/case variants). `WELL_TYPE` and `FLOW_KIND` are both **TEMPORAL**.

**Database-design implication:** neither `WELL_TYPE` nor `FLOW_KIND` can be flattened onto a one-row-per-wellbore dimension without losing real information — both belong on the daily fact table (where they already live), or a type-2 history structure if a wellbore-status dimension is built later. Given how close to redundant the two columns are (99.9% agreement, with the 0.1% disagreement now fully explained rather than mysterious), Section 24 should explicitly decide whether the fact table needs both columns or whether one is derivable from the other plus a documented exception list for the 18 known rows — carrying both forward untouched for now, since neither notebook nor Section 23 has made that call.

Handing off to **Sections 13 and 14** with this context now available: production-volume validation (13) can be scoped to `WELL_TYPE = OP` / `FLOW_KIND = production` rows where the CONTEXTUAL pattern is clean, and injection-volume validation (14) should treat the WI/injection combo (clean) separately from the 18 WI/production rows (mixed, transition-period), rather than validating all "WI-type" rows as one uniform population.


## 13. Production-volume validation

**Main question:** are oil, gas, and produced-water volumes internally consistent with the recorded operating state?

This section uses the categorical breakdown Section 12 already established: `WELL_TYPE = OP` / `FLOW_KIND = production` (9,143 rows) is the clean producing population; `WELL_TYPE = WI` / `FLOW_KIND = injection` (6,473 rows) never carries production volumes; and 18 `WELL_TYPE = WI` / `FLOW_KIND = production` rows are a known transition population that gets handled separately here rather than folded into either producer or injector statistics.

Rates (`volume / ON_STREAM_HRS`, only where hours > 0) are computed as **diagnostic Series only** — never written back into `daily_df`. Large day-to-day jumps are flagged for **REVIEW**, not automatically treated as errors. A new issue ID (`DQ-005`) is only introduced if the evidence in this section actually warrants one — not created in advance.


In [127]:
# --- TEMPORARY LOADING FOR SECTION 13 ---
# Section 1 (Load source data) has not been implemented yet in this notebook.
# This block loads only what Section 13 needs, and is clearly marked so it can
# be deleted once Section 1 provides `daily_df` for the whole notebook.
if "daily_df" not in globals():
    from pathlib import Path
    import pandas as pd

    PROJECT_ROOT = Path.cwd().parent
    WORKBOOK_PATH = PROJECT_ROOT / "data" / "raw" / "Volve production data.xlsx"
    DAILY_SHEET_NAME = "Daily Production Data"

    if not WORKBOOK_PATH.exists():
        raise FileNotFoundError(f"Source workbook not found at {WORKBOOK_PATH}")

    daily_df = pd.read_excel(WORKBOOK_PATH, sheet_name=DAILY_SHEET_NAME)
    print(f"[Section 13 temporary load] daily_df loaded: {daily_df.shape}")
else:
    print(f"Using daily_df already loaded earlier in the notebook: {daily_df.shape}")


Using daily_df already loaded earlier in the notebook: (15634, 24)


In [128]:
# Reuse Section 9-12 derived state if available, recompute the minimum if not.
if "dateprd_parsed" not in globals():
    dateprd_parsed = pd.to_datetime(daily_df["DATEPRD"], errors="coerce")
if "daily_with_dates" not in globals():
    daily_with_dates = daily_df.copy()
    daily_with_dates["DATEPRD_parsed"] = dateprd_parsed
if "hours" not in globals():
    hours = daily_df["ON_STREAM_HRS"]

print("Ready: daily_with_dates, dateprd_parsed, hours")


Ready: daily_with_dates, dateprd_parsed, hours


In [129]:
required_columns = [
    "BORE_OIL_VOL", "BORE_GAS_VOL", "BORE_WAT_VOL", "WELL_TYPE", "FLOW_KIND",
    "ON_STREAM_HRS", "NPD_WELL_BORE_CODE", "NPD_WELL_BORE_NAME", "DATEPRD",
]
missing_columns = [c for c in required_columns if c not in daily_df.columns]
if missing_columns:
    raise ValueError(f"Required column(s) missing from daily_df: {missing_columns}")
print(f"Required columns present: {required_columns}")

PRODUCTION_VOLUME_COLS = ["BORE_OIL_VOL", "BORE_GAS_VOL", "BORE_WAT_VOL"]
REPORT_COLS = [
    "DATEPRD_parsed", "NPD_WELL_BORE_CODE", "NPD_WELL_BORE_NAME", "WELL_TYPE", "FLOW_KIND",
    "ON_STREAM_HRS", "BORE_OIL_VOL", "BORE_GAS_VOL", "BORE_WAT_VOL", "BORE_WI_VOL",
]


Required columns present: ['BORE_OIL_VOL', 'BORE_GAS_VOL', 'BORE_WAT_VOL', 'WELL_TYPE', 'FLOW_KIND', 'ON_STREAM_HRS', 'NPD_WELL_BORE_CODE', 'NPD_WELL_BORE_NAME', 'DATEPRD']


### 13.1 Basic validity — negative / zero / positive / NULL


In [130]:
def volume_basic_validity(col: str) -> dict:
    s = daily_df[col]
    return {
        "column": col,
        "non_null_count": int(s.notna().sum()),
        "null_count": int(s.isna().sum()),
        "negative_count": int((s < 0).sum()),
        "zero_count": int((s == 0).sum()),
        "positive_count": int((s > 0).sum()),
    }


production_basic_validity = pd.DataFrame([volume_basic_validity(c) for c in PRODUCTION_VOLUME_COLS])
production_basic_validity


,column,non_null_count,null_count,negative_count,zero_count,positive_count
0,BORE_OIL_VOL,9161,6473,0,1153,8008
1,BORE_GAS_VOL,9161,6473,0,1150,8011
2,BORE_WAT_VOL,9161,6473,4,1526,7631


In [131]:
# Negative values only occur in BORE_WAT_VOL (Section 8's earlier profiling
# already flagged 4 rows) - inspected directly here rather than left as a count.
negative_volume_rows = daily_with_dates.loc[
    (daily_df["BORE_OIL_VOL"] < 0) | (daily_df["BORE_GAS_VOL"] < 0) | (daily_df["BORE_WAT_VOL"] < 0),
    REPORT_COLS,
].sort_values("DATEPRD_parsed")
print(f"Rows with any negative production volume: {len(negative_volume_rows)}")
negative_volume_rows


Rows with any negative production volume: 4


,DATEPRD_parsed,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,WELL_TYPE,FLOW_KIND,ON_STREAM_HRS,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL
1982,2008-04-23,5599,15/9-F-12,OP,production,24.000,2735.53,422115.01,-14.19,NaN
5350,2009-03-03,5351,15/9-F-14,OP,production,24.000,4339.13,616094.56,-0.95,NaN
3502,2012-08-13,5599,15/9-F-12,OP,production,0.625,632.96,12123.37,-457.84,NaN
6558,2012-08-13,5351,15/9-F-14,OP,production,0.625,202.53,3754.12,-59.19,NaN


### 13.2 Extreme values by wellbore

Descriptive distribution only (min / median / p95 / max) — not a formal outlier test. That is Section 20's job; this just reports magnitude and scale per wellbore.


In [132]:
def distribution_by_wellbore(col: str) -> pd.DataFrame:
    rows = []
    for code_value, group in daily_df.groupby("NPD_WELL_BORE_CODE"):
        s = group[col]
        rows.append(
            {
                "npd_code": code_value,
                "wellbore_name": group["NPD_WELL_BORE_NAME"].iloc[0],
                "count": int(s.notna().sum()),
                "min": s.min(),
                "median": s.median(),
                "p95": s.quantile(0.95),
                "max": s.max(),
            }
        )
    return pd.DataFrame(rows).sort_values("npd_code").reset_index(drop=True)


oil_distribution_by_wellbore = distribution_by_wellbore("BORE_OIL_VOL")
oil_distribution_by_wellbore


,npd_code,wellbore_name,count,min,median,p95,max
0,5351,15/9-F-14,3056,0.0,880.785,4095.4025,5644.37
1,5599,15/9-F-12,3056,0.0,697.655,5080.9025,5901.84
2,5693,15/9-F-4,0,NaN,NaN,NaN,NaN
3,5769,15/9-F-5,160,0.0,307.885,360.9145,396.80
4,7078,15/9-F-11,1165,0.0,1077.760,1749.5240,2064.61
5,7289,15/9-F-15 D,978,0.0,175.330,278.3100,513.12
6,7405,15/9-F-1 C,746,0.0,200.685,743.8150,1549.81


In [133]:
gas_distribution_by_wellbore = distribution_by_wellbore("BORE_GAS_VOL")
gas_distribution_by_wellbore


,npd_code,wellbore_name,count,min,median,p95,max
0,5351,15/9-F-14,3056,0.0,142362.865,566211.7300,789974.73
1,5599,15/9-F-12,3056,0.0,107734.345,725470.5275,851131.52
2,5693,15/9-F-4,0,NaN,NaN,NaN,NaN
3,5769,15/9-F-5,160,0.0,49502.490,57575.7300,62250.56
4,7078,15/9-F-11,1165,0.0,163219.280,255698.1580,300167.59
5,7289,15/9-F-15 D,978,0.0,27023.825,42560.5780,77600.88
6,7405,15/9-F-1 C,746,0.0,30850.565,107666.1375,221707.31


In [134]:
water_distribution_by_wellbore = distribution_by_wellbore("BORE_WAT_VOL")
water_distribution_by_wellbore


,npd_code,wellbore_name,count,min,median,p95,max
0,5351,15/9-F-14,3056,-59.19,2965.715,4033.4600,5691.77
1,5599,15/9-F-12,3056,-457.84,1349.840,5073.7900,8019.74
2,5693,15/9-F-4,0,NaN,NaN,NaN,NaN
3,5769,15/9-F-5,160,0.00,80.665,181.0595,334.07
4,7078,15/9-F-11,1165,0.00,461.160,2590.4180,3559.67
5,7289,15/9-F-15 D,978,0.00,6.060,228.2875,352.29
6,7405,15/9-F-1 C,746,0.00,77.045,935.2050,1645.08


### 13.3 Production volumes when `FLOW_KIND`/`WELL_TYPE` indicate injection

Two separate tests, since Section 12 showed `WELL_TYPE` and `FLOW_KIND` are ~99.9% but not 100% redundant: the narrower `FLOW_KIND = injection` test (already shown clean in Section 12.6) and the broader `WELL_TYPE = WI` test (which also catches the 18 known transition rows).


In [135]:
injection_flow_with_production_mask = (daily_df["FLOW_KIND"] == "injection") & (
    (daily_df["BORE_OIL_VOL"] > 0) | (daily_df["BORE_GAS_VOL"] > 0) | (daily_df["BORE_WAT_VOL"] > 0)
)
wi_type_with_production_mask = (daily_df["WELL_TYPE"] == "WI") & (
    (daily_df["BORE_OIL_VOL"] > 0) | (daily_df["BORE_GAS_VOL"] > 0) | (daily_df["BORE_WAT_VOL"] > 0)
)

print(f"FLOW_KIND = injection with positive oil/gas/water (REVIEW): {int(injection_flow_with_production_mask.sum())}")
print(f"WELL_TYPE = WI with positive oil/gas/water (REVIEW):        {int(wi_type_with_production_mask.sum())}")


FLOW_KIND = injection with positive oil/gas/water (REVIEW): 0
WELL_TYPE = WI with positive oil/gas/water (REVIEW):        9


In [136]:
wi_type_with_production_rows = (
    daily_with_dates.loc[wi_type_with_production_mask, REPORT_COLS]
    .sort_values(["NPD_WELL_BORE_CODE", "DATEPRD_parsed"])
    .reset_index(drop=True)
)
wi_type_with_production_rows


,DATEPRD_parsed,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,WELL_TYPE,FLOW_KIND,ON_STREAM_HRS,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL
0,2016-05-04,5769,15/9-F-5,WI,production,12.0,328.70,52387.43,136.13,0.0
1,2016-07-26,5769,15/9-F-5,WI,production,12.0,340.52,54468.73,78.67,0.0
2,2016-07-27,5769,15/9-F-5,WI,production,12.0,334.18,53468.31,85.23,0.0
3,2016-07-28,5769,15/9-F-5,WI,production,12.0,338.69,54190.20,88.61,0.0
4,2016-07-29,5769,15/9-F-5,WI,production,12.0,342.77,53787.03,89.70,0.0
5,2016-07-30,5769,15/9-F-5,WI,production,12.0,349.25,55339.69,91.80,0.0
6,2016-07-31,5769,15/9-F-5,WI,production,12.0,342.39,53103.90,88.56,0.0
7,2016-08-01,5769,15/9-F-5,WI,production,12.0,351.38,53406.08,85.88,0.0
8,2014-07-07,7405,15/9-F-1 C,WI,production,24.0,522.00,77846.76,154.47,NaN


### 13.4 Positive production with `ON_STREAM_HRS = 0`


In [137]:
zero_hours_positive_production_mask = (hours == 0) & (
    (daily_df["BORE_OIL_VOL"] > 0) | (daily_df["BORE_GAS_VOL"] > 0) | (daily_df["BORE_WAT_VOL"] > 0)
)
zero_hours_positive_production_rows = (
    daily_with_dates.loc[zero_hours_positive_production_mask, REPORT_COLS]
    .sort_values("DATEPRD_parsed")
    .reset_index(drop=True)
)
print(f"ON_STREAM_HRS = 0 with positive oil/gas/water (REVIEW): {len(zero_hours_positive_production_rows)}")
zero_hours_positive_production_rows


ON_STREAM_HRS = 0 with positive oil/gas/water (REVIEW): 3

,DATEPRD_parsed,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,WELL_TYPE,FLOW_KIND,ON_STREAM_HRS,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL
0,2011-12-26,5599,15/9-F-12,OP,production,0.0,0.00,7.09,9.71,NaN
1,2011-12-26,5351,15/9-F-14,OP,production,0.0,0.00,56.38,45.12,NaN
2,2015-01-17,7078,15/9-F-11,OP,production,0.0,1026.57,150773.50,461.02,NaN


### 13.5 Production rates (diagnostic only, not written back)

`rate = volume / ON_STREAM_HRS`, computed **only** where `ON_STREAM_HRS > 0`, kept as standalone Series — `daily_df` is never modified. A rate normalizes for operating time and is a better basis for spotting suspicious values than a raw daily volume.


In [138]:
positive_hours_mask = hours > 0

oil_rate = daily_df["BORE_OIL_VOL"].where(positive_hours_mask) / hours.where(positive_hours_mask)
gas_rate = daily_df["BORE_GAS_VOL"].where(positive_hours_mask) / hours.where(positive_hours_mask)
water_rate = daily_df["BORE_WAT_VOL"].where(positive_hours_mask) / hours.where(positive_hours_mask)

rate_distribution = pd.DataFrame(
    {
        "oil_rate_per_hour": oil_rate.describe(),
        "gas_rate_per_hour": gas_rate.describe(),
        "water_rate_per_hour": water_rate.describe(),
    }
)
rate_distribution


,oil_rate_per_hour,gas_rate_per_hour,water_rate_per_hour
count,8020.000000,8020.000000,8020.000000
mean,54.192192,7957.036747,82.454606
std,57.827879,8057.746919,72.712158
min,0.000000,0.000000,-732.544000
25%,11.947083,1867.978646,7.410233
50%,32.347117,4955.397500,74.209375
75%,71.625354,10921.220521,144.883333
max,1012.736000,53200.080000,474.314167


In [139]:
def rate_stats_by_wellbore(rate_series: pd.Series) -> pd.DataFrame:
    rows = []
    for code_value, group in daily_df.groupby("NPD_WELL_BORE_CODE"):
        r = rate_series.loc[group.index]
        rows.append(
            {
                "npd_code": code_value,
                "wellbore_name": group["NPD_WELL_BORE_NAME"].iloc[0],
                "count": int(r.notna().sum()),
                "mean": r.mean(),
                "median": r.median(),
                "max": r.max(),
            }
        )
    return pd.DataFrame(rows).sort_values("npd_code").reset_index(drop=True)


oil_rate_by_wellbore = rate_stats_by_wellbore(oil_rate)
oil_rate_by_wellbore


,npd_code,wellbore_name,count,mean,median,max
0,5351,15/9-F-14,2724,62.853691,44.928125,324.048000
1,5599,15/9-F-12,2838,69.922430,34.554583,1012.736000
2,5693,15/9-F-4,0,NaN,NaN,NaN
3,5769,15/9-F-5,129,14.233331,13.236667,29.281667
4,7078,15/9-F-11,1122,43.779435,45.735417,86.025417
5,7289,15/9-F-15 D,769,8.365396,8.029167,28.073409
6,7405,15/9-F-1 C,438,17.302412,14.277917,64.575417


### 13.6 Large day-to-day jumps (REVIEW, not automatically errors)

For each wellbore, sorted chronologically, the percentage change from the previous recorded value. Only evaluated where the previous value was positive (avoids meaningless infinite jumps from a previous zero). Flagged at a >200% change (more than tripling, or dropping by more than two-thirds) — a screening threshold, not a correctness judgement.


In [140]:
JUMP_REVIEW_THRESHOLD_PCT = 200

jump_rows = []
for col in PRODUCTION_VOLUME_COLS:
    for code_value, group in daily_with_dates.groupby("NPD_WELL_BORE_CODE"):
        g = group.dropna(subset=["DATEPRD_parsed", col]).sort_values("DATEPRD_parsed")
        prev_val = g[col].shift(1)
        prev_date = g["DATEPRD_parsed"].shift(1)
        pct_change = (g[col] - prev_val) / prev_val * 100
        flagged = (prev_val > 0) & (pct_change.abs() > JUMP_REVIEW_THRESHOLD_PCT)
        for idx in g.index[flagged]:
            jump_rows.append(
                {
                    "column": col,
                    "npd_code": code_value,
                    "wellbore_name": g.loc[idx, "NPD_WELL_BORE_NAME"],
                    "previous_date": prev_date.loc[idx].date(),
                    "current_date": g.loc[idx, "DATEPRD_parsed"].date(),
                    "previous_value": prev_val.loc[idx],
                    "current_value": g.loc[idx, col],
                    "pct_change": round(pct_change.loc[idx], 1),
                }
            )

large_jumps = (
    pd.DataFrame(jump_rows)
    if jump_rows
    else pd.DataFrame(columns=["column", "npd_code", "wellbore_name", "previous_date", "current_date", "previous_value", "current_value", "pct_change"])
)
large_jumps = large_jumps.reindex(large_jumps["pct_change"].abs().sort_values(ascending=False).index).reset_index(drop=True) if len(large_jumps) else large_jumps
print(f"Day-to-day jumps exceeding {JUMP_REVIEW_THRESHOLD_PCT}% change: {len(large_jumps)}")


Day-to-day jumps exceeding 200% change: 411


In [141]:
if len(large_jumps):
    print("By column:")
    print(large_jumps.groupby("column").size())
    print("\nBy wellbore:")
    print(large_jumps.groupby("npd_code").size())


By column:
column
BORE_GAS_VOL    131
BORE_OIL_VOL    129
BORE_WAT_VOL    151
dtype: int64

By wellbore:
npd_code
5351    154
5599    172
5769      3
7078     42
7289     19
7405     21
dtype: int64


In [142]:
large_jumps.head(30)


,column,npd_code,wellbore_name,previous_date,current_date,previous_value,current_value,pct_change
0,BORE_GAS_VOL,5599,15/9-F-12,2011-12-26,2011-12-27,7.09,84899.76,1197357.8
1,BORE_GAS_VOL,5351,15/9-F-14,2011-12-26,2011-12-27,56.38,162228.58,287641.4
2,BORE_WAT_VOL,5599,15/9-F-12,2009-08-04,2009-08-05,0.07,156.88,224014.3
3,BORE_OIL_VOL,7289,15/9-F-15 D,2016-04-06,2016-04-07,0.17,262.93,154564.7
4,BORE_GAS_VOL,7289,15/9-F-15 D,2016-04-06,2016-04-07,28.86,42371.22,146716.4
5,BORE_WAT_VOL,7289,15/9-F-15 D,2016-04-06,2016-04-07,0.29,291.51,100420.7
6,BORE_WAT_VOL,5599,15/9-F-12,2008-05-09,2008-05-10,0.01,6.39,63800.0
7,BORE_WAT_VOL,5599,15/9-F-12,2008-05-23,2008-05-24,19.71,8019.72,40588.6
8,BORE_OIL_VOL,5599,15/9-F-12,2009-08-04,2009-08-05,11.98,4407.16,36687.6
9,BORE_GAS_VOL,5599,15/9-F-12,2010-08-28,2010-08-29,535.16,155880.56,29027.8


### 13.7 The 18 Section 12 transition rows, handled separately

`WELL_TYPE = WI` / `FLOW_KIND = production` — deliberately not folded into either the producer or injector population statistics above.


In [143]:
transition_rows_mask = (daily_df["WELL_TYPE"] == "WI") & (daily_df["FLOW_KIND"] == "production")
transition_rows_detail = (
    daily_with_dates.loc[transition_rows_mask, REPORT_COLS]
    .sort_values(["NPD_WELL_BORE_CODE", "DATEPRD_parsed"])
    .reset_index(drop=True)
)
print(f"WELL_TYPE=WI & FLOW_KIND=production rows: {len(transition_rows_detail)}")
transition_rows_detail


WELL_TYPE=WI & FLOW_KIND=production rows: 18


,DATEPRD_parsed,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,WELL_TYPE,FLOW_KIND,ON_STREAM_HRS,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL
0,2016-04-11,5769,15/9-F-5,WI,production,0.0,0.00,0.00,0.00,NaN
1,2016-05-04,5769,15/9-F-5,WI,production,12.0,328.70,52387.43,136.13,0.0
2,2016-07-26,5769,15/9-F-5,WI,production,12.0,340.52,54468.73,78.67,0.0
3,2016-07-27,5769,15/9-F-5,WI,production,12.0,334.18,53468.31,85.23,0.0
4,2016-07-28,5769,15/9-F-5,WI,production,12.0,338.69,54190.20,88.61,0.0
5,2016-07-29,5769,15/9-F-5,WI,production,12.0,342.77,53787.03,89.70,0.0
6,2016-07-30,5769,15/9-F-5,WI,production,12.0,349.25,55339.69,91.80,0.0
7,2016-07-31,5769,15/9-F-5,WI,production,12.0,342.39,53103.90,88.56,0.0
8,2016-08-01,5769,15/9-F-5,WI,production,12.0,351.38,53406.08,85.88,0.0
9,2016-09-06,5769,15/9-F-5,WI,production,0.0,0.00,0.00,0.00,0.0


In [144]:
# Final production-volume verdict.
n_negative = len(negative_volume_rows)
n_injection_flow_with_production = int(injection_flow_with_production_mask.sum())
n_wi_type_with_production = int(wi_type_with_production_mask.sum())
n_zero_hours_positive_production = len(zero_hours_positive_production_rows)
n_large_jumps = len(large_jumps)

# Anything outside the already-characterized 18 WELL_TYPE=WI/FLOW_KIND=production
# transition population would be a genuinely new, unexplained inconsistency.
unexplained_wi_production = n_wi_type_with_production - len(transition_rows_detail[
    (transition_rows_detail["BORE_OIL_VOL"] > 0)
    | (transition_rows_detail["BORE_GAS_VOL"] > 0)
    | (transition_rows_detail["BORE_WAT_VOL"] > 0)
])

print("=" * 60)
print("SECTION 13 - PRODUCTION-VOLUME VALIDATION SUMMARY")
print("=" * 60)
print(f"Negative volumes (all in BORE_WAT_VOL):                {n_negative}")
print(f"FLOW_KIND=injection with positive production:          {n_injection_flow_with_production}")
print(f"WELL_TYPE=WI with positive production:                 {n_wi_type_with_production}")
print(f"  - of which inside the known 18-row transition set:   {n_wi_type_with_production - unexplained_wi_production}")
print(f"  - unexplained (outside that set):                    {unexplained_wi_production}")
print(f"ON_STREAM_HRS=0 with positive production:               {n_zero_hours_positive_production}")
print(f"Large day-to-day jumps (>{JUMP_REVIEW_THRESHOLD_PCT}%):                    {n_large_jumps}")


SECTION 13 - PRODUCTION-VOLUME VALIDATION SUMMARY
Negative volumes (all in BORE_WAT_VOL):                4
FLOW_KIND=injection with positive production:          0
WELL_TYPE=WI with positive production:                 9
  - of which inside the known 18-row transition set:   9
  - unexplained (outside that set):                    0
ON_STREAM_HRS=0 with positive production:               3
Large day-to-day jumps (>200%):                    411


**Interpretation for later relational modelling:**

**Main question answered: production volumes are, overall, highly consistent with recorded operating state.** The two most direct state/volume consistency tests came back essentially clean: `FLOW_KIND = injection` with positive oil/gas/water is **0/6,473** rows, and `WELL_TYPE = WI` with positive production is **9/6,491** rows — and all 9 fall entirely inside the 18-row transition population Section 12 already characterized (0 unexplained). Section 12's account of these columns holds up under direct volume testing, not just the categorical/missingness evidence from before.

**Two genuinely new findings surfaced, not yet covered by any existing issue ID:**

1. **4 negative `BORE_WAT_VOL` values** (-14.19, -0.95, -457.84, -59.19), all on `WELL_TYPE = OP`/`FLOW_KIND = production` rows (wellbores 5351, 5599). Two of the four share the same date (2012-08-13, wells 5599 and 5351) and an identical fractional `ON_STREAM_HRS = 0.625` — a specific, testable coincidence worth carrying forward, not just a count.
2. **1 unexplained case of positive production with `ON_STREAM_HRS = 0`**: wellbore 7078 on 2015-01-17, with 1,026.57 bbl oil and 150,773.5 gas recorded against zero operating hours — a real state/volume mismatch with no adjacent-day explanation found in this section.

**The other two zero-hours-positive-production rows are not mysterious — they resolve into a coherent story once cross-referenced with 13.6.** Both are wells 5599 and 5351 on 2011-12-26, each with a small positive volume (7.09 gas / 9.71 water for 5599; 56.38 gas for 5351) against zero hours. The very next day, 2011-12-27, both wells show the **two largest gas-volume jumps in the entire 411-row jump table** (7.09 → 84,899.76, and 56.38 → 162,228.58) — full production resuming the day immediately after a shared near-zero day. That is much more consistent with a brief joint shutdown/restricted-flow event followed by recovery than with a data error, though this notebook does not resolve that as fact — it is a hypothesis the pattern strongly supports.

**13.6 — 411 jumps exceeding 200% is a lot in absolute terms, but proportionally consistent across wellbores** (roughly 2-5% of each wellbore's row count, from 3/160 for 5769 to 172/3,056 for 5599) rather than concentrated in one well, which reads more like routine allocation volatility than a defect. One methodological caveat worth carrying into Section 20: most of the largest percentage jumps here are dominated by a **near-zero previous value** (e.g. 0.01 → 6.39, 0.04 → 7.77) — a percentage-change threshold alone is easily dominated by baseline-near-zero artifacts rather than large absolute swings, and any automated version of this check (Section 25) should likely combine a percentage threshold with a minimum absolute-value floor.

**DQ-005 is introduced here** — the first new issue ID minted since DQ-004, and only because the evidence above genuinely warrants one distinct from DQ-001 through DQ-004: **production-volume/state inconsistency**, covering the 4 negative `BORE_WAT_VOL` rows and the 1 unexplained zero-hours/positive-volume row (wellbore 7078, 2015-01-17). The two 2011-12-26 rows are tracked under the same test but noted as likely explained by the shutdown/recovery pattern above, not left equally unresolved.

```
DQ-001  Pre-field-life injection records
DQ-002  Shared 12-day reporting gap
DQ-003  NULL ON_STREAM_HRS
DQ-004  ON_STREAM_HRS > 24 / possible DST effect
DQ-005  Production-volume/state inconsistency (negative BORE_WAT_VOL; 1 unexplained zero-hours/positive-volume row)
```

For relational modelling: none of this blocks building the production side of the daily fact table — every value here is a real recorded number, not a parsing or structural failure. `BORE_OIL_VOL`/`BORE_GAS_VOL`/`BORE_WAT_VOL` should load as nullable numeric columns without a `CHECK (>= 0)` constraint until DQ-005 is resolved (a hard non-negative constraint would currently reject 4 real source rows). The diagnostic rate series (`oil_rate`, `gas_rate`, `water_rate`) were computed for this analysis only and were never written into `daily_df` — they exist purely as evidence, consistent with keeping the source untouched throughout this notebook.


## 14. Injection-volume validation

**Main question:** is `BORE_WI_VOL` internally consistent with the recorded operating state, and does `ON_STREAM_HRS` mean the same thing for injection records as it does for production records? The second question is not assumed answered — Section 14 specifically watches for evidence that it might not.

Nine checks: basic validity, injection by operational state (keeping the 18 transition rows identifiable), positive injection outside the expected injection state, the operating-hours mirror of Section 13's check (explicitly linked to DQ-001/DQ-003), a diagnostic-only injection rate, day-to-day change analysis using both absolute and percentage change (applying Section 13's lesson about near-zero baselines), a dedicated DQ-001/DQ-003 investigation for wellbores 5693/5769, a dedicated look at the 18 transition rows, and a final verdict. `DQ-006` is only introduced if a genuinely unexplained inconsistency turns up — not merely because injection occurs during an already-understood transition state.


In [145]:
# --- TEMPORARY LOADING FOR SECTION 14 ---
# Section 1 (Load source data) has not been implemented yet in this notebook.
# This block loads only what Section 14 needs, and is clearly marked so it can
# be deleted once Section 1 provides `daily_df` for the whole notebook.
if "daily_df" not in globals():
    from pathlib import Path
    import pandas as pd

    PROJECT_ROOT = Path.cwd().parent
    WORKBOOK_PATH = PROJECT_ROOT / "data" / "raw" / "Volve production data.xlsx"
    DAILY_SHEET_NAME = "Daily Production Data"

    if not WORKBOOK_PATH.exists():
        raise FileNotFoundError(f"Source workbook not found at {WORKBOOK_PATH}")

    daily_df = pd.read_excel(WORKBOOK_PATH, sheet_name=DAILY_SHEET_NAME)
    print(f"[Section 14 temporary load] daily_df loaded: {daily_df.shape}")
else:
    print(f"Using daily_df already loaded earlier in the notebook: {daily_df.shape}")


Using daily_df already loaded earlier in the notebook: (15634, 24)


In [146]:
# Reuse Section 9-13 derived state if available, recompute the minimum if not.
if "dateprd_parsed" not in globals():
    dateprd_parsed = pd.to_datetime(daily_df["DATEPRD"], errors="coerce")
if "daily_with_dates" not in globals():
    daily_with_dates = daily_df.copy()
    daily_with_dates["DATEPRD_parsed"] = dateprd_parsed
if "hours" not in globals():
    hours = daily_df["ON_STREAM_HRS"]
if "EXPECTED_FIELD_START" not in globals():
    EXPECTED_FIELD_START = pd.Timestamp("2008-01-01")
if "before_field_life_mask" not in globals():
    before_field_life_mask = dateprd_parsed.notna() & (dateprd_parsed < EXPECTED_FIELD_START)

print("Ready: daily_with_dates, dateprd_parsed, hours, before_field_life_mask")


Ready: daily_with_dates, dateprd_parsed, hours, before_field_life_mask


In [147]:
required_columns = [
    "BORE_WI_VOL", "BORE_OIL_VOL", "BORE_GAS_VOL", "BORE_WAT_VOL", "WELL_TYPE", "FLOW_KIND",
    "ON_STREAM_HRS", "NPD_WELL_BORE_CODE", "NPD_WELL_BORE_NAME", "DATEPRD",
]
missing_columns = [c for c in required_columns if c not in daily_df.columns]
if missing_columns:
    raise ValueError(f"Required column(s) missing from daily_df: {missing_columns}")
print(f"Required columns present: {required_columns}")

REPORT_COLS = [
    "DATEPRD_parsed", "NPD_WELL_BORE_CODE", "NPD_WELL_BORE_NAME", "WELL_TYPE", "FLOW_KIND",
    "ON_STREAM_HRS", "BORE_WI_VOL", "BORE_OIL_VOL", "BORE_GAS_VOL", "BORE_WAT_VOL",
]


Required columns present: ['BORE_WI_VOL', 'BORE_OIL_VOL', 'BORE_GAS_VOL', 'BORE_WAT_VOL', 'WELL_TYPE', 'FLOW_KIND', 'ON_STREAM_HRS', 'NPD_WELL_BORE_CODE', 'NPD_WELL_BORE_NAME', 'DATEPRD']


### 14.1 `BORE_WI_VOL` basic validity


In [148]:
wi_series = daily_df["BORE_WI_VOL"]
wi_basic_validity = pd.DataFrame(
    [
        {
            "non_null_count": int(wi_series.notna().sum()),
            "null_count": int(wi_series.isna().sum()),
            "negative_count": int((wi_series < 0).sum()),
            "zero_count": int((wi_series == 0).sum()),
            "positive_count": int((wi_series > 0).sum()),
            "min": wi_series.min(),
            "median": wi_series.median(),
            "mean": wi_series.mean(),
            "max": wi_series.max(),
        }
    ]
)
wi_basic_validity


,non_null_count,null_count,negative_count,zero_count,positive_count,min,median,mean,max
0,5706,9928,0,302,5404,0.0,5504.739769,5315.480815,10013.6


In [149]:
negative_wi_rows = daily_with_dates.loc[wi_series < 0, REPORT_COLS].sort_values("DATEPRD_parsed")
print(f"Rows with negative BORE_WI_VOL (REVIEW, not automatic deletion): {len(negative_wi_rows)}")
negative_wi_rows


Rows with negative BORE_WI_VOL (REVIEW, not automatic deletion): 0


,DATEPRD_parsed,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,WELL_TYPE,FLOW_KIND,ON_STREAM_HRS,BORE_WI_VOL,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL


### 14.2 Injection by operational state

For each `WELL_TYPE` × `FLOW_KIND` combination — the 18-row `WI`/`production` transition population stays visible as its own row rather than being absorbed into either the producer or injector totals.


In [150]:
def wi_state_row(g: pd.DataFrame) -> pd.Series:
    s = g["BORE_WI_VOL"]
    return pd.Series(
        {
            "rows": len(g),
            "wi_null": int(s.isna().sum()),
            "wi_zero": int((s == 0).sum()),
            "wi_positive": int((s > 0).sum()),
            "wi_negative": int((s < 0).sum()),
            "total_injected_volume": s.sum(),
        }
    )


injection_by_state = daily_df.groupby(["WELL_TYPE", "FLOW_KIND"], dropna=False).apply(wi_state_row).reset_index()
injection_by_state


,WELL_TYPE,FLOW_KIND,rows,wi_null,wi_zero,wi_positive,wi_negative,total_injected_volume
0,OP,production,9143.0,9143.0,0.0,0.0,0.0,0.000000e+00
1,WI,injection,6473.0,782.0,287.0,5404.0,0.0,3.033013e+07
2,WI,production,18.0,3.0,15.0,0.0,0.0,0.000000e+00


### 14.3 Positive injection outside the expected injection state

`BORE_WI_VOL > 0` where the categorical state suggests production (`WELL_TYPE = OP` or `FLOW_KIND = production`). Not automatically called wrong — Section 12 already showed real transition periods exist.


In [151]:
positive_wi_outside_injection_mask = (wi_series > 0) & ((daily_df["WELL_TYPE"] == "OP") | (daily_df["FLOW_KIND"] == "production"))
positive_wi_outside_injection_rows = (
    daily_with_dates.loc[positive_wi_outside_injection_mask, REPORT_COLS]
    .sort_values(["NPD_WELL_BORE_CODE", "DATEPRD_parsed"])
    .reset_index(drop=True)
)
print(f"Positive BORE_WI_VOL where state suggests production (REVIEW): {len(positive_wi_outside_injection_rows)}")
positive_wi_outside_injection_rows


Positive BORE_WI_VOL where state suggests production (REVIEW): 0


,DATEPRD_parsed,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,WELL_TYPE,FLOW_KIND,ON_STREAM_HRS,BORE_WI_VOL,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL


### 14.4 Injection vs. operating hours

Mirrors Section 13.4. The NULL-hours population is explicitly linked to **DQ-001** (pre-field-life injection records) and **DQ-003** (NULL `ON_STREAM_HRS`) — this is the test that tells us what those 285 measurement-empty records actually contain.


In [152]:
zero_hours_positive_wi_mask = (hours == 0) & (wi_series > 0)
null_hours_positive_wi_mask = hours.isna() & (wi_series > 0)

print(f"BORE_WI_VOL > 0 with ON_STREAM_HRS = 0 (REVIEW):                          {int(zero_hours_positive_wi_mask.sum())}")
print(f"BORE_WI_VOL > 0 with ON_STREAM_HRS NULL (REVIEW, linked to DQ-001/DQ-003): {int(null_hours_positive_wi_mask.sum())}")


BORE_WI_VOL > 0 with ON_STREAM_HRS = 0 (REVIEW):                          31
BORE_WI_VOL > 0 with ON_STREAM_HRS NULL (REVIEW, linked to DQ-001/DQ-003): 0


In [153]:
zero_hours_positive_wi_rows = daily_with_dates.loc[zero_hours_positive_wi_mask, REPORT_COLS].sort_values("DATEPRD_parsed")
zero_hours_positive_wi_rows


,DATEPRD_parsed,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,WELL_TYPE,FLOW_KIND,ON_STREAM_HRS,BORE_WI_VOL,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL
9342,2008-08-07,5693,15/9-F-4,WI,injection,0.0,0.088160,NaN,NaN,NaN
10387,2011-06-18,5693,15/9-F-4,WI,injection,0.0,6263.692353,NaN,NaN,NaN
13834,2011-10-16,5769,15/9-F-5,WI,injection,0.0,48.548819,NaN,NaN,NaN
10509,2011-10-18,5693,15/9-F-4,WI,injection,0.0,0.008047,NaN,NaN,NaN
10552,2011-11-30,5693,15/9-F-4,WI,injection,0.0,0.045601,NaN,NaN,NaN
10553,2011-12-01,5693,15/9-F-4,WI,injection,0.0,0.012574,NaN,NaN,NaN
13883,2011-12-04,5769,15/9-F-5,WI,injection,0.0,45.386962,NaN,NaN,NaN
10561,2011-12-09,5693,15/9-F-4,WI,injection,0.0,0.020954,NaN,NaN,NaN
10577,2011-12-25,5693,15/9-F-4,WI,injection,0.0,0.978442,NaN,NaN,NaN
13904,2011-12-25,5769,15/9-F-5,WI,injection,0.0,54.799771,NaN,NaN,NaN


**These 31 rows should not be treated as one uniform population** — characterizing them by magnitude, rather than just counting them, before deciding whether they are explainable.


In [154]:
wi_zero_hours_magnitude = pd.cut(
    zero_hours_positive_wi_rows["BORE_WI_VOL"],
    bins=[0, 1, 100, float("inf")],
    labels=["trace (<1 m3)", "moderate (1-100 m3)", "large (>100 m3)"],
    right=False,
)
print("BORE_WI_VOL magnitude among the 31 zero-hours-positive-injection rows:")
print(wi_zero_hours_magnitude.value_counts().sort_index())


BORE_WI_VOL magnitude among the 31 zero-hours-positive-injection rows:
BORE_WI_VOL
trace (<1 m3)          17
moderate (1-100 m3)    12
large (>100 m3)         2
Name: count, dtype: int64


In [155]:
large_zero_hours_wi_rows = zero_hours_positive_wi_rows.loc[zero_hours_positive_wi_rows["BORE_WI_VOL"] > 100]
large_zero_hours_wi_rows


,DATEPRD_parsed,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,WELL_TYPE,FLOW_KIND,ON_STREAM_HRS,BORE_WI_VOL,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL
10387,2011-06-18,5693,15/9-F-4,WI,injection,0.0,6263.692353,NaN,NaN,NaN
14351,2013-03-16,5769,15/9-F-5,WI,injection,0.0,148.655136,NaN,NaN,NaN


In [156]:
# Compare how common this pattern is for injection vs. production (Section
# 13 found only 3 zero-hours-positive-production rows in total).
production_positive_mask = (daily_df["BORE_OIL_VOL"] > 0) | (daily_df["BORE_GAS_VOL"] > 0) | (daily_df["BORE_WAT_VOL"] > 0)
n_production_positive = int(production_positive_mask.sum())
n_production_zero_hours_positive = 3  # from Section 13.4

n_injection_positive = int((wi_series > 0).sum())
n_injection_zero_hours_positive = len(zero_hours_positive_wi_rows)

production_rate_pct = n_production_zero_hours_positive / n_production_positive * 100
injection_rate_pct = n_injection_zero_hours_positive / n_injection_positive * 100

print(f"Zero-hours-yet-positive rate, production: {n_production_zero_hours_positive}/{n_production_positive} = {production_rate_pct:.3f}%")
print(f"Zero-hours-yet-positive rate, injection:   {n_injection_zero_hours_positive}/{n_injection_positive} = {injection_rate_pct:.3f}%")


Zero-hours-yet-positive rate, production: 3/8011 = 0.037%
Zero-hours-yet-positive rate, injection:   31/5404 = 0.574%


### 14.5 Injection rate (diagnostic only, not written back)

`water_injection_rate = BORE_WI_VOL / ON_STREAM_HRS`, only where both are positive. A standalone Series for screening — `daily_df` is never modified.


In [157]:
positive_wi_and_hours_mask = (wi_series > 0) & (hours > 0)
water_injection_rate = wi_series.where(positive_wi_and_hours_mask) / hours.where(positive_wi_and_hours_mask)


def rate_summary_by_wellbore(rate_series: pd.Series) -> pd.DataFrame:
    rows = []
    for code_value, group in daily_df.groupby("NPD_WELL_BORE_CODE"):
        r = rate_series.loc[group.index]
        rows.append(
            {
                "npd_code": code_value,
                "wellbore_name": group["NPD_WELL_BORE_NAME"].iloc[0],
                "count": int(r.notna().sum()),
                "median": r.median(),
                "mean": r.mean(),
                "p95": r.quantile(0.95),
                "max": r.max(),
            }
        )
    return pd.DataFrame(rows).sort_values("npd_code").reset_index(drop=True)


water_injection_rate_by_wellbore = rate_summary_by_wellbore(water_injection_rate)
water_injection_rate_by_wellbore


,npd_code,wellbore_name,count,median,mean,p95,max
0,5351,15/9-F-14,0,NaN,NaN,NaN,NaN
1,5599,15/9-F-12,0,NaN,NaN,NaN,NaN
2,5693,15/9-F-4,2828,248.717489,252.036144,363.885417,1571.803890
3,5769,15/9-F-5,2545,239.420894,243.562388,357.975833,858.326404
4,7078,15/9-F-11,0,NaN,NaN,NaN,NaN
5,7289,15/9-F-15 D,0,NaN,NaN,NaN,NaN
6,7405,15/9-F-1 C,0,NaN,NaN,NaN,NaN


**Watching for whether `ON_STREAM_HRS` means the same thing for injection as for production**, rather than assuming it: comparing its distribution when a well is actively producing vs. actively injecting.


In [158]:
hours_when_producing = hours[(daily_df["BORE_OIL_VOL"] > 0) | (daily_df["BORE_GAS_VOL"] > 0) | (daily_df["BORE_WAT_VOL"] > 0)]
hours_when_injecting = hours[wi_series > 0]

hours_comparison = pd.DataFrame(
    {
        "ON_STREAM_HRS_when_producing": hours_when_producing.describe(),
        "ON_STREAM_HRS_when_injecting": hours_when_injecting.describe(),
    }
)
hours_comparison


,ON_STREAM_HRS_when_producing,ON_STREAM_HRS_when_injecting
count,8011.000000,5404.000000
mean,23.049525,22.594248
std,3.513600,4.208352
min,0.000000,0.000000
25%,24.000000,24.000000
50%,24.000000,24.000000
75%,24.000000,24.000000
max,25.000000,25.000000


### 14.6 Day-to-day change analysis

Applying Section 13's lesson: percentage change alone is dominated by near-zero-baseline artifacts. Both absolute and percentage change are computed here, and the effect of adding a minimum absolute-volume floor on top of the percentage threshold is shown directly — no final automated threshold is chosen; that is Section 25's job.


In [159]:
wi_change_rows = []
for code_value, group in daily_with_dates.groupby("NPD_WELL_BORE_CODE"):
    g = group.dropna(subset=["DATEPRD_parsed", "BORE_WI_VOL"]).sort_values("DATEPRD_parsed")
    prev_val = g["BORE_WI_VOL"].shift(1)
    prev_date = g["DATEPRD_parsed"].shift(1)
    abs_change = g["BORE_WI_VOL"] - prev_val
    pct_change = (abs_change / prev_val * 100).where(prev_val > 0)
    valid = prev_val.notna()
    for idx in g.index[valid]:
        wi_change_rows.append(
            {
                "npd_code": code_value,
                "wellbore_name": g.loc[idx, "NPD_WELL_BORE_NAME"],
                "previous_date": prev_date.loc[idx].date(),
                "current_date": g.loc[idx, "DATEPRD_parsed"].date(),
                "previous_value": prev_val.loc[idx],
                "current_value": g.loc[idx, "BORE_WI_VOL"],
                "abs_change": abs_change.loc[idx],
                "pct_change": pct_change.loc[idx],
            }
        )

wi_changes = pd.DataFrame(wi_change_rows)
print(f"Consecutive same-wellbore BORE_WI_VOL comparisons: {len(wi_changes)}")


Consecutive same-wellbore BORE_WI_VOL comparisons: 5704


In [160]:
PCT_THRESHOLD = 200
ABS_FLOOR_CANDIDATES = [0, 50, 100, 500, 1000]

floor_demo_rows = []
for floor in ABS_FLOOR_CANDIDATES:
    flagged = wi_changes[(wi_changes["pct_change"].abs() > PCT_THRESHOLD) & (wi_changes["abs_change"].abs() >= floor)]
    floor_demo_rows.append({"abs_change_floor": floor, "flags_at_pct>200%_AND_floor": len(flagged)})

pd.DataFrame(floor_demo_rows)


,abs_change_floor,flags_at_pct>200%_AND_floor
0,0,116
1,50,116
2,100,116
3,500,114
4,1000,114


In [161]:
wi_changes.reindex(wi_changes["abs_change"].abs().sort_values(ascending=False).index).head(15).reset_index(drop=True)


,npd_code,wellbore_name,previous_date,current_date,previous_value,current_value,abs_change,pct_change
0,5769,15/9-F-5,2010-04-19,2010-04-23,0.000000,7889.000000,7889.000000,NaN
1,5693,15/9-F-4,2016-08-02,2016-08-03,7704.044976,0.044462,-7704.000514,-99.999423
2,5693,15/9-F-4,2008-08-08,2008-08-09,1390.000000,9086.460449,7696.460449,553.702191
3,5693,15/9-F-4,2010-11-03,2010-11-04,7376.109421,11.000000,-7365.109421,-99.850870
4,5693,15/9-F-4,2008-08-26,2008-08-27,7481.385118,184.358871,-7297.026247,-97.535766
5,5693,15/9-F-4,2016-07-23,2016-07-24,1295.000000,8523.000000,7228.000000,558.146718
6,5693,15/9-F-4,2016-07-21,2016-07-22,7174.000000,0.000000,-7174.000000,-100.000000
7,5693,15/9-F-4,2010-04-22,2010-04-23,1476.000000,8464.000000,6988.000000,473.441734
8,5769,15/9-F-5,2011-04-08,2011-04-09,8406.000000,1466.000000,-6940.000000,-82.560076
9,5769,15/9-F-5,2009-10-18,2009-10-19,1510.000000,8297.130218,6787.130218,449.478822


### 14.7 Pre-2008 / NULL-hours injection records (`DQ-001` / `DQ-003`)

Wellbores 5693 and 5769 specifically. The precise question: do these records contain actual injection-volume measurements, or are they essentially dated well-state records with no measurements at all?


In [162]:
dq_wellbores = [5693, 5769]
dq001_dq003_mask = daily_with_dates["NPD_WELL_BORE_CODE"].isin(dq_wellbores) & (before_field_life_mask | hours.isna())
dq001_dq003_rows = (
    daily_with_dates.loc[dq001_dq003_mask, REPORT_COLS]
    .sort_values(["NPD_WELL_BORE_CODE", "DATEPRD_parsed"])
    .reset_index(drop=True)
)
print(f"DQ-001/DQ-003 rows (wellbores 5693/5769, pre-2008 or NULL ON_STREAM_HRS): {len(dq001_dq003_rows)}")


DQ-001/DQ-003 rows (wellbores 5693/5769, pre-2008 or NULL ON_STREAM_HRS): 285


In [163]:
measurement_cols = ["BORE_WI_VOL", "BORE_OIL_VOL", "BORE_GAS_VOL", "BORE_WAT_VOL"]
has_any_measurement = (dq001_dq003_rows[measurement_cols] > 0).any(axis=1)

print(f"Of these {len(dq001_dq003_rows)} rows, {int(has_any_measurement.sum())} contain at least one positive volume measurement.")
print(f"{int((~has_any_measurement).sum())} rows have zero/NULL across all four volume columns.")


Of these 285 rows, 0 contain at least one positive volume measurement.
285 rows have zero/NULL across all four volume columns.


In [164]:
dq001_dq003_rows.head(10)


,DATEPRD_parsed,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,WELL_TYPE,FLOW_KIND,ON_STREAM_HRS,BORE_WI_VOL,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL
0,2007-09-01,5693,15/9-F-4,WI,injection,NaN,NaN,NaN,NaN,NaN
1,2007-09-02,5693,15/9-F-4,WI,injection,NaN,NaN,NaN,NaN,NaN
2,2007-09-03,5693,15/9-F-4,WI,injection,NaN,NaN,NaN,NaN,NaN
3,2007-09-04,5693,15/9-F-4,WI,injection,NaN,NaN,NaN,NaN,NaN
4,2007-09-05,5693,15/9-F-4,WI,injection,NaN,NaN,NaN,NaN,NaN
5,2007-09-06,5693,15/9-F-4,WI,injection,NaN,NaN,NaN,NaN,NaN
6,2007-09-07,5693,15/9-F-4,WI,injection,NaN,NaN,NaN,NaN,NaN
7,2007-09-08,5693,15/9-F-4,WI,injection,NaN,NaN,NaN,NaN,NaN
8,2007-09-09,5693,15/9-F-4,WI,injection,NaN,NaN,NaN,NaN,NaN
9,2007-09-10,5693,15/9-F-4,WI,injection,NaN,NaN,NaN,NaN,NaN


### 14.8 The 18 transition rows

State labels, injection volume, production volumes, and on-stream hours together — does physical activity support one label more strongly than the other?


In [165]:
transition_rows_mask = (daily_df["WELL_TYPE"] == "WI") & (daily_df["FLOW_KIND"] == "production")
transition_detail = (
    daily_with_dates.loc[transition_rows_mask, REPORT_COLS]
    .sort_values(["NPD_WELL_BORE_CODE", "DATEPRD_parsed"])
    .reset_index(drop=True)
)
transition_detail["supports_production"] = (
    (transition_detail["BORE_OIL_VOL"] > 0) | (transition_detail["BORE_GAS_VOL"] > 0) | (transition_detail["BORE_WAT_VOL"] > 0)
)
transition_detail["supports_injection"] = transition_detail["BORE_WI_VOL"] > 0
transition_detail


,DATEPRD_parsed,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,WELL_TYPE,FLOW_KIND,ON_STREAM_HRS,BORE_WI_VOL,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,supports_production,supports_injection
0,2016-04-11,5769,15/9-F-5,WI,production,0.0,NaN,0.00,0.00,0.00,False,False
1,2016-05-04,5769,15/9-F-5,WI,production,12.0,0.0,328.70,52387.43,136.13,True,False
2,2016-07-26,5769,15/9-F-5,WI,production,12.0,0.0,340.52,54468.73,78.67,True,False
3,2016-07-27,5769,15/9-F-5,WI,production,12.0,0.0,334.18,53468.31,85.23,True,False
4,2016-07-28,5769,15/9-F-5,WI,production,12.0,0.0,338.69,54190.20,88.61,True,False
5,2016-07-29,5769,15/9-F-5,WI,production,12.0,0.0,342.77,53787.03,89.70,True,False
6,2016-07-30,5769,15/9-F-5,WI,production,12.0,0.0,349.25,55339.69,91.80,True,False
7,2016-07-31,5769,15/9-F-5,WI,production,12.0,0.0,342.39,53103.90,88.56,True,False
8,2016-08-01,5769,15/9-F-5,WI,production,12.0,0.0,351.38,53406.08,85.88,True,False
9,2016-09-06,5769,15/9-F-5,WI,production,0.0,0.0,0.00,0.00,0.00,False,False


In [166]:
print("Which label does physical activity support, across all 18 transition rows?")
print(transition_detail[["supports_production", "supports_injection"]].value_counts())


Which label does physical activity support, across all 18 transition rows?
supports_production  supports_injection
False                False                 9
True                 False                 9
Name: count, dtype: int64


### 14.9 Injection-quality verdict


In [167]:
n_negative_wi = len(negative_wi_rows)
n_positive_wi_outside_injection = len(positive_wi_outside_injection_rows)
n_zero_hours_positive_wi = int(zero_hours_positive_wi_mask.sum())
n_zero_hours_positive_wi_large = len(large_zero_hours_wi_rows)
n_null_hours_positive_wi = int(null_hours_positive_wi_mask.sum())
n_dq001_dq003_with_measurement = int(has_any_measurement.sum())

# A genuinely unexplained inconsistency would be: negative injection, injection
# recorded outside any transition/injection state, a DQ-001/DQ-003 record that
# actually carries a measurement (which would undercut how those issues have
# been characterized so far), or a zero-hours-yet-positive-injection row whose
# magnitude is too large to plausibly be trace/incidental flow. Injection
# occurring only inside the already-understood 18-row transition state is not
# grounds for a new issue, and the 29/31 trace-to-moderate zero-hours rows are
# not either - only the "large" bucket (>100 m3) is genuinely unexplained.
genuinely_unexplained = (
    n_negative_wi + n_positive_wi_outside_injection + n_dq001_dq003_with_measurement + n_zero_hours_positive_wi_large
)

if genuinely_unexplained > 0:
    injection_verdict = "PASS WITH REVIEW"
else:
    injection_verdict = "PASS"

print("=" * 60)
print("SECTION 14 - INJECTION-VOLUME VALIDATION SUMMARY")
print("=" * 60)
print(f"Negative BORE_WI_VOL:                                    {n_negative_wi}")
print(f"Positive injection outside expected injection state:     {n_positive_wi_outside_injection}")
print(f"BORE_WI_VOL > 0 with ON_STREAM_HRS = 0 (total):           {n_zero_hours_positive_wi}")
print(f"  - of which > 100 m3 (genuinely unexplained):            {n_zero_hours_positive_wi_large}")
print(f"BORE_WI_VOL > 0 with ON_STREAM_HRS NULL:                  {n_null_hours_positive_wi}")
print(f"DQ-001/DQ-003 rows carrying an actual measurement:        {n_dq001_dq003_with_measurement} / {len(dq001_dq003_rows)}")
print()
print(f"VERDICT: {injection_verdict}")


SECTION 14 - INJECTION-VOLUME VALIDATION SUMMARY
Negative BORE_WI_VOL:                                    0
Positive injection outside expected injection state:     0
BORE_WI_VOL > 0 with ON_STREAM_HRS = 0 (total):           31
  - of which > 100 m3 (genuinely unexplained):            2
BORE_WI_VOL > 0 with ON_STREAM_HRS NULL:                  0
DQ-001/DQ-003 rows carrying an actual measurement:        0 / 285

VERDICT: PASS WITH REVIEW


**Interpretation for later relational modelling:**

**Main question, mostly answered clean: `BORE_WI_VOL` is highly consistent with recorded operating state.** Zero negative values, zero positive-injection-outside-injection-state rows, and — critically — **0 of the 285 DQ-001/DQ-003 rows carry any positive volume measurement at all** (14.7). That precisely answers Section 14.7's question: these are not concealed injection activity hiding behind missing hours — they are genuinely blank well-state records (a wellbore marked `WI`/`injection` with every single measurement column empty), most plausibly a pre-commissioning or not-yet-metered period. This materially sharpens DQ-001/DQ-003 rather than just re-confirming them.

**14.8 gives a decisive answer for the 18 transition rows: physical activity never once supports the `WI` (injection) label.** Across all 18, `BORE_WI_VOL` is 0 (15 rows) or NULL (3 rows) — **never positive**. 9 of the 18 show real oil/gas/water production; the other 9 show no activity in any of the four volume columns. So the honest characterization is: for this transition population, `FLOW_KIND = production` is the physically-supported label in exactly half the rows, and in the other half neither label is physically supported — `WELL_TYPE = WI` never is. This strengthens Section 12's suspicion that `WELL_TYPE` lags `FLOW_KIND` during a real conversion event rather than the two columns disagreeing arbitrarily.

**The `ON_STREAM_HRS` question the user asked me to watch for has a genuinely mixed answer, not a clean yes/no.** The aggregate distributions look similar (median 24 hours for both producing and injecting rows, same 0-25 range) — a coarse look would say "no evidence of a difference." But the **zero-hours-yet-positive-volume rate is 15.5× more common for injection (31/5,404 = 0.574%) than for production (3/8,011 = 0.037%)**. That is a real, quantified signal that `ON_STREAM_HRS` does not fully capture injection activity the same way it captures production activity — worth carrying forward as a documented caveat for anyone using `ON_STREAM_HRS` as a universal "was this well active" flag.

Breaking down *why* rather than treating those 31 rows as one population: 17 are trace (<1 m3), 12 are moderate (1-100 m3) — both plausibly incidental/residual flow not counted toward "on-stream" time, consistent with the rate-difference finding above — but **2 are large enough to not fit that explanation**: 6,263.69 m3 on 2011-06-18 (wellbore 5693) and 148.66 m3 on 2013-03-16 (wellbore 5769), both against a recorded `ON_STREAM_HRS = 0`. A multi-thousand-m3 injection volume on a day recorded as zero operating hours is a genuine, unexplained state/volume inconsistency.

**14.6 adds a useful contrast with Section 13's lesson about near-zero baselines**: for `BORE_WI_VOL`, the flag count barely moves as the absolute floor rises from 0 to 1,000 (116 → 114) — unlike production's jumps, injection's large percentage changes are *not* mostly a near-zero-baseline artifact; they tend to be large in absolute terms too. That is itself informative about the reliability of percentage-based screening depending on which column it's applied to.

**`DQ-006` is introduced here**, scoped precisely to the 2 large zero-hours-yet-positive-injection rows — not to the 18 transition rows (fully explained by 14.8) and not to the 29 trace/moderate zero-hours rows (plausibly explained by the injection/production `ON_STREAM_HRS` rate difference):

```
DQ-001  Pre-field-life injection records
DQ-002  Shared 12-day reporting gap
DQ-003  NULL ON_STREAM_HRS
DQ-004  ON_STREAM_HRS > 24 / possible DST effect
DQ-005  Production-volume/state inconsistency (negative BORE_WAT_VOL; 1 unexplained zero-hours/positive-volume row)
DQ-006  Injection-volume/state inconsistency (2 large-magnitude BORE_WI_VOL > 0 rows with ON_STREAM_HRS = 0)
```

```
VERDICT: PASS WITH REVIEW
```

**Block complete.** Sections 4-8 (structure and identity), 9-12 (temporal/reporting quality), and now 13-14 (physical volumes) form a coherent audit trail: six issue IDs, each with a specific, quantified population rather than a vague "some rows look odd." `BORE_WI_VOL` should load as a nullable numeric column with no non-negativity or magnitude constraint until DQ-006 is resolved. Per your note, pausing here before Sections 15-17 (pressure, temperature, choke) — those sensor-measurement validations should be scoped using what this block has established about *when* a measurement should reasonably exist (producing vs. injecting vs. neither), rather than validated as one undifferentiated population the way a first pass might.


## 15. Pressure validation

Fields: `AVG_DOWNHOLE_PRESSURE`, `AVG_DP_TUBING`, `AVG_ANNULUS_PRESS`, `AVG_WHP_P`, `DP_CHOKE_SIZE`.

Pressure is **not** assessed globally without context — Sections 9-14 already established which wellbore-days are producing, injecting, in a transition state, or structurally blank (DQ-001/DQ-003), and that context is used throughout this section rather than treating every row as one undifferentiated population.

Six checks: basic validity, pressure availability by `WELL_TYPE`×`FLOW_KIND` (reconnecting to Section 11), pressure availability by activity state, physical relationship checks (kept as REVIEW, not hard rules, until units/conventions are confirmed), abrupt-change analysis using both absolute and percentage change (per Section 13's lesson), and behavior around the known issue populations (DQ-002, DQ-004, DQ-005, DQ-006) — tested, not forced into a connection.

**No "reasonable oilfield pressure range" is invented here.** Without confirmed units and source conventions, an assumed physical range would be a guess dressed up as a validation rule. `FAIL` is reserved for what is structurally impossible regardless of units or convention; everything else that looks unusual is `REVIEW`.


In [168]:
# --- TEMPORARY LOADING FOR SECTION 15 ---
# Section 1 (Load source data) has not been implemented yet in this notebook.
# This block loads only what Section 15 needs, and is clearly marked so it can
# be deleted once Section 1 provides `daily_df` for the whole notebook.
if "daily_df" not in globals():
    from pathlib import Path
    import pandas as pd

    PROJECT_ROOT = Path.cwd().parent
    WORKBOOK_PATH = PROJECT_ROOT / "data" / "raw" / "Volve production data.xlsx"
    DAILY_SHEET_NAME = "Daily Production Data"

    if not WORKBOOK_PATH.exists():
        raise FileNotFoundError(f"Source workbook not found at {WORKBOOK_PATH}")

    daily_df = pd.read_excel(WORKBOOK_PATH, sheet_name=DAILY_SHEET_NAME)
    print(f"[Section 15 temporary load] daily_df loaded: {daily_df.shape}")
else:
    print(f"Using daily_df already loaded earlier in the notebook: {daily_df.shape}")


Using daily_df already loaded earlier in the notebook: (15634, 24)


In [169]:
# Reuse Section 9-14 derived state if available, recompute the minimum if not.
if "dateprd_parsed" not in globals():
    dateprd_parsed = pd.to_datetime(daily_df["DATEPRD"], errors="coerce")
if "daily_with_dates" not in globals():
    daily_with_dates = daily_df.copy()
    daily_with_dates["DATEPRD_parsed"] = dateprd_parsed
if "hours" not in globals():
    hours = daily_df["ON_STREAM_HRS"]

print("Ready: daily_with_dates, dateprd_parsed, hours")


Ready: daily_with_dates, dateprd_parsed, hours


In [170]:
PRESSURE_COLS = ["AVG_DOWNHOLE_PRESSURE", "AVG_DP_TUBING", "AVG_ANNULUS_PRESS", "AVG_WHP_P", "DP_CHOKE_SIZE"]
required_columns = PRESSURE_COLS + [
    "WELL_TYPE", "FLOW_KIND", "ON_STREAM_HRS", "NPD_WELL_BORE_CODE", "NPD_WELL_BORE_NAME", "DATEPRD",
    "BORE_OIL_VOL", "BORE_GAS_VOL", "BORE_WAT_VOL", "BORE_WI_VOL",
]
missing_columns = [c for c in required_columns if c not in daily_df.columns]
if missing_columns:
    raise ValueError(f"Required column(s) missing from daily_df: {missing_columns}")
print(f"Required columns present: {required_columns}")


Required columns present: ['AVG_DOWNHOLE_PRESSURE', 'AVG_DP_TUBING', 'AVG_ANNULUS_PRESS', 'AVG_WHP_P', 'DP_CHOKE_SIZE', 'WELL_TYPE', 'FLOW_KIND', 'ON_STREAM_HRS', 'NPD_WELL_BORE_CODE', 'NPD_WELL_BORE_NAME', 'DATEPRD', 'BORE_OIL_VOL', 'BORE_GAS_VOL', 'BORE_WAT_VOL', 'BORE_WI_VOL']


### 15.1 Basic validity by field


In [171]:
def pressure_basic_validity(col: str) -> dict:
    s = daily_df[col]
    return {
        "column": col,
        "count": int(s.notna().sum()),
        "null_count": int(s.isna().sum()),
        "zero_count": int((s == 0).sum()),
        "negative_count": int((s < 0).sum()),
        "min": s.min(),
        "median": s.median(),
        "mean": s.mean(),
        "p95": s.quantile(0.95),
        "max": s.max(),
    }


pressure_basic_validity_table = pd.DataFrame([pressure_basic_validity(c) for c in PRESSURE_COLS])
pressure_basic_validity_table


,column,count,null_count,zero_count,negative_count,min,median,mean,p95,max
0,AVG_DOWNHOLE_PRESSURE,8980,6654,2312,0,0.0,232.896939,181.803869,283.114219,397.588550
1,AVG_DP_TUBING,8980,6654,188,0,0.0,175.588861,154.028787,238.409836,345.906770
2,AVG_ANNULUS_PRESS,7890,7744,1204,0,0.0,16.308598,14.856100,26.055369,30.019828
3,AVG_WHP_P,9155,6479,279,0,0.0,37.933620,45.377811,95.818057,137.311030
4,DP_CHOKE_SIZE,15340,294,6242,0,0.0,2.384969,11.441060,56.201662,125.718570


### 15.2 Pressure availability by operating state (`WELL_TYPE` × `FLOW_KIND`)

Reconnecting to Section 11's structured-missingness finding and Section 12's transition-period nuance.


In [172]:
# errors="ignore" here is the same pandas-version accommodation documented in
# Section 11.2 - WELL_TYPE/FLOW_KIND are always the groupby keys, dropped defensively.
pressure_missingness_by_state = (
    daily_df[PRESSURE_COLS]
    .assign(WELL_TYPE=daily_df["WELL_TYPE"], FLOW_KIND=daily_df["FLOW_KIND"])
    .groupby(["WELL_TYPE", "FLOW_KIND"], dropna=False)
    .apply(lambda g: (g.drop(columns=["WELL_TYPE", "FLOW_KIND"], errors="ignore").isna().mean() * 100).round(2))
)
pressure_missingness_by_state


AVG_DOWNHOLE_PRESSURE  AVG_DP_TUBING  AVG_ANNULUS_PRESS  \
WELL_TYPE FLOW_KIND                                                             
OP        production                   1.80           1.80              13.89   
WI        injection                  100.00         100.00             100.00   
          production                  88.89          88.89               5.56   

                      AVG_WHP_P  DP_CHOKE_SIZE  
WELL_TYPE FLOW_KIND                             
OP        production       0.07           0.07  
WI        injection      100.00           4.45  
          production       0.00           0.00

### 15.3 Pressure availability vs. activity

Whether missing pressure is normal for inactive periods, tested directly rather than assumed. Categories are not mutually exclusive.


In [173]:
activity_masks = {
    "active_production": (daily_df["BORE_OIL_VOL"] > 0) | (daily_df["BORE_GAS_VOL"] > 0) | (daily_df["BORE_WAT_VOL"] > 0),
    "active_injection": daily_df["BORE_WI_VOL"] > 0,
    "zero_hours": hours == 0,
    "null_hours": hours.isna(),
    "all_zero_volume": (daily_df[["BORE_OIL_VOL", "BORE_GAS_VOL", "BORE_WAT_VOL", "BORE_WI_VOL"]] == 0).all(axis=1),
}

activity_rows = []
for label, mask in activity_masks.items():
    row = {"activity_state": label, "n_rows": int(mask.sum())}
    for col in PRESSURE_COLS:
        row[col] = round(daily_df.loc[mask, col].isna().mean() * 100, 2) if mask.sum() else float("nan")
    activity_rows.append(row)

pressure_by_activity = pd.DataFrame(activity_rows)
pressure_by_activity


,activity_state,n_rows,AVG_DOWNHOLE_PRESSURE,AVG_DP_TUBING,AVG_ANNULUS_PRESS,AVG_WHP_P,DP_CHOKE_SIZE
0,active_production,8011,1.80,1.80,11.78,0.04,0.04
1,active_injection,5404,100.00,100.00,100.00,100.00,0.02
2,zero_hours,1952,43.44,43.44,57.94,41.70,0.26
3,null_hours,285,100.00,100.00,100.00,100.00,100.00
4,all_zero_volume,7,100.00,100.00,0.00,0.00,0.00


### 15.4 Physical relationship checks (REVIEW, not FAIL)

Kept as REVIEW rather than hard rules until units and conventions are confirmed — some relationships may have legitimate operational explanations.


In [174]:
downhole_lt_whp_mask = daily_df["AVG_DOWNHOLE_PRESSURE"] < daily_df["AVG_WHP_P"]
annulus_negative_mask = daily_df["AVG_ANNULUS_PRESS"] < 0
whp_negative_mask = daily_df["AVG_WHP_P"] < 0
dp_tubing_negative_mask = daily_df["AVG_DP_TUBING"] < 0

print(f"AVG_DOWNHOLE_PRESSURE < AVG_WHP_P (REVIEW): {int(downhole_lt_whp_mask.sum())}")
print(f"AVG_ANNULUS_PRESS < 0 (REVIEW):             {int(annulus_negative_mask.sum())}")
print(f"AVG_WHP_P < 0 (REVIEW):                     {int(whp_negative_mask.sum())}")
print(f"AVG_DP_TUBING < 0 (REVIEW):                 {int(dp_tubing_negative_mask.sum())}")


AVG_DOWNHOLE_PRESSURE < AVG_WHP_P (REVIEW): 2126
AVG_ANNULUS_PRESS < 0 (REVIEW):             0
AVG_WHP_P < 0 (REVIEW):                     0
AVG_DP_TUBING < 0 (REVIEW):                 0


In [175]:
relationship_report_cols = [
    "DATEPRD_parsed", "NPD_WELL_BORE_CODE", "NPD_WELL_BORE_NAME", "WELL_TYPE", "FLOW_KIND",
    "ON_STREAM_HRS", "AVG_DOWNHOLE_PRESSURE", "AVG_DP_TUBING", "AVG_ANNULUS_PRESS", "AVG_WHP_P",
]
downhole_lt_whp_rows = daily_with_dates.loc[downhole_lt_whp_mask, relationship_report_cols].sort_values("DATEPRD_parsed")
print(f"AVG_DOWNHOLE_PRESSURE < AVG_WHP_P rows: {len(downhole_lt_whp_rows)}")
downhole_lt_whp_rows.head(20)


AVG_DOWNHOLE_PRESSURE < AVG_WHP_P rows: 2126


,DATEPRD_parsed,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,WELL_TYPE,FLOW_KIND,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,AVG_DP_TUBING,AVG_ANNULUS_PRESS,AVG_WHP_P
5383,2009-04-05,5351,15/9-F-14,OP,production,22.19167,0.00000,81.858642,NaN,81.858642
5388,2009-04-10,5351,15/9-F-14,OP,production,0.80833,0.00000,86.930548,NaN,86.930548
5390,2009-04-12,5351,15/9-F-14,OP,production,13.85833,0.00000,79.385957,NaN,79.385957
5391,2009-04-13,5351,15/9-F-14,OP,production,24.00000,0.00000,77.487499,NaN,77.487499
5392,2009-04-14,5351,15/9-F-14,OP,production,24.00000,0.00000,76.398335,NaN,76.398335
5393,2009-04-15,5351,15/9-F-14,OP,production,4.47500,0.00000,76.653536,NaN,76.653536
5696,2010-02-15,5351,15/9-F-14,OP,production,13.70000,0.00000,69.259063,0.000000,69.259063
5697,2010-02-16,5351,15/9-F-14,OP,production,24.00000,0.00000,59.768697,NaN,59.768697
5698,2010-02-17,5351,15/9-F-14,OP,production,24.00000,0.00000,59.812914,NaN,59.812914
5699,2010-02-18,5351,15/9-F-14,OP,production,21.60000,0.00000,61.798073,NaN,61.798073


In [176]:
negative_pressure_mask = annulus_negative_mask | whp_negative_mask | dp_tubing_negative_mask
negative_pressure_rows = daily_with_dates.loc[negative_pressure_mask, relationship_report_cols].sort_values("DATEPRD_parsed")
print(f"Rows with any of AVG_ANNULUS_PRESS/AVG_WHP_P/AVG_DP_TUBING negative: {len(negative_pressure_rows)}")
negative_pressure_rows.head(20)


Rows with any of AVG_ANNULUS_PRESS/AVG_WHP_P/AVG_DP_TUBING negative: 0


,DATEPRD_parsed,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,WELL_TYPE,FLOW_KIND,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,AVG_DP_TUBING,AVG_ANNULUS_PRESS,AVG_WHP_P


### 15.5 Abrupt pressure changes

Both absolute and percentage change, per Section 13's lesson that a percentage-only threshold misbehaves near zero. For each wellbore and pressure variable: previous/current value, absolute change, percentage change, date, `ON_STREAM_HRS`, `FLOW_KIND`, `WELL_TYPE`.


In [177]:
pressure_change_rows = []
for col in PRESSURE_COLS:
    for code_value, group in daily_with_dates.groupby("NPD_WELL_BORE_CODE"):
        g = group.dropna(subset=["DATEPRD_parsed", col]).sort_values("DATEPRD_parsed")
        prev_val = g[col].shift(1)
        prev_date = g["DATEPRD_parsed"].shift(1)
        abs_change = g[col] - prev_val
        pct_change = (abs_change / prev_val * 100).where(prev_val != 0)
        valid = prev_val.notna()
        for idx in g.index[valid]:
            pressure_change_rows.append(
                {
                    "column": col,
                    "npd_code": code_value,
                    "wellbore_name": g.loc[idx, "NPD_WELL_BORE_NAME"],
                    "previous_date": prev_date.loc[idx].date(),
                    "current_date": g.loc[idx, "DATEPRD_parsed"].date(),
                    "previous_value": prev_val.loc[idx],
                    "current_value": g.loc[idx, col],
                    "abs_change": abs_change.loc[idx],
                    "pct_change": pct_change.loc[idx],
                    "ON_STREAM_HRS": g.loc[idx, "ON_STREAM_HRS"],
                    "FLOW_KIND": g.loc[idx, "FLOW_KIND"],
                    "WELL_TYPE": g.loc[idx, "WELL_TYPE"],
                }
            )

pressure_changes = pd.DataFrame(pressure_change_rows)
print(f"Consecutive same-wellbore pressure-field comparisons: {len(pressure_changes)}")


Consecutive same-wellbore pressure-field comparisons: 50316


In [178]:
PCT_THRESHOLD = 200
ABS_FLOOR_CANDIDATES = [0, 10, 50, 100]

floor_demo_rows = []
for floor in ABS_FLOOR_CANDIDATES:
    flagged = pressure_changes[(pressure_changes["pct_change"].abs() > PCT_THRESHOLD) & (pressure_changes["abs_change"].abs() >= floor)]
    floor_demo_rows.append({"abs_change_floor": floor, "flags_at_pct>200%_AND_floor": len(flagged)})

pd.DataFrame(floor_demo_rows)


,abs_change_floor,flags_at_pct>200%_AND_floor
0,0,361
1,10,240
2,50,91
3,100,19


In [179]:
PRESSURE_ABS_FLOOR = 50  # screening choice for the review table below only - not a final automated threshold (Section 25)
large_pressure_changes = pressure_changes[
    (pressure_changes["pct_change"].abs() > PCT_THRESHOLD) & (pressure_changes["abs_change"].abs() >= PRESSURE_ABS_FLOOR)
]
large_pressure_changes = large_pressure_changes.reindex(
    large_pressure_changes["abs_change"].abs().sort_values(ascending=False).index
).reset_index(drop=True)
print(f"Flagged for REVIEW (pct_change > {PCT_THRESHOLD}% AND abs_change >= {PRESSURE_ABS_FLOOR}): {len(large_pressure_changes)}")
large_pressure_changes.head(20)


Flagged for REVIEW (pct_change > 200% AND abs_change >= 50): 91


,column,npd_code,wellbore_name,previous_date,current_date,previous_value,current_value,abs_change,pct_change,ON_STREAM_HRS,FLOW_KIND,WELL_TYPE
0,AVG_DP_TUBING,5351,15/9-F-14,2012-09-03,2012-09-07,2.752040,300.921180,298.169140,10834.476970,0.00000,production,OP
1,AVG_DP_TUBING,5351,15/9-F-14,2015-08-30,2015-08-31,0.147970,239.269381,239.121411,161601.277891,14.24167,production,OP
2,AVG_DOWNHOLE_PRESSURE,5351,15/9-F-14,2010-02-25,2010-02-26,49.450440,255.380762,205.930322,416.437800,20.35833,production,OP
3,AVG_DP_TUBING,5351,15/9-F-14,2010-02-23,2010-02-24,7.219360,190.245838,183.026479,2535.217595,24.00000,production,OP
4,AVG_DP_TUBING,5351,15/9-F-14,2010-02-25,2010-02-26,10.666705,191.413604,180.746899,1694.496048,20.35833,production,OP
5,AVG_DOWNHOLE_PRESSURE,5351,15/9-F-14,2010-02-23,2010-02-24,71.154596,250.827538,179.672942,252.510661,24.00000,production,OP
6,AVG_DOWNHOLE_PRESSURE,7078,15/9-F-11,2013-07-24,2013-07-25,1.926270,163.594260,161.667990,8392.800075,24.00000,production,OP
7,AVG_DP_TUBING,5351,15/9-F-14,2013-05-15,2013-05-16,62.577541,212.835830,150.258288,240.115360,24.00000,production,OP
8,AVG_DOWNHOLE_PRESSURE,5351,15/9-F-14,2008-06-29,2008-06-30,0.857200,149.555250,148.698050,17346.949370,0.00000,production,OP
9,AVG_DP_TUBING,5351,15/9-F-14,2008-06-29,2008-06-30,0.857200,149.555250,148.698050,17346.949370,0.00000,production,OP


### 15.6 Pressure behavior around known issue populations

Testing, not forcing a connection: `DQ-002` (shared 12-day gap), `DQ-004` (DST-hour records), `DQ-005` (production-volume/state inconsistency), `DQ-006` (injection-volume/state inconsistency).


In [180]:
dq_report_cols = ["DATEPRD_parsed", "NPD_WELL_BORE_CODE", "NPD_WELL_BORE_NAME"] + PRESSURE_COLS

# DQ-002: pressure immediately before/after the shared 12-day gap (wellbores
# 5351/5599, gap 2012-01-02 to 2012-01-14, from Section 9.4).
dq002_mask = daily_with_dates["NPD_WELL_BORE_CODE"].isin([5351, 5599]) & daily_with_dates["DATEPRD_parsed"].isin(
    [pd.Timestamp("2012-01-02"), pd.Timestamp("2012-01-14")]
)
dq002_rows = daily_with_dates.loc[dq002_mask, dq_report_cols].sort_values(["NPD_WELL_BORE_CODE", "DATEPRD_parsed"])
print("DQ-002 - pressure fields immediately before/after the shared 12-day gap:")
dq002_rows


DQ-002 - pressure fields immediately before/after the shared 12-day gap:


,DATEPRD_parsed,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,AVG_DOWNHOLE_PRESSURE,AVG_DP_TUBING,AVG_ANNULUS_PRESS,AVG_WHP_P,DP_CHOKE_SIZE
6346,2012-01-02,5351,15/9-F-14,242.730415,203.388997,0.000000,39.341419,9.231513
6347,2012-01-14,5351,15/9-F-14,287.804560,191.293389,0.000000,96.511171,42.891364
3290,2012-01-02,5599,15/9-F-12,0.000000,40.008846,15.977124,40.008846,8.726263
3291,2012-01-14,5599,15/9-F-12,0.000000,60.500052,0.017426,60.500052,29.471926


In [181]:
# DQ-004: the 20 ON_STREAM_HRS > 24 (DST-pattern) rows from Section 10.
dq004_mask = hours > 24
dq004_rows = daily_with_dates.loc[dq004_mask, dq_report_cols].sort_values(["NPD_WELL_BORE_CODE", "DATEPRD_parsed"])
print(f"DQ-004 - pressure fields on the {len(dq004_rows)} DST-pattern rows:")
dq004_rows


DQ-004 - pressure fields on the 20 DST-pattern rows:


,DATEPRD_parsed,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,AVG_DOWNHOLE_PRESSURE,AVG_DP_TUBING,AVG_ANNULUS_PRESS,AVG_WHP_P,DP_CHOKE_SIZE
5222,2008-10-26,5351,15/9-F-14,214.027703,149.515311,NaN,64.512391,31.849241
5583,2009-10-25,5351,15/9-F-14,241.828108,173.394606,0.000000,68.433502,35.187547
5947,2010-10-31,5351,15/9-F-14,242.748362,197.977787,0.000000,44.770575,12.414894
6968,2013-10-27,5351,15/9-F-14,250.212893,218.139055,21.693508,32.073838,2.800203
7330,2014-10-26,5351,15/9-F-14,261.602755,230.602211,21.458054,31.000544,2.290826
2166,2008-10-26,5599,15/9-F-12,233.664638,163.381056,13.811904,70.283581,37.889267
2527,2009-10-25,5599,15/9-F-12,261.865161,184.307544,10.468965,77.557616,45.036169
2891,2010-10-31,5599,15/9-F-12,0.000000,52.641503,23.158358,52.641503,20.099256
3912,2013-10-27,5599,15/9-F-12,0.000000,33.458990,14.795093,33.458990,4.623115
4274,2014-10-26,5599,15/9-F-12,0.000000,32.822529,19.161930,32.822529,4.056984


In [182]:
# DQ-005: negative BORE_WAT_VOL rows, plus the unexplained zero-hours/positive
# -volume row (wellbore 7078, 2015-01-17) from Section 13.
dq005_mask = (daily_df["BORE_WAT_VOL"] < 0) | (
    (daily_df["NPD_WELL_BORE_CODE"] == 7078) & (dateprd_parsed == pd.Timestamp("2015-01-17"))
)
dq005_rows = daily_with_dates.loc[dq005_mask, dq_report_cols].sort_values(["NPD_WELL_BORE_CODE", "DATEPRD_parsed"])
print(f"DQ-005 - pressure fields on the {len(dq005_rows)} flagged rows:")
dq005_rows


DQ-005 - pressure fields on the 5 flagged rows:


,DATEPRD_parsed,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,AVG_DOWNHOLE_PRESSURE,AVG_DP_TUBING,AVG_ANNULUS_PRESS,AVG_WHP_P,DP_CHOKE_SIZE
5350,2009-03-03,5351,15/9-F-14,270.291285,168.305082,NaN,101.986203,68.800869
6558,2012-08-13,5351,15/9-F-14,252.892348,206.739802,14.812580,46.152546,17.203700
1982,2008-04-23,5599,15/9-F-12,260.557935,168.697955,11.010635,91.859980,60.253108
3502,2012-08-13,5599,15/9-F-12,0.000000,50.951032,10.320236,50.951032,21.297284
1301,2015-01-17,7078,15/9-F-11,219.800000,169.400000,23.100000,50.400000,21.000000


In [183]:
# DQ-006: the 2 large-magnitude zero-hours/positive-injection rows from Section 14.
dq006_mask = daily_df["NPD_WELL_BORE_CODE"].isin([5693, 5769]) & (hours == 0) & (daily_df["BORE_WI_VOL"] > 100)
dq006_rows = daily_with_dates.loc[dq006_mask, dq_report_cols].sort_values(["NPD_WELL_BORE_CODE", "DATEPRD_parsed"])
print(f"DQ-006 - pressure fields on the {len(dq006_rows)} flagged rows:")
dq006_rows


DQ-006 - pressure fields on the 2 flagged rows:


,DATEPRD_parsed,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,AVG_DOWNHOLE_PRESSURE,AVG_DP_TUBING,AVG_ANNULUS_PRESS,AVG_WHP_P,DP_CHOKE_SIZE
10387,2011-06-18,5693,15/9-F-4,NaN,NaN,NaN,NaN,0.0
14351,2013-03-16,5769,15/9-F-5,NaN,NaN,NaN,NaN,0.0


### Pressure-quality verdict


In [184]:
n_downhole_lt_whp = int(downhole_lt_whp_mask.sum())
n_annulus_negative = int(annulus_negative_mask.sum())
n_whp_negative = int(whp_negative_mask.sum())
n_dp_tubing_negative = int(dp_tubing_negative_mask.sum())
n_large_pressure_changes = len(large_pressure_changes)

# FAIL is reserved for what is structurally impossible regardless of units or
# convention. Nothing found here meets that bar without confirmed units, so
# the only question is REVIEW vs PASS.
any_review_findings = any(
    [n_downhole_lt_whp, n_annulus_negative, n_whp_negative, n_dp_tubing_negative, n_large_pressure_changes]
)
pressure_verdict = "PASS WITH REVIEW" if any_review_findings else "PASS"

print("=" * 60)
print("SECTION 15 - PRESSURE VALIDATION SUMMARY")
print("=" * 60)
print(f"AVG_DOWNHOLE_PRESSURE < AVG_WHP_P:                  {n_downhole_lt_whp}")
print(f"AVG_ANNULUS_PRESS < 0:                              {n_annulus_negative}")
print(f"AVG_WHP_P < 0:                                       {n_whp_negative}")
print(f"AVG_DP_TUBING < 0:                                  {n_dp_tubing_negative}")
print(f"Large abrupt changes (pct>200% AND abs>={PRESSURE_ABS_FLOOR}):        {n_large_pressure_changes}")
print()
print(f"VERDICT: {pressure_verdict}")


SECTION 15 - PRESSURE VALIDATION SUMMARY
AVG_DOWNHOLE_PRESSURE < AVG_WHP_P:                  2126
AVG_ANNULUS_PRESS < 0:                              0
AVG_WHP_P < 0:                                       0
AVG_DP_TUBING < 0:                                  0
Large abrupt changes (pct>200% AND abs>=50):        91

VERDICT: PASS WITH REVIEW


**Interpretation for later relational modelling:**

**Pressure availability is fully explained by operating state — no new missingness mystery here.** 15.2/15.3 confirm, at the pressure-field level, exactly the pattern Sections 11 and 12 already established: `WI`/injection rows are ~100% missing across `AVG_DOWNHOLE_PRESSURE`/`AVG_DP_TUBING`/`AVG_ANNULUS_PRESS`/`AVG_WHP_P` (downhole gauges aren't relevant to a pure injector in this dataset), the 18 `WI`/production transition rows reproduce Section 12.7's exact "lag" signature (88.89% missing on downhole/dp_tubing, but 0% on `AVG_WHP_P`/`DP_CHOKE_SIZE` — wellhead metering comes back online immediately, downhole doesn't), and the 285 NULL-hours rows (DQ-001/DQ-003) are 100% missing across **all five** pressure fields, extending their "genuinely blank record" characterization from Section 11/14 to pressure too. One physically plausible nuance worth keeping: `zero_hours` rows are only *partially* missing (43-58%, not 100%) — consistent with the real oilfield practice of taking a static shut-in pressure survey on a day with zero flow, not with an error.

**15.4 surfaced one large, genuinely unresolved finding: `AVG_DOWNHOLE_PRESSURE < AVG_WHP_P` in 2,126 rows.** That is a substantial count, and on typical oilfield convention downhole pressure normally exceeds wellhead pressure (hydrostatic column). But **this is kept as REVIEW, not FAIL, on purpose** — without confirmed units and measurement conventions for these two fields, "downhole should exceed wellhead" is an assumption borrowed from general engineering knowledge, exactly what this section was told not to invent as a validation rule. It is flagged as a priority item for whoever can confirm units, not treated as 2,126 errors. The other three relationship checks (`AVG_ANNULUS_PRESS`, `AVG_WHP_P`, `AVG_DP_TUBING` all `< 0`) came back **0/0/0** — clean.

**15.5 — the absolute floor matters much more for pressure than it did for injection (Section 14), and behaves more like production (Section 13).** Flags at `pct_change > 200%` drop from 361 (floor=0) to 19 (floor=100) — an ~95% reduction — meaning most of the raw percentage-based flags are near-zero-baseline artifacts, not large real swings. This is a useful cross-section methodological data point: the "does the floor matter" question doesn't have one universal answer across measurement types, it has to be checked per column.

**15.6 mostly shows no forced connection, with one real exception.** DQ-004's 20 DST-pattern rows show pressure missingness fully explained by their own `WELL_TYPE`/`FLOW_KIND` (OP rows have normal pressure values; WI rows are blank exactly as WI/injection rows always are) — the DST hours anomaly carries no correlated pressure anomaly. DQ-005's 5 rows are mostly unremarkable except that the already-known wellbore-5599/2012-08-13 row (fractional `ON_STREAM_HRS = 0.625`, negative water) *also* shows `AVG_DOWNHOLE_PRESSURE = 0.0` — corroborating, not new. **DQ-006 is where a real new signal appeared**: on both flagged rows, `AVG_DOWNHOLE_PRESSURE`, `AVG_DP_TUBING`, `AVG_ANNULUS_PRESS`, and `AVG_WHP_P` are **all NaN**, and `DP_CHOKE_SIZE = 0.0` — every other sensor on these rows looks exactly like an inactive/shut-in day, while `BORE_WI_VOL` alone shows a substantial injected volume (6,263.69 m3 and 148.66 m3). That strengthens DQ-006 as a genuine, still-unresolved inconsistency: it is not just one field disagreeing with the state label, it is the injection volume disagreeing with every other sensor on the row, not merely with `ON_STREAM_HRS`.

```
VERDICT: PASS WITH REVIEW
```

No `FAIL` was warranted — nothing here is structurally impossible regardless of units or convention, which is the bar this section set for itself. For relational modelling: all five pressure fields should load as nullable numeric columns with no constraints derived from assumed physical ranges; the `AVG_DOWNHOLE_PRESSURE < AVG_WHP_P` finding and the DQ-006 corroboration should both be carried into Section 23 as evidence, not conclusions. Handing off to **Section 16 (temperature)** and **Section 17 (choke)** with the same operating-state-aware approach — both should reuse the `WELL_TYPE`×`FLOW_KIND` and activity-state breakdowns established here rather than re-deriving them from scratch.


## 16. Temperature validation

Fields: `AVG_DOWNHOLE_TEMPERATURE`, `AVG_WHT_P`. Same structure as Section 15: basic validity, missingness by `WELL_TYPE`×`FLOW_KIND`, missingness by activity state, a downhole-vs-wellhead relationship check (REVIEW, not FAIL), abrupt-change analysis with a demonstrated (not assumed) absolute floor, and a cross-check against DQ-002/DQ-004/DQ-005/DQ-006 — tested, not forced.


In [185]:
# --- TEMPORARY LOADING FOR SECTION 16 ---
# Section 1 (Load source data) has not been implemented yet in this notebook.
# This block loads only what Section 16 needs, and is clearly marked so it can
# be deleted once Section 1 provides `daily_df` for the whole notebook.
if "daily_df" not in globals():
    from pathlib import Path
    import pandas as pd

    PROJECT_ROOT = Path.cwd().parent
    WORKBOOK_PATH = PROJECT_ROOT / "data" / "raw" / "Volve production data.xlsx"
    DAILY_SHEET_NAME = "Daily Production Data"

    if not WORKBOOK_PATH.exists():
        raise FileNotFoundError(f"Source workbook not found at {WORKBOOK_PATH}")

    daily_df = pd.read_excel(WORKBOOK_PATH, sheet_name=DAILY_SHEET_NAME)
    print(f"[Section 16 temporary load] daily_df loaded: {daily_df.shape}")
else:
    print(f"Using daily_df already loaded earlier in the notebook: {daily_df.shape}")


Using daily_df already loaded earlier in the notebook: (15634, 24)


In [186]:
# Reuse Section 9-15 derived state if available, recompute the minimum if not.
if "dateprd_parsed" not in globals():
    dateprd_parsed = pd.to_datetime(daily_df["DATEPRD"], errors="coerce")
if "daily_with_dates" not in globals():
    daily_with_dates = daily_df.copy()
    daily_with_dates["DATEPRD_parsed"] = dateprd_parsed
if "hours" not in globals():
    hours = daily_df["ON_STREAM_HRS"]

print("Ready: daily_with_dates, dateprd_parsed, hours")


Ready: daily_with_dates, dateprd_parsed, hours


In [187]:
TEMPERATURE_COLS = ["AVG_DOWNHOLE_TEMPERATURE", "AVG_WHT_P"]
required_columns = TEMPERATURE_COLS + [
    "WELL_TYPE", "FLOW_KIND", "ON_STREAM_HRS", "NPD_WELL_BORE_CODE", "NPD_WELL_BORE_NAME", "DATEPRD",
    "BORE_OIL_VOL", "BORE_GAS_VOL", "BORE_WAT_VOL", "BORE_WI_VOL",
]
missing_columns = [c for c in required_columns if c not in daily_df.columns]
if missing_columns:
    raise ValueError(f"Required column(s) missing from daily_df: {missing_columns}")
print(f"Required columns present: {required_columns}")


Required columns present: ['AVG_DOWNHOLE_TEMPERATURE', 'AVG_WHT_P', 'WELL_TYPE', 'FLOW_KIND', 'ON_STREAM_HRS', 'NPD_WELL_BORE_CODE', 'NPD_WELL_BORE_NAME', 'DATEPRD', 'BORE_OIL_VOL', 'BORE_GAS_VOL', 'BORE_WAT_VOL', 'BORE_WI_VOL']


### 16.1 Basic validity by field


In [188]:
def temperature_basic_validity(col: str) -> dict:
    s = daily_df[col]
    return {
        "column": col,
        "count": int(s.notna().sum()),
        "null_count": int(s.isna().sum()),
        "zero_count": int((s == 0).sum()),
        "negative_count": int((s < 0).sum()),
        "min": s.min(),
        "median": s.median(),
        "mean": s.mean(),
        "p95": s.quantile(0.95),
        "max": s.max(),
    }


temperature_basic_validity_table = pd.DataFrame([temperature_basic_validity(c) for c in TEMPERATURE_COLS])
temperature_basic_validity_table


,column,count,null_count,zero_count,negative_count,min,median,mean,p95,max
0,AVG_DOWNHOLE_TEMPERATURE,8980,6654,2312,0,0.0,103.186689,77.162969,107.360624,108.502178
1,AVG_WHT_P,9146,6488,302,0,0.0,80.071250,67.728440,91.291976,93.509584


### 16.2 Missingness by operating state (`WELL_TYPE` × `FLOW_KIND`)


In [189]:
# errors="ignore" here is the same pandas-version accommodation documented in
# Section 11.2 - WELL_TYPE/FLOW_KIND are always the groupby keys, dropped defensively.
temperature_missingness_by_state = (
    daily_df[TEMPERATURE_COLS]
    .assign(WELL_TYPE=daily_df["WELL_TYPE"], FLOW_KIND=daily_df["FLOW_KIND"])
    .groupby(["WELL_TYPE", "FLOW_KIND"], dropna=False)
    .apply(lambda g: (g.drop(columns=["WELL_TYPE", "FLOW_KIND"], errors="ignore").isna().mean() * 100).round(2))
)
temperature_missingness_by_state


AVG_DOWNHOLE_TEMPERATURE  AVG_WHT_P
WELL_TYPE FLOW_KIND                                      
OP        production                      1.80       0.15
WI        injection                     100.00     100.00
          production                     88.89       5.56

### 16.3 Missingness by activity state


In [190]:
activity_masks = {
    "active_production": (daily_df["BORE_OIL_VOL"] > 0) | (daily_df["BORE_GAS_VOL"] > 0) | (daily_df["BORE_WAT_VOL"] > 0),
    "active_injection": daily_df["BORE_WI_VOL"] > 0,
    "zero_hours": hours == 0,
    "null_hours": hours.isna(),
}

activity_rows = []
for label, mask in activity_masks.items():
    row = {"activity_state": label, "n_rows": int(mask.sum())}
    for col in TEMPERATURE_COLS:
        row[col] = round(daily_df.loc[mask, col].isna().mean() * 100, 2) if mask.sum() else float("nan")
    activity_rows.append(row)

temperature_by_activity = pd.DataFrame(activity_rows)
temperature_by_activity


,activity_state,n_rows,AVG_DOWNHOLE_TEMPERATURE,AVG_WHT_P
0,active_production,8011,1.80,0.04
1,active_injection,5404,100.00,100.00
2,zero_hours,1952,43.44,42.16
3,null_hours,285,100.00,100.00


### 16.4 Downhole vs. wellhead temperature (REVIEW, not FAIL)

Compared only where both exist. Not automatically failed where downhole < wellhead — flagged for REVIEW until units and measurement conventions are confirmed, exactly as Section 15 treated the analogous pressure relationship.


In [191]:
both_present_mask = daily_df["AVG_DOWNHOLE_TEMPERATURE"].notna() & daily_df["AVG_WHT_P"].notna()
downhole_lt_wellhead_temp_mask = both_present_mask & (daily_df["AVG_DOWNHOLE_TEMPERATURE"] < daily_df["AVG_WHT_P"])

print(f"Rows with both temperature fields present: {int(both_present_mask.sum())}")
print(f"AVG_DOWNHOLE_TEMPERATURE < AVG_WHT_P (REVIEW):                {int(downhole_lt_wellhead_temp_mask.sum())}")


Rows with both temperature fields present: 8980
AVG_DOWNHOLE_TEMPERATURE < AVG_WHT_P (REVIEW):                2117


In [192]:
temp_relationship_cols = [
    "DATEPRD_parsed", "NPD_WELL_BORE_CODE", "NPD_WELL_BORE_NAME", "WELL_TYPE", "FLOW_KIND",
    "ON_STREAM_HRS", "AVG_DOWNHOLE_TEMPERATURE", "AVG_WHT_P",
]
downhole_lt_wellhead_temp_rows = (
    daily_with_dates.loc[downhole_lt_wellhead_temp_mask, temp_relationship_cols].sort_values("DATEPRD_parsed")
)
downhole_lt_wellhead_temp_rows.head(20)


,DATEPRD_parsed,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,WELL_TYPE,FLOW_KIND,ON_STREAM_HRS,AVG_DOWNHOLE_TEMPERATURE,AVG_WHT_P
4975,2008-02-20,5351,15/9-F-14,OP,production,0.0,0.000000,0.000020
4977,2008-02-22,5351,15/9-F-14,OP,production,0.0,0.000000,0.000150
4985,2008-03-01,5351,15/9-F-14,OP,production,0.0,0.000000,0.000020
4986,2008-03-02,5351,15/9-F-14,OP,production,0.0,0.000000,0.000130
4987,2008-03-03,5351,15/9-F-14,OP,production,0.0,0.000000,0.003010
4988,2008-03-04,5351,15/9-F-14,OP,production,0.0,0.000000,0.006040
4998,2008-03-14,5351,15/9-F-14,OP,production,0.0,0.000000,0.000030
5001,2008-03-17,5351,15/9-F-14,OP,production,0.0,0.000000,0.001010
5002,2008-03-18,5351,15/9-F-14,OP,production,0.0,0.000000,0.000830
5003,2008-03-19,5351,15/9-F-14,OP,production,0.0,0.000000,0.000450


### 16.5 Abrupt temperature changes

Absolute and percentage change together, with the effect of an absolute floor demonstrated rather than assumed.


In [193]:
temperature_change_rows = []
for col in TEMPERATURE_COLS:
    for code_value, group in daily_with_dates.groupby("NPD_WELL_BORE_CODE"):
        g = group.dropna(subset=["DATEPRD_parsed", col]).sort_values("DATEPRD_parsed")
        prev_val = g[col].shift(1)
        prev_date = g["DATEPRD_parsed"].shift(1)
        abs_change = g[col] - prev_val
        pct_change = (abs_change / prev_val * 100).where(prev_val != 0)
        valid = prev_val.notna()
        for idx in g.index[valid]:
            temperature_change_rows.append(
                {
                    "column": col,
                    "npd_code": code_value,
                    "wellbore_name": g.loc[idx, "NPD_WELL_BORE_NAME"],
                    "previous_date": prev_date.loc[idx].date(),
                    "current_date": g.loc[idx, "DATEPRD_parsed"].date(),
                    "previous_value": prev_val.loc[idx],
                    "current_value": g.loc[idx, col],
                    "abs_change": abs_change.loc[idx],
                    "pct_change": pct_change.loc[idx],
                    "ON_STREAM_HRS": g.loc[idx, "ON_STREAM_HRS"],
                    "FLOW_KIND": g.loc[idx, "FLOW_KIND"],
                    "WELL_TYPE": g.loc[idx, "WELL_TYPE"],
                }
            )

temperature_changes = pd.DataFrame(temperature_change_rows)
print(f"Consecutive same-wellbore temperature-field comparisons: {len(temperature_changes)}")


Consecutive same-wellbore temperature-field comparisons: 18115


In [194]:
PCT_THRESHOLD = 200
ABS_FLOOR_CANDIDATES = [0, 5, 10, 25]

floor_demo_rows = []
for floor in ABS_FLOOR_CANDIDATES:
    flagged = temperature_changes[(temperature_changes["pct_change"].abs() > PCT_THRESHOLD) & (temperature_changes["abs_change"].abs() >= floor)]
    floor_demo_rows.append({"abs_change_floor": floor, "flags_at_pct>200%_AND_floor": len(flagged)})

pd.DataFrame(floor_demo_rows)


,abs_change_floor,flags_at_pct>200%_AND_floor
0,0,98
1,5,92
2,10,92
3,25,87


In [195]:
TEMPERATURE_ABS_FLOOR = 10  # screening choice for the review table below only - not a final automated threshold (Section 25)
large_temperature_changes = temperature_changes[
    (temperature_changes["pct_change"].abs() > PCT_THRESHOLD) & (temperature_changes["abs_change"].abs() >= TEMPERATURE_ABS_FLOOR)
]
large_temperature_changes = large_temperature_changes.reindex(
    large_temperature_changes["abs_change"].abs().sort_values(ascending=False).index
).reset_index(drop=True)
print(f"Flagged for REVIEW (pct_change > {PCT_THRESHOLD}% AND abs_change >= {TEMPERATURE_ABS_FLOOR}): {len(large_temperature_changes)}")
large_temperature_changes.head(20)


Flagged for REVIEW (pct_change > 200% AND abs_change >= 10): 92


,column,npd_code,wellbore_name,previous_date,current_date,previous_value,current_value,abs_change,pct_change,ON_STREAM_HRS,FLOW_KIND,WELL_TYPE
0,AVG_DOWNHOLE_TEMPERATURE,5351,15/9-F-14,2010-02-25,2010-02-26,20.959721,106.219102,85.259380,406.777266,20.35833,production,OP
1,AVG_WHT_P,5599,15/9-F-12,2013-03-24,2013-03-25,1.913030,86.294360,84.381330,4410.873327,12.86000,production,OP
2,AVG_WHT_P,5599,15/9-F-12,2014-02-11,2014-02-12,7.516610,87.258864,79.742254,1060.880561,22.50000,production,OP
3,AVG_WHT_P,5599,15/9-F-12,2014-05-10,2014-05-11,10.742180,88.854917,78.112737,727.159075,15.18000,production,OP
4,AVG_WHT_P,5599,15/9-F-12,2011-07-24,2011-07-25,13.203310,90.336674,77.133364,584.197176,17.40000,production,OP
5,AVG_DOWNHOLE_TEMPERATURE,5351,15/9-F-14,2010-02-23,2010-02-24,30.076100,106.348972,76.272872,253.599608,24.00000,production,OP
6,AVG_WHT_P,5351,15/9-F-14,2013-03-21,2013-03-22,0.991590,77.203253,76.211663,7685.803885,17.97500,production,OP
7,AVG_WHT_P,5599,15/9-F-12,2011-12-29,2011-12-30,13.318210,89.222580,75.904370,569.929219,24.00000,production,OP
8,AVG_WHT_P,5599,15/9-F-12,2013-06-23,2013-06-26,13.936320,89.109026,75.172706,539.401405,12.66667,production,OP
9,AVG_WHT_P,5599,15/9-F-12,2011-07-18,2011-07-20,15.059450,89.919640,74.860190,497.097768,21.00000,production,OP


### 16.6 Temperature behavior around known issue populations

Testing, not forcing a connection: `DQ-002`, `DQ-004`, `DQ-005`, `DQ-006`.


In [196]:
dq_report_cols = ["DATEPRD_parsed", "NPD_WELL_BORE_CODE", "NPD_WELL_BORE_NAME"] + TEMPERATURE_COLS

dq002_mask = daily_with_dates["NPD_WELL_BORE_CODE"].isin([5351, 5599]) & daily_with_dates["DATEPRD_parsed"].isin(
    [pd.Timestamp("2012-01-02"), pd.Timestamp("2012-01-14")]
)
dq004_mask = hours > 24
dq005_mask = (daily_df["BORE_WAT_VOL"] < 0) | (
    (daily_df["NPD_WELL_BORE_CODE"] == 7078) & (dateprd_parsed == pd.Timestamp("2015-01-17"))
)
dq006_mask = daily_df["NPD_WELL_BORE_CODE"].isin([5693, 5769]) & (hours == 0) & (daily_df["BORE_WI_VOL"] > 100)

print("DQ-002 - temperature fields immediately before/after the shared 12-day gap:")
dq002_rows = daily_with_dates.loc[dq002_mask, dq_report_cols].sort_values(["NPD_WELL_BORE_CODE", "DATEPRD_parsed"])
dq002_rows


DQ-002 - temperature fields immediately before/after the shared 12-day gap:


,DATEPRD_parsed,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,AVG_DOWNHOLE_TEMPERATURE,AVG_WHT_P
6346,2012-01-02,5351,15/9-F-14,102.771801,86.007620
6347,2012-01-14,5351,15/9-F-14,100.328454,24.467062
3290,2012-01-02,5599,15/9-F-12,0.000000,88.495962
3291,2012-01-14,5599,15/9-F-12,0.000000,46.064784


In [197]:
print(f"DQ-004 - temperature fields on the {int(dq004_mask.sum())} DST-pattern rows:")
dq004_rows = daily_with_dates.loc[dq004_mask, dq_report_cols].sort_values(["NPD_WELL_BORE_CODE", "DATEPRD_parsed"])
dq004_rows


DQ-004 - temperature fields on the 20 DST-pattern rows:


,DATEPRD_parsed,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,AVG_DOWNHOLE_TEMPERATURE,AVG_WHT_P
5222,2008-10-26,5351,15/9-F-14,105.109841,78.139393
5583,2009-10-25,5351,15/9-F-14,105.788536,81.802541
5947,2010-10-31,5351,15/9-F-14,106.136905,90.931452
6968,2013-10-27,5351,15/9-F-14,100.271465,87.112258
7330,2014-10-26,5351,15/9-F-14,99.645467,85.646127
2166,2008-10-26,5599,15/9-F-12,106.112080,75.273871
2527,2009-10-25,5599,15/9-F-12,106.782569,75.845229
2891,2010-10-31,5599,15/9-F-12,0.000000,88.527274
3912,2013-10-27,5599,15/9-F-12,0.000000,89.321837
4274,2014-10-26,5599,15/9-F-12,0.000000,87.203167


In [198]:
print(f"DQ-005 - temperature fields on the {int(dq005_mask.sum())} flagged rows:")
dq005_rows = daily_with_dates.loc[dq005_mask, dq_report_cols].sort_values(["NPD_WELL_BORE_CODE", "DATEPRD_parsed"])
dq005_rows


DQ-005 - temperature fields on the 5 flagged rows:


,DATEPRD_parsed,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,AVG_DOWNHOLE_TEMPERATURE,AVG_WHT_P
5350,2009-03-03,5351,15/9-F-14,105.463574,79.554519
6558,2012-08-13,5351,15/9-F-14,101.886570,83.714672
1982,2008-04-23,5599,15/9-F-12,105.771069,73.584250
3502,2012-08-13,5599,15/9-F-12,0.000000,89.012162
1301,2015-01-17,7078,15/9-F-11,106.200000,64.800000


In [199]:
print(f"DQ-006 - temperature fields on the {int(dq006_mask.sum())} flagged rows:")
dq006_rows = daily_with_dates.loc[dq006_mask, dq_report_cols].sort_values(["NPD_WELL_BORE_CODE", "DATEPRD_parsed"])
dq006_rows


DQ-006 - temperature fields on the 2 flagged rows:


,DATEPRD_parsed,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,AVG_DOWNHOLE_TEMPERATURE,AVG_WHT_P
10387,2011-06-18,5693,15/9-F-4,NaN,NaN
14351,2013-03-16,5769,15/9-F-5,NaN,NaN


### Temperature-quality verdict


In [200]:
n_downhole_lt_wellhead_temp = int(downhole_lt_wellhead_temp_mask.sum())
n_large_temperature_changes = len(large_temperature_changes)

# FAIL is reserved for what is structurally impossible regardless of units or
# convention - nothing here meets that bar without confirmed units.
any_review_findings = any([n_downhole_lt_wellhead_temp, n_large_temperature_changes])
temperature_verdict = "PASS WITH REVIEW" if any_review_findings else "PASS"

print("=" * 60)
print("SECTION 16 - TEMPERATURE VALIDATION SUMMARY")
print("=" * 60)
print(f"AVG_DOWNHOLE_TEMPERATURE < AVG_WHT_P:                       {n_downhole_lt_wellhead_temp}")
print(f"Large abrupt changes (pct>200% AND abs>={TEMPERATURE_ABS_FLOOR}):                {n_large_temperature_changes}")
print()
print(f"VERDICT: {temperature_verdict}")


SECTION 16 - TEMPERATURE VALIDATION SUMMARY
AVG_DOWNHOLE_TEMPERATURE < AVG_WHT_P:                       2117
Large abrupt changes (pct>200% AND abs>=10):                92

VERDICT: PASS WITH REVIEW


**Interpretation for later relational modelling:**

**Temperature reproduces pressure's pattern almost exactly, at the field level.** 16.2/16.3 show the same signature as Section 15: `WI`/injection rows ~100% missing on both temperature fields, the 18 `WI`/production transition rows show the same lag (88.89% missing on `AVG_DOWNHOLE_TEMPERATURE` vs. only 5.56% on `AVG_WHT_P` — wellhead temperature comes back online with production, downhole doesn't), and the 285 NULL-hours rows are 100% missing on both. Nothing here changes Section 15's account of *when* a sensor measurement should reasonably exist — it corroborates it with an independent field.

**16.4 — `AVG_DOWNHOLE_TEMPERATURE < AVG_WHT_P` in 2,117 of 8,980 rows (~23.6%), kept as REVIEW.** This is close in both count and proportion to Section 15's `AVG_DOWNHOLE_PRESSURE < AVG_WHP_P` finding (2,126 rows) — worth noting as a pattern rather than two unrelated findings, though this notebook does not test here whether they are literally the same rows. As with pressure, "downhole should exceed wellhead" is a plausible convention, not a confirmed one, so this stays REVIEW pending units/documentation.

**16.5 — the absolute floor matters only modestly for temperature** (98 → 87 flags as the floor rises from 0 to 25, a ~11% reduction), closer to injection's behavior (Section 14, where the floor barely mattered) than to production's or pressure's (where it mattered a great deal). A third data point confirming the floor's effect is column-specific, not universal.

**16.6 corroborates rather than contradicts the DQ populations, with one exact echo.** The DQ-002 boundary check shows wellbore 5599 at `AVG_DOWNHOLE_TEMPERATURE = 0.0` on **both** 2012-01-02 and 2012-01-14 — the same pattern Section 15 found for pressure on the same two dates. The DQ-005 row for wellbore 5599 on 2012-08-13 (already flagged for negative water, a 0.625-hour day, and `AVG_DOWNHOLE_PRESSURE = 0.0`) now also shows `AVG_DOWNHOLE_TEMPERATURE = 0.0` — the same row, the same "zeroed downhole readings" signature, in a second independent field. And **`DQ-006`'s two rows show both temperature fields as NaN**, extending the chain exactly as anticipated:

```
DQ-006 → zero hours → positive injection → no pressure measurements → no temperature measurements
```

```
VERDICT: PASS WITH REVIEW
```

No `FAIL` — nothing structurally impossible without confirmed units. For relational modelling: both temperature fields load as nullable numeric with no assumed physical bound. The DQ-006 chain is now strong enough that Section 17 (choke) should complete it rather than treat DQ-006 as settled.


## 17. Choke validation

Fields: `AVG_CHOKE_SIZE_P`, `AVG_CHOKE_UOM`, `DP_CHOKE_SIZE`. Section 12.8 already confirmed `AVG_CHOKE_UOM` is always `"%"` wherever present — this section tests the *values*, not the unit label again.

Any result above 100% is kept as **REVIEW**, not FAIL — `AVG_CHOKE_SIZE_P` is assumed to be a percentage-like opening measure, but nothing in this notebook has confirmed it is a strict physical opening percentage bounded at 100. That assumption needs source documentation, not just column naming.

This section also explicitly tests the DQ-006 pair for choke behavior, extending the record-level contradiction chain from Section 15/16: `DQ-006 → zero hours → positive injection → no pressure measurements → (choke?)`.


In [201]:
# --- TEMPORARY LOADING FOR SECTION 17 ---
# Section 1 (Load source data) has not been implemented yet in this notebook.
# This block loads only what Section 17 needs, and is clearly marked so it can
# be deleted once Section 1 provides `daily_df` for the whole notebook.
if "daily_df" not in globals():
    from pathlib import Path
    import pandas as pd

    PROJECT_ROOT = Path.cwd().parent
    WORKBOOK_PATH = PROJECT_ROOT / "data" / "raw" / "Volve production data.xlsx"
    DAILY_SHEET_NAME = "Daily Production Data"

    if not WORKBOOK_PATH.exists():
        raise FileNotFoundError(f"Source workbook not found at {WORKBOOK_PATH}")

    daily_df = pd.read_excel(WORKBOOK_PATH, sheet_name=DAILY_SHEET_NAME)
    print(f"[Section 17 temporary load] daily_df loaded: {daily_df.shape}")
else:
    print(f"Using daily_df already loaded earlier in the notebook: {daily_df.shape}")


Using daily_df already loaded earlier in the notebook: (15634, 24)


In [202]:
# Reuse Section 9-16 derived state if available, recompute the minimum if not.
if "dateprd_parsed" not in globals():
    dateprd_parsed = pd.to_datetime(daily_df["DATEPRD"], errors="coerce")
if "daily_with_dates" not in globals():
    daily_with_dates = daily_df.copy()
    daily_with_dates["DATEPRD_parsed"] = dateprd_parsed
if "hours" not in globals():
    hours = daily_df["ON_STREAM_HRS"]

required_columns = [
    "AVG_CHOKE_SIZE_P", "AVG_CHOKE_UOM", "DP_CHOKE_SIZE", "WELL_TYPE", "FLOW_KIND", "ON_STREAM_HRS",
    "NPD_WELL_BORE_CODE", "NPD_WELL_BORE_NAME", "DATEPRD", "BORE_OIL_VOL", "BORE_GAS_VOL", "BORE_WAT_VOL", "BORE_WI_VOL",
]
missing_columns = [c for c in required_columns if c not in daily_df.columns]
if missing_columns:
    raise ValueError(f"Required column(s) missing from daily_df: {missing_columns}")
print(f"Required columns present: {required_columns}")


Required columns present: ['AVG_CHOKE_SIZE_P', 'AVG_CHOKE_UOM', 'DP_CHOKE_SIZE', 'WELL_TYPE', 'FLOW_KIND', 'ON_STREAM_HRS', 'NPD_WELL_BORE_CODE', 'NPD_WELL_BORE_NAME', 'DATEPRD', 'BORE_OIL_VOL', 'BORE_GAS_VOL', 'BORE_WAT_VOL', 'BORE_WI_VOL']


### 17.1 `AVG_CHOKE_SIZE_P` value ranges

Below 0, exactly 0, between 0 and 100, above 100, NULL. Above-100 results are REVIEW, not FAIL, per this section's objective.


In [203]:
choke = daily_df["AVG_CHOKE_SIZE_P"]

choke_range_summary = pd.DataFrame(
    [
        {
            "below_0": int((choke < 0).sum()),
            "exactly_0": int((choke == 0).sum()),
            "between_0_and_100": int(((choke > 0) & (choke < 100)).sum()),
            "exactly_100": int((choke == 100).sum()),
            "above_100": int((choke > 100).sum()),
            "null": int(choke.isna().sum()),
            "total": len(choke),
        }
    ]
)
choke_range_summary


,below_0,exactly_0,between_0_and_100,exactly_100,above_100,null,total
0,0,227,6569,2123,0,6715,15634


In [204]:
above_100_rows = daily_with_dates.loc[
    choke > 100,
    ["DATEPRD_parsed", "NPD_WELL_BORE_CODE", "NPD_WELL_BORE_NAME", "WELL_TYPE", "FLOW_KIND", "ON_STREAM_HRS", "AVG_CHOKE_SIZE_P"],
].sort_values("AVG_CHOKE_SIZE_P", ascending=False)
print(f"AVG_CHOKE_SIZE_P > 100 (REVIEW, not FAIL - percentage definition not yet confirmed): {len(above_100_rows)}")
above_100_rows.head(20)


AVG_CHOKE_SIZE_P > 100 (REVIEW, not FAIL - percentage definition not yet confirmed): 0


,DATEPRD_parsed,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,WELL_TYPE,FLOW_KIND,ON_STREAM_HRS,AVG_CHOKE_SIZE_P


### 17.2 Choke vs. production/injection activity

Positive production with choke = 0, and positive injection with choke = 0.


In [205]:
positive_production_mask = (daily_df["BORE_OIL_VOL"] > 0) | (daily_df["BORE_GAS_VOL"] > 0) | (daily_df["BORE_WAT_VOL"] > 0)
positive_injection_mask = daily_df["BORE_WI_VOL"] > 0

zero_choke_positive_production_mask = (choke == 0) & positive_production_mask
zero_choke_positive_injection_mask = (choke == 0) & positive_injection_mask

print(f"Positive production with AVG_CHOKE_SIZE_P = 0 (REVIEW): {int(zero_choke_positive_production_mask.sum())}")
print(f"Positive injection with AVG_CHOKE_SIZE_P = 0 (REVIEW):  {int(zero_choke_positive_injection_mask.sum())}")


Positive production with AVG_CHOKE_SIZE_P = 0 (REVIEW): 0
Positive injection with AVG_CHOKE_SIZE_P = 0 (REVIEW):  0


In [206]:
choke_report_cols = [
    "DATEPRD_parsed", "NPD_WELL_BORE_CODE", "NPD_WELL_BORE_NAME", "WELL_TYPE", "FLOW_KIND",
    "ON_STREAM_HRS", "AVG_CHOKE_SIZE_P", "BORE_OIL_VOL", "BORE_GAS_VOL", "BORE_WAT_VOL", "BORE_WI_VOL",
]
zero_choke_positive_production_rows = daily_with_dates.loc[zero_choke_positive_production_mask, choke_report_cols].sort_values("DATEPRD_parsed")
zero_choke_positive_production_rows.head(20)


,DATEPRD_parsed,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,WELL_TYPE,FLOW_KIND,ON_STREAM_HRS,AVG_CHOKE_SIZE_P,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL


### 17.3 Choke vs. operating hours

`AVG_CHOKE_SIZE_P > 0` with `ON_STREAM_HRS = 0`, and with `ON_STREAM_HRS` NULL.


In [207]:
positive_choke_zero_hours_mask = (choke > 0) & (hours == 0)
positive_choke_null_hours_mask = (choke > 0) & hours.isna()

print(f"AVG_CHOKE_SIZE_P > 0 with ON_STREAM_HRS = 0 (REVIEW):    {int(positive_choke_zero_hours_mask.sum())}")
print(f"AVG_CHOKE_SIZE_P > 0 with ON_STREAM_HRS NULL (REVIEW):   {int(positive_choke_null_hours_mask.sum())}")


AVG_CHOKE_SIZE_P > 0 with ON_STREAM_HRS = 0 (REVIEW):    672
AVG_CHOKE_SIZE_P > 0 with ON_STREAM_HRS NULL (REVIEW):   0


### 17.4 Abrupt choke changes by wellbore

Absolute and percentage change together, same convention as Sections 13-16.


In [208]:
choke_change_rows = []
for code_value, group in daily_with_dates.groupby("NPD_WELL_BORE_CODE"):
    g = group.dropna(subset=["DATEPRD_parsed", "AVG_CHOKE_SIZE_P"]).sort_values("DATEPRD_parsed")
    prev_val = g["AVG_CHOKE_SIZE_P"].shift(1)
    prev_date = g["DATEPRD_parsed"].shift(1)
    abs_change = g["AVG_CHOKE_SIZE_P"] - prev_val
    pct_change = (abs_change / prev_val * 100).where(prev_val != 0)
    valid = prev_val.notna()
    for idx in g.index[valid]:
        choke_change_rows.append(
            {
                "npd_code": code_value,
                "wellbore_name": g.loc[idx, "NPD_WELL_BORE_NAME"],
                "previous_date": prev_date.loc[idx].date(),
                "current_date": g.loc[idx, "DATEPRD_parsed"].date(),
                "previous_value": prev_val.loc[idx],
                "current_value": g.loc[idx, "AVG_CHOKE_SIZE_P"],
                "abs_change": abs_change.loc[idx],
                "pct_change": pct_change.loc[idx],
            }
        )

choke_changes = pd.DataFrame(choke_change_rows)
print(f"Consecutive same-wellbore AVG_CHOKE_SIZE_P comparisons: {len(choke_changes)}")


Consecutive same-wellbore AVG_CHOKE_SIZE_P comparisons: 8913


In [209]:
PCT_THRESHOLD = 200
ABS_FLOOR_CANDIDATES = [0, 5, 10, 25]

floor_demo_rows = []
for floor in ABS_FLOOR_CANDIDATES:
    flagged = choke_changes[(choke_changes["pct_change"].abs() > PCT_THRESHOLD) & (choke_changes["abs_change"].abs() >= floor)]
    floor_demo_rows.append({"abs_change_floor": floor, "flags_at_pct>200%_AND_floor": len(flagged)})

pd.DataFrame(floor_demo_rows)


,abs_change_floor,flags_at_pct>200%_AND_floor
0,0,142
1,5,131
2,10,108
3,25,77


In [210]:
CHOKE_ABS_FLOOR = 10
large_choke_changes = choke_changes[
    (choke_changes["pct_change"].abs() > PCT_THRESHOLD) & (choke_changes["abs_change"].abs() >= CHOKE_ABS_FLOOR)
]
large_choke_changes = large_choke_changes.reindex(
    large_choke_changes["abs_change"].abs().sort_values(ascending=False).index
).reset_index(drop=True)
print(f"Flagged for REVIEW (pct_change > {PCT_THRESHOLD}% AND abs_change >= {CHOKE_ABS_FLOOR}): {len(large_choke_changes)}")
large_choke_changes.head(20)


Flagged for REVIEW (pct_change > 200% AND abs_change >= 10): 108


,npd_code,wellbore_name,previous_date,current_date,previous_value,current_value,abs_change,pct_change
0,5599,15/9-F-12,2014-01-29,2014-01-31,5.334822,100.000000,94.665178,1774.476684
1,5599,15/9-F-12,2013-07-29,2013-07-30,6.775480,99.254298,92.478818,1364.904256
2,5599,15/9-F-12,2013-12-11,2013-12-12,3.587429,95.403409,91.815981,2559.381450
3,5351,15/9-F-14,2013-11-03,2013-11-04,9.051365,100.000000,90.948635,1004.805714
4,5599,15/9-F-12,2015-08-30,2015-08-31,0.598615,89.857433,89.258818,14910.894214
5,5599,15/9-F-12,2014-04-07,2014-04-08,10.994698,99.901605,88.906907,808.634393
6,7078,15/9-F-11,2016-04-18,2016-04-19,14.241494,100.000000,85.758506,602.173503
7,5351,15/9-F-14,2012-09-29,2012-09-30,15.688110,99.989307,84.301198,537.357271
8,5599,15/9-F-12,2014-02-11,2014-02-12,0.622567,84.606057,83.983490,13489.874481
9,5599,15/9-F-12,2014-03-25,2014-03-26,16.338922,99.622297,83.283376,509.723820


### 17.5 Does `DP_CHOKE_SIZE` behave consistently with `AVG_CHOKE_SIZE_P`?

Neither column's exact definition is confirmed, so this tests correlation/co-movement rather than assuming one is derived from the other.


In [211]:
both_choke_present_mask = daily_df["AVG_CHOKE_SIZE_P"].notna() & daily_df["DP_CHOKE_SIZE"].notna()
print(f"Rows with both choke fields present: {int(both_choke_present_mask.sum())}")

choke_correlation = daily_df.loc[both_choke_present_mask, ["AVG_CHOKE_SIZE_P", "DP_CHOKE_SIZE"]].corr()
choke_correlation


Rows with both choke fields present: 8915


,AVG_CHOKE_SIZE_P,DP_CHOKE_SIZE
AVG_CHOKE_SIZE_P,1.000000,-0.551193
DP_CHOKE_SIZE,-0.551193,1.000000


In [212]:
# Do the two fields agree on "choke open" vs "choke shut"? A simple structural
# cross-check: AVG_CHOKE_SIZE_P == 0 vs DP_CHOKE_SIZE == 0.
choke_zero_agreement = pd.crosstab(
    (daily_df.loc[both_choke_present_mask, "AVG_CHOKE_SIZE_P"] == 0).map({True: "size_p = 0", False: "size_p > 0"}),
    (daily_df.loc[both_choke_present_mask, "DP_CHOKE_SIZE"] == 0).map({True: "dp = 0", False: "dp > 0"}),
)
choke_zero_agreement


DP_CHOKE_SIZE,dp = 0,dp > 0
AVG_CHOKE_SIZE_P,,
size_p = 0,15,212
size_p > 0,39,8649


### 17.6 DQ-006 choke check

Extending the chain from Sections 15 and 16: `DQ-006 → zero hours → positive injection → no pressure measurements → (temperature?) → choke?`. Section 15 already found `DP_CHOKE_SIZE = 0.0` on both DQ-006 rows; this checks `AVG_CHOKE_SIZE_P` specifically.


In [213]:
dq006_mask = daily_df["NPD_WELL_BORE_CODE"].isin([5693, 5769]) & (hours == 0) & (daily_df["BORE_WI_VOL"] > 100)
dq006_choke_cols = [
    "DATEPRD_parsed", "NPD_WELL_BORE_CODE", "NPD_WELL_BORE_NAME", "WELL_TYPE", "FLOW_KIND",
    "ON_STREAM_HRS", "BORE_WI_VOL", "AVG_CHOKE_SIZE_P", "AVG_CHOKE_UOM", "DP_CHOKE_SIZE",
]
dq006_rows = daily_with_dates.loc[dq006_mask, dq006_choke_cols].sort_values("DATEPRD_parsed")
print(f"DQ-006 - choke fields on the {len(dq006_rows)} flagged rows:")
dq006_rows


DQ-006 - choke fields on the 2 flagged rows:


,DATEPRD_parsed,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,WELL_TYPE,FLOW_KIND,ON_STREAM_HRS,BORE_WI_VOL,AVG_CHOKE_SIZE_P,AVG_CHOKE_UOM,DP_CHOKE_SIZE
10387,2011-06-18,5693,15/9-F-4,WI,injection,0.0,6263.692353,NaN,NaN,0.0
14351,2013-03-16,5769,15/9-F-5,WI,injection,0.0,148.655136,NaN,NaN,0.0


### Choke-quality verdict


In [214]:
n_above_100 = len(above_100_rows)
n_zero_choke_positive_production = int(zero_choke_positive_production_mask.sum())
n_zero_choke_positive_injection = int(zero_choke_positive_injection_mask.sum())
n_positive_choke_zero_hours = int(positive_choke_zero_hours_mask.sum())
n_positive_choke_null_hours = int(positive_choke_null_hours_mask.sum())
n_large_choke_changes = len(large_choke_changes)

# FAIL is reserved for what is structurally impossible regardless of units or
# convention - the >100% results are REVIEW because the percentage definition
# itself is unconfirmed, not because the values are assumed wrong.
any_review_findings = any(
    [
        n_above_100,
        n_zero_choke_positive_production,
        n_zero_choke_positive_injection,
        n_positive_choke_zero_hours,
        n_positive_choke_null_hours,
        n_large_choke_changes,
    ]
)
choke_verdict = "PASS WITH REVIEW" if any_review_findings else "PASS"

print("=" * 60)
print("SECTION 17 - CHOKE VALIDATION SUMMARY")
print("=" * 60)
print(f"AVG_CHOKE_SIZE_P > 100 (REVIEW, unit/definition not confirmed): {n_above_100}")
print(f"Positive production with choke = 0:                             {n_zero_choke_positive_production}")
print(f"Positive injection with choke = 0:                              {n_zero_choke_positive_injection}")
print(f"Choke > 0 with ON_STREAM_HRS = 0:                               {n_positive_choke_zero_hours}")
print(f"Choke > 0 with ON_STREAM_HRS NULL:                              {n_positive_choke_null_hours}")
print(f"Large abrupt choke changes (pct>200% AND abs>={CHOKE_ABS_FLOOR}):                {n_large_choke_changes}")
print()
print(f"VERDICT: {choke_verdict}")


SECTION 17 - CHOKE VALIDATION SUMMARY
AVG_CHOKE_SIZE_P > 100 (REVIEW, unit/definition not confirmed): 0
Positive production with choke = 0:                             0
Positive injection with choke = 0:                              0
Choke > 0 with ON_STREAM_HRS = 0:                               672
Choke > 0 with ON_STREAM_HRS NULL:                              0
Large abrupt choke changes (pct>200% AND abs>=10):                108

VERDICT: PASS WITH REVIEW


**Interpretation for later relational modelling:**

**`AVG_CHOKE_SIZE_P` never exceeds 100 in this dataset** (0 rows above 100, out of 8,919 populated values; 2,123 sit at exactly 100). The REVIEW-not-FAIL test for values >100 was correctly available but found nothing to flag — a reassuring, not a wasted, check. **17.2 is fully clean**: 0 rows of positive production with choke=0, 0 rows of positive injection with choke=0 — choke tracks production activity precisely where it is populated at all.

**17.3 is this section's main finding: `AVG_CHOKE_SIZE_P > 0` with `ON_STREAM_HRS = 0` in 672 rows** — an order of magnitude larger than any equivalent zero-hours-vs-measurement check in Sections 13-16 (which ranged from 0 to 91). This is not treated as 672 errors; a plausible explanation is that `AVG_CHOKE_SIZE_P` records the choke's physical position/setting, which can remain open even on a day logged with zero on-stream hours (e.g. between flow periods, or a valve left at a set position) — a genuinely different physical quantity from "hours flowing." That is a hypothesis this notebook is flagging, not concluding; it is the largest REVIEW population in the operating-measurement block (Sections 15-17) and deserves attention in Section 23.

**17.5 shows a physically coherent relationship, not a redundancy.** `AVG_CHOKE_SIZE_P` and `DP_CHOKE_SIZE` correlate at **-0.55** — negative, and that direction makes physical sense for a choke: a larger opening (higher `%`) typically produces a smaller differential pressure across it, and vice versa. They are not measuring the same thing, but they move together in the direction real choke behavior would predict. The zero/non-zero agreement crosstab shows 97.2% consistency (8,664 of 8,915 rows agree on "shut" vs. "open"); the remaining 251 rows (212 where `AVG_CHOKE_SIZE_P = 0` but `DP_CHOKE_SIZE > 0`, 39 the other way) are a small, specific disagreement population worth carrying forward rather than a systemic problem.

**17.6 completes the DQ-006 contradiction chain, with one precision correction to how Section 15 described it.** Section 15 found `DP_CHOKE_SIZE = 0.0` on both DQ-006 rows and that finding holds. But `AVG_CHOKE_SIZE_P` and `AVG_CHOKE_UOM` are both **NaN**, not `0` — a different fact (`DP_CHOKE_SIZE = 0` is a recorded zero differential pressure; `AVG_CHOKE_SIZE_P = NULL` is no recorded choke-opening percentage at all), and the distinction matters given how carefully this notebook has kept NULL and 0 apart everywhere else. With that correction, the full chain for both DQ-006 rows (wellbore 5693 on 2011-06-18, wellbore 5769 on 2013-03-16) is now:

```
DQ-006 → ON_STREAM_HRS = 0
       → BORE_WI_VOL > 100 m3 (substantial recorded injection)
       → AVG_DOWNHOLE_PRESSURE / AVG_DP_TUBING / AVG_ANNULUS_PRESS / AVG_WHP_P all NULL
       → AVG_DOWNHOLE_TEMPERATURE / AVG_WHT_P both NULL
       → AVG_CHOKE_SIZE_P / AVG_CHOKE_UOM both NULL, DP_CHOKE_SIZE = 0
```

Every sensor on these two rows is either missing or reads as a zero/shut-in value, while `BORE_WI_VOL` alone reports substantial injected volume. That is now a fully characterized, coherent record-level contradiction — not resolved (this notebook still does not decide whether the volume, the state fields, or the sensors are "wrong"), but no longer a vague "2 odd rows." This is exactly the kind of finding Section 18 should synthesize into a physically interpretable record state rather than re-deriving.

```
VERDICT: PASS WITH REVIEW
```

No `FAIL` here either. **Operating-measurement block complete (Sections 15-17)**: pressure, temperature, and choke all show the same operating-state-driven missingness structure, the same "downhole/gauge lags wellhead/surface" transition signature, the same floor-sensitivity variation worth remembering for Section 25, and — via DQ-006 — the same two rows behaving as a coherent, fully-blank sensor record despite a real recorded volume. Handing off to **Section 18 (cross-variable consistency)**, whose job is to combine this block's evidence (plus Sections 10, 13, 14) into a small set of physically interpretable record states — e.g. "active + positive production + no sensors," "inactive + positive injection + no sensors," "transition state + production-like behavior" — rather than repeating any of the checks already done here.


## 18. Cross-variable consistency

This is **synthesis, not another battery of tests**. Sections 9-17 already tested every field individually; this section combines that evidence into a small number of diagnostic **record states** and answers one question: *how many records are internally coherent given everything learned so far?*

Seven states, in this precedence order (first match wins):

1. **TRANSITION_STATE** — one of the 18 already-identified `WELL_TYPE=WI`/`FLOW_KIND=production` rows (Section 12).
2. **BLANK_STATE_RECORD** — `ON_STREAM_HRS` is NULL (the DQ-001/DQ-003 population).
3. **INCONSISTENT** — positive production or injection volume recorded against `ON_STREAM_HRS = 0` — evidence conflicts materially.
4. **INACTIVE_RECORDED** — `ON_STREAM_HRS = 0` with no material production/injection — a normal shut-in day.
5. **ACTIVE_PRODUCTION** — positive production volume with operating evidence (`ON_STREAM_HRS > 0`).
6. **ACTIVE_INJECTION** — positive injection volume with operating evidence (`ON_STREAM_HRS > 0`).
7. **REVIEW** — `ON_STREAM_HRS > 0` but no material production or injection recorded — doesn't fit confidently anywhere else.

These are **diagnostic classifications only** — nothing is written back to `daily_df`, nothing here becomes a permanent business label. `DQ-007` is **not** created by default just because rows land in INCONSISTENT or REVIEW; it is only introduced if this section's breakdown surfaces a genuinely new pattern not already covered by DQ-001 through DQ-006.


In [215]:
# --- TEMPORARY LOADING FOR SECTION 18 ---
# Section 1 (Load source data) has not been implemented yet in this notebook.
# This block loads only what Section 18 needs, and is clearly marked so it can
# be deleted once Section 1 provides `daily_df` for the whole notebook.
if "daily_df" not in globals():
    from pathlib import Path
    import pandas as pd

    PROJECT_ROOT = Path.cwd().parent
    WORKBOOK_PATH = PROJECT_ROOT / "data" / "raw" / "Volve production data.xlsx"
    DAILY_SHEET_NAME = "Daily Production Data"

    if not WORKBOOK_PATH.exists():
        raise FileNotFoundError(f"Source workbook not found at {WORKBOOK_PATH}")

    daily_df = pd.read_excel(WORKBOOK_PATH, sheet_name=DAILY_SHEET_NAME)
    print(f"[Section 18 temporary load] daily_df loaded: {daily_df.shape}")
else:
    print(f"Using daily_df already loaded earlier in the notebook: {daily_df.shape}")

if "hours" not in globals():
    hours = daily_df["ON_STREAM_HRS"]


Using daily_df already loaded earlier in the notebook: (15634, 24)


### Record-state classification


In [216]:
has_positive_production = (daily_df["BORE_OIL_VOL"] > 0) | (daily_df["BORE_GAS_VOL"] > 0) | (daily_df["BORE_WAT_VOL"] > 0)
has_positive_injection = daily_df["BORE_WI_VOL"] > 0
is_transition_row = (daily_df["WELL_TYPE"] == "WI") & (daily_df["FLOW_KIND"] == "production")

record_state = pd.Series(index=daily_df.index, dtype="object")

record_state[is_transition_row] = "TRANSITION_STATE"

remaining = record_state.isna()
record_state[remaining & hours.isna()] = "BLANK_STATE_RECORD"

remaining = record_state.isna()
inconsistent_mask = remaining & (hours == 0) & (has_positive_production | has_positive_injection)
record_state[inconsistent_mask] = "INCONSISTENT"

remaining = record_state.isna()
record_state[remaining & (hours == 0)] = "INACTIVE_RECORDED"

remaining = record_state.isna()
record_state[remaining & (hours > 0) & has_positive_production] = "ACTIVE_PRODUCTION"

remaining = record_state.isna()
record_state[remaining & (hours > 0) & has_positive_injection] = "ACTIVE_INJECTION"

remaining = record_state.isna()
record_state[remaining & (hours > 0)] = "REVIEW"

print(f"Unclassified rows remaining (should be 0): {int(record_state.isna().sum())}")


Unclassified rows remaining (should be 0): 0


In [217]:
record_state_summary = (
    record_state.value_counts(dropna=False)
    .rename_axis("record_state")
    .reset_index(name="count")
)
record_state_summary["pct"] = (record_state_summary["count"] / len(daily_df) * 100).round(2)
record_state_summary


,record_state,count,pct
0,ACTIVE_PRODUCTION,7999,51.16
1,ACTIVE_INJECTION,5373,34.37
2,INACTIVE_RECORDED,1909,12.21
3,BLANK_STATE_RECORD,285,1.82
4,INCONSISTENT,34,0.22
5,TRANSITION_STATE,18,0.12
6,REVIEW,16,0.10


In [218]:
coherent_states = {"TRANSITION_STATE", "BLANK_STATE_RECORD", "INACTIVE_RECORDED", "ACTIVE_PRODUCTION", "ACTIVE_INJECTION"}
n_coherent = int(record_state.isin(coherent_states).sum())
n_total = len(daily_df)

print(f"Internally coherent records (fit a clean state, given Sections 9-17): {n_coherent} / {n_total} ({n_coherent / n_total * 100:.2f}%)")
print(f"Records requiring further interpretation (INCONSISTENT + REVIEW):     {n_total - n_coherent} / {n_total} ({(n_total - n_coherent) / n_total * 100:.2f}%)")


Internally coherent records (fit a clean state, given Sections 9-17): 15584 / 15634 (99.68%)
Records requiring further interpretation (INCONSISTENT + REVIEW):     50 / 15634 (0.32%)


### Which known DQ issues account for the INCONSISTENT population?


In [219]:
dq005_mask = (daily_df["BORE_WAT_VOL"] < 0) | (
    (daily_df["NPD_WELL_BORE_CODE"] == 7078) & (pd.to_datetime(daily_df["DATEPRD"]) == pd.Timestamp("2015-01-17"))
)
dq006_mask = daily_df["NPD_WELL_BORE_CODE"].isin([5693, 5769]) & (hours == 0) & (daily_df["BORE_WI_VOL"] > 100)

# "Covered" means investigated in an earlier section, whether or not it earned
# a formal DQ ID. Section 13.6 cross-referenced the two 2011-12-26 zero-hours-
# positive-production rows (5599, 5351) against next-day jumps and found a
# plausible shutdown/recovery explanation - investigated, not escalated.
# Section 14.4 characterized all 31 zero-hours-positive-injection rows by
# magnitude and only escalated the 2 large ones to DQ-006 - the other 29
# (trace/moderate) were investigated and left deliberately unescalated, not
# overlooked. Treating those as "uncovered" here would spuriously reopen a
# question Sections 13-14 already answered.
shutdown_recovery_mask = daily_df["NPD_WELL_BORE_CODE"].isin([5599, 5351]) & (pd.to_datetime(daily_df["DATEPRD"]) == pd.Timestamp("2011-12-26"))
trace_moderate_wi_mask = (hours == 0) & (daily_df["BORE_WI_VOL"] > 0) & (daily_df["BORE_WI_VOL"] <= 100)

inconsistent_rows = record_state[record_state == "INCONSISTENT"]
explained_by_dq005 = int((dq005_mask & (record_state == "INCONSISTENT")).sum())
explained_by_dq006 = int((dq006_mask & (record_state == "INCONSISTENT")).sum())
explained_by_shutdown_recovery = int((shutdown_recovery_mask & (record_state == "INCONSISTENT")).sum())
explained_by_trace_moderate_wi = int((trace_moderate_wi_mask & (record_state == "INCONSISTENT")).sum())
unexplained_inconsistent = int(
    len(inconsistent_rows)
    - explained_by_dq005
    - explained_by_dq006
    - explained_by_shutdown_recovery
    - explained_by_trace_moderate_wi
)

print(f"INCONSISTENT total:                                                    {len(inconsistent_rows)}")
print(f"  - explained by DQ-005 (negative water / unexplained zero-hours production): {explained_by_dq005}")
print(f"  - explained by DQ-006 (large zero-hours/positive-injection):                {explained_by_dq006}")
print(f"  - investigated in 13.6 (2011-12-26 shutdown/recovery, not escalated):       {explained_by_shutdown_recovery}")
print(f"  - investigated in 14.4 (trace/moderate zero-hours injection, not escalated): {explained_by_trace_moderate_wi}")
print(f"  - not covered by any prior investigation:                                  {unexplained_inconsistent}")


INCONSISTENT total:                                                    34
  - explained by DQ-005 (negative water / unexplained zero-hours production): 1
  - explained by DQ-006 (large zero-hours/positive-injection):                2
  - investigated in 13.6 (2011-12-26 shutdown/recovery, not escalated):       2
  - investigated in 14.4 (trace/moderate zero-hours injection, not escalated): 29
  - not covered by any prior investigation:                                  0


In [220]:
# Characterize whatever is genuinely not covered, rather than assuming it needs a new ID.
uncovered_mask = (
    (record_state == "INCONSISTENT")
    & (~dq005_mask)
    & (~dq006_mask)
    & (~shutdown_recovery_mask)
    & (~trace_moderate_wi_mask)
)
uncovered_rows = daily_df.loc[
    uncovered_mask,
    ["NPD_WELL_BORE_CODE", "NPD_WELL_BORE_NAME", "WELL_TYPE", "FLOW_KIND", "ON_STREAM_HRS", "BORE_OIL_VOL", "BORE_GAS_VOL", "BORE_WAT_VOL", "BORE_WI_VOL"],
]
print(f"Rows not covered by any prior investigation: {len(uncovered_rows)}")
uncovered_rows


Rows not covered by any prior investigation: 0


,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,WELL_TYPE,FLOW_KIND,ON_STREAM_HRS,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL


### Characterizing the REVIEW population


In [221]:
review_rows = daily_df.loc[record_state == "REVIEW"]
print(f"REVIEW total: {len(review_rows)}")

if len(review_rows):
    print("\nBy WELL_TYPE / FLOW_KIND:")
    print(review_rows.groupby(["WELL_TYPE", "FLOW_KIND"]).size())
    print("\nBy wellbore:")
    print(review_rows.groupby("NPD_WELL_BORE_CODE").size())


REVIEW total: 16

By WELL_TYPE / FLOW_KIND:
WELL_TYPE  FLOW_KIND 
OP         production    12
WI         injection      4
dtype: int64

By wellbore:
NPD_WELL_BORE_CODE
5351    1
5599    1
5693    2
5769    2
7289    2
7405    8
dtype: int64


### DQ-007 decision

Only introduced if the breakdowns above show a pattern not already covered by DQ-001 through DQ-006 — not by default for every REVIEW/INCONSISTENT row.


In [222]:
introduce_dq007 = unexplained_inconsistent > 0

print(f"Genuinely new, unexplained pattern found: {introduce_dq007}")
if introduce_dq007:
    print(f"-> DQ-007 warranted: {unexplained_inconsistent} INCONSISTENT row(s) not covered by any prior investigation.")
else:
    print("-> No DQ-007. Every INCONSISTENT row is already accounted for: DQ-005, DQ-006, or a specific")
    print("   investigation in Section 13.6 or 14.4 that examined it and chose not to escalate it.")
    print("   REVIEW rows (if any) reflect ON_STREAM_HRS > 0 with no material volume recorded -")
    print("   an ambiguous but not evidence-conflicting state, not treated as a new issue here.")


Genuinely new, unexplained pattern found: False
-> No DQ-007. Every INCONSISTENT row is already accounted for: DQ-005, DQ-006, or a specific
   investigation in Section 13.6 or 14.4 that examined it and chose not to escalate it.
   REVIEW rows (if any) reflect ON_STREAM_HRS > 0 with no material volume recorded -
   an ambiguous but not evidence-conflicting state, not treated as a new issue here.


In [223]:
print("=" * 60)
print("SECTION 18 - CROSS-VARIABLE CONSISTENCY SUMMARY")
print("=" * 60)
print(record_state_summary.to_string(index=False))
print()
print(f"Coherent: {n_coherent}/{n_total} ({n_coherent / n_total * 100:.2f}%)")
print(f"INCONSISTENT accounted for by prior sections: {len(inconsistent_rows) - unexplained_inconsistent}/{len(inconsistent_rows)}")
print(f"New issue (DQ-007) warranted: {introduce_dq007}")


SECTION 18 - CROSS-VARIABLE CONSISTENCY SUMMARY
      record_state  count   pct
 ACTIVE_PRODUCTION   7999 51.16
  ACTIVE_INJECTION   5373 34.37
 INACTIVE_RECORDED   1909 12.21
BLANK_STATE_RECORD    285  1.82
      INCONSISTENT     34  0.22
  TRANSITION_STATE     18  0.12
            REVIEW     16  0.10

Coherent: 15584/15634 (99.68%)
INCONSISTENT accounted for by prior sections: 34/34
New issue (DQ-007) warranted: False


**Interpretation for later relational modelling:**

**Answering this section's question directly: 15,584 of 15,634 records (99.68%) are internally coherent** given everything Sections 9-17 established — they classify cleanly as `ACTIVE_PRODUCTION` (7,999), `ACTIVE_INJECTION` (5,373), `INACTIVE_RECORDED` (1,909), `BLANK_STATE_RECORD` (285), or `TRANSITION_STATE` (18). Only **50 rows (0.32%)** need further interpretation: 34 `INCONSISTENT` + 16 `REVIEW`.

**All 34 `INCONSISTENT` rows are already accounted for by prior sections — not merely by formal DQ IDs.** My first pass at this cross-check only tested against the two *formal* DQ-005/DQ-006 masks and found "31 uncovered," which would have spuriously reopened a question Sections 13 and 14 already answered — caught and fixed before finalizing. The correct accounting is:

| Population | Count | Status |
|---|---|---|
| Negative water / unexplained zero-hours production | 1 | `DQ-005` |
| Large zero-hours/positive-injection | 2 | `DQ-006` |
| 2011-12-26 zero-hours rows (wellbores 5599, 5351) | 2 | Investigated in 13.6 — explained by next-day recovery jump, not escalated |
| Trace/moderate zero-hours-positive-injection | 29 | Investigated in 14.4 — explained by the injection/production `ON_STREAM_HRS` rate difference, not escalated |

1 + 2 + 2 + 29 = 34. **No new pattern — `DQ-007` is not created.** Sections 13.6 and 14.4 did the actual investigative work; this section's job was to confirm nothing was missed, not to redo it.

**The 16 `REVIEW` rows are a genuinely new observation, but a mild one**: `ON_STREAM_HRS > 0` with no material oil/gas/water/injection volume recorded (not NULL — exactly zero across the board on an "active" day). Concentrated somewhat in wellbore 7405 (8/16) and split `OP`/production (12) vs. `WI`/injection (4). This doesn't conflict with any evidence the way `INCONSISTENT` rows do — a well can legitimately show zero measured output on an operating day (ramp-up, brief test, measurement lag) — so it stays `REVIEW` rather than becoming a seventh issue ID. If a future pass finds this population recurring in a structured way, that would be new evidence; today it's 16 rows with no pattern beyond mild wellbore concentration.

**For relational modelling**: the 99.68% coherence rate is a strong signal that the confirmed Section 4 grain (`NPD_WELL_BORE_CODE + DATEPRD`) can carry a straightforward fact table — this is not a dataset requiring extensive pre-load remediation. The record-state classification here is diagnostic only (nothing written back, no permanent labels) but the underlying logic — `TRANSITION_STATE` → `BLANK_STATE_RECORD` → `INCONSISTENT` → `INACTIVE_RECORDED` → `ACTIVE_PRODUCTION`/`ACTIVE_INJECTION` → `REVIEW`, in that precedence order — is a reasonable candidate for a `REVIEW`-classification view or automated check later (Section 25), separate from the schema itself.

Moving to **Section 19 (duplicates)**, which Section 4 already answered (0 duplicate `NPD_WELL_BORE_CODE + DATEPRD` combinations) — kept intentionally short, confirming rather than re-investigating.


## 19. Duplicate-record assessment

Section 4 already established: 15,634 daily records, 15,634 distinct `NPD_WELL_BORE_CODE + DATEPRD` combinations, 0 duplicated key combinations. This section confirms that finding still holds and checks two things Section 4 didn't need to: exact whole-row duplicates, and whether any candidate-key duplicate would be exact vs. conflicting. Kept short on purpose — there is no reason to build a large analysis block around a population that is empty.


In [224]:
# --- TEMPORARY LOADING FOR SECTION 19 ---
if "daily_df" not in globals():
    from pathlib import Path
    import pandas as pd

    PROJECT_ROOT = Path.cwd().parent
    WORKBOOK_PATH = PROJECT_ROOT / "data" / "raw" / "Volve production data.xlsx"
    daily_df = pd.read_excel(WORKBOOK_PATH, sheet_name="Daily Production Data")
    print(f"[Section 19 temporary load] daily_df loaded: {daily_df.shape}")
else:
    print(f"Using daily_df already loaded earlier in the notebook: {daily_df.shape}")


Using daily_df already loaded earlier in the notebook: (15634, 24)


In [225]:
n_exact_duplicate_rows = int(daily_df.duplicated(keep=False).sum())
n_candidate_key_duplicates = int(daily_df.duplicated(subset=["NPD_WELL_BORE_CODE", "DATEPRD"], keep=False).sum())

print(f"Exact duplicate rows (identical across all columns): {n_exact_duplicate_rows}")
print(f"Candidate-key duplicates (NPD_WELL_BORE_CODE + DATEPRD): {n_candidate_key_duplicates}")
print(f"Conflicting duplicates (same key, different other values): {max(n_candidate_key_duplicates - n_exact_duplicate_rows, 0)}")


Exact duplicate rows (identical across all columns): 0
Candidate-key duplicates (NPD_WELL_BORE_CODE + DATEPRD): 0
Conflicting duplicates (same key, different other values): 0


In [226]:
print("Reference: Section 4 confirmed the daily grain (NPD_WELL_BORE_CODE + DATEPRD) is unique -")
print(f"15,634 rows, 15,634 distinct combinations, 0 duplicates. Result here: {n_candidate_key_duplicates} duplicates found.")
print()
duplicate_verdict = "CONFIRMED" if n_exact_duplicate_rows == 0 and n_candidate_key_duplicates == 0 else "NOT CONFIRMED"
print(f"VERDICT: {duplicate_verdict}")


Reference: Section 4 confirmed the daily grain (NPD_WELL_BORE_CODE + DATEPRD) is unique -
15,634 rows, 15,634 distinct combinations, 0 duplicates. Result here: 0 duplicates found.

VERDICT: CONFIRMED


**Interpretation:** no duplicate-record remediation is needed before schema design. This reconfirms rather than changes Section 4's finding — `NPD_WELL_BORE_CODE + DATEPRD` remains a validated primary-key candidate for the daily fact table.


## 20. Numeric outlier assessment

Sections 13-17 already examined unusual values for every numeric measurement in context (production volumes, injection volumes, pressures, temperatures, choke, operating hours). This section **consolidates that evidence into one table rather than re-running outlier detection** — no new IQR/z-score sweep, no rediscovery of what earlier sections already characterized.


In [227]:
import pandas as pd

outlier_summary_rows = [
    {
        "variable": "Oil volume (BORE_OIL_VOL)",
        "extreme_values": "0 negative; day-to-day jumps: 129 (>200%, floor-sensitive)",
        "interpretation": "REVIEW - routine allocation volatility (Section 13.6)",
    },
    {
        "variable": "Gas volume (BORE_GAS_VOL)",
        "extreme_values": "0 negative; day-to-day jumps: 131 (>200%, floor-sensitive)",
        "interpretation": "REVIEW - routine allocation volatility (Section 13.6)",
    },
    {
        "variable": "Water volume (BORE_WAT_VOL)",
        "extreme_values": "4 negative; day-to-day jumps: 151 (>200%, floor-sensitive)",
        "interpretation": "DQ-005 (negative values); rest REVIEW - allocation volatility",
    },
    {
        "variable": "Water injection (BORE_WI_VOL)",
        "extreme_values": "0 negative; 31 zero-hours/positive rows; jumps: 116 (>200%, NOT floor-sensitive)",
        "interpretation": "DQ-006 (2 large zero-hours rows); 29 trace/moderate REVIEW (14.4); jumps genuinely large, not baseline artifacts",
    },
    {
        "variable": "On-stream hours (ON_STREAM_HRS)",
        "extreme_values": "20 rows > 24 hours; 285 NULL",
        "interpretation": "DQ-004 (DST hypothesis, unconfirmed); DQ-003 (NULL population)",
    },
    {
        "variable": "Pressure (4 fields)",
        "extreme_values": "0 negative; downhole < WHP in 2,126 rows; jumps: 19-361 depending on floor (highly floor-sensitive)",
        "interpretation": "REVIEW - relationship unconfirmed pending units (Section 15.4)",
    },
    {
        "variable": "Temperature (2 fields)",
        "extreme_values": "0 negative; downhole < WHT in 2,117 rows; jumps: 87-98 depending on floor (mildly floor-sensitive)",
        "interpretation": "REVIEW - relationship unconfirmed pending units (Section 16.4)",
    },
    {
        "variable": "Choke (AVG_CHOKE_SIZE_P)",
        "extreme_values": "0 negative, 0 above 100; 672 positive-choke/zero-hours rows; jumps: 77-142 depending on floor",
        "interpretation": "REVIEW - largest open population in the operating-measurement block (Section 17.3)",
    },
]

outlier_summary = pd.DataFrame(outlier_summary_rows)
outlier_summary


,variable,extreme_values,interpretation
0,Oil volume (BORE_OIL_VOL),"0 negative; day-to-day jumps: 129 (>200%, floo...",REVIEW - routine allocation volatility (Sectio...
1,Gas volume (BORE_GAS_VOL),"0 negative; day-to-day jumps: 131 (>200%, floo...",REVIEW - routine allocation volatility (Sectio...
2,Water volume (BORE_WAT_VOL),"4 negative; day-to-day jumps: 151 (>200%, floo...",DQ-005 (negative values); rest REVIEW - alloca...
3,Water injection (BORE_WI_VOL),0 negative; 31 zero-hours/positive rows; jumps...,DQ-006 (2 large zero-hours rows); 29 trace/mod...
4,On-stream hours (ON_STREAM_HRS),20 rows > 24 hours; 285 NULL,"DQ-004 (DST hypothesis, unconfirmed); DQ-003 (..."
5,Pressure (4 fields),"0 negative; downhole < WHP in 2,126 rows; jump...",REVIEW - relationship unconfirmed pending unit...
6,Temperature (2 fields),"0 negative; downhole < WHT in 2,117 rows; jump...",REVIEW - relationship unconfirmed pending unit...
7,Choke (AVG_CHOKE_SIZE_P),"0 negative, 0 above 100; 672 positive-choke/ze...",REVIEW - largest open population in the operat...


**Methodological conclusion, already established across Sections 13-17 and restated here because it governs how this table should be read:**

> **Statistical outlier ≠ data-quality error.**

For an industrial time series like this one, whether an unusual value is a defect depends on well-specific context, operating state, absolute magnitude (not just relative change), and adjacent observations — not on a generic threshold. Every REVIEW row in the table above was investigated with that context (Sections 13.6, 14.4-14.6, 15.4-15.6, 16.4-16.6, 17.3-17.6) rather than flagged by a bare statistical rule. No global outlier-removal or capping step follows from this section — it is reference documentation of what was already found, not a new filtering pass.


## 21. Daily-to-monthly reconciliation

This is the section with direct relevance to SQL and database validation: does aggregating the daily source reproduce Equinor's supplied monthly source?

```
15,634 DAILY RECORDS
        |
        | aggregate
        v
wellbore + year + month
        |
        v
SUM(on-stream hours), SUM(oil), SUM(gas), SUM(water), SUM(water injection)
        |
        v
COMPARE
        |
        v
Equinor Monthly Production Data
```

The same shape of query will later run **inside PostgreSQL** and get compared against this same monthly reference:

```sql
SELECT
    npd_wellbore_code,
    EXTRACT(YEAR FROM production_date)  AS production_year,
    EXTRACT(MONTH FROM production_date) AS production_month,
    SUM(oil_volume)             AS oil_volume_sum,
    SUM(gas_volume)             AS gas_volume_sum,
    SUM(water_volume)           AS water_volume_sum,
    SUM(water_injection_volume) AS water_injection_volume_sum
FROM daily_production
GROUP BY npd_wellbore_code, production_year, production_month;
```

That gives the project an end-to-end validation chain: Excel daily source → PostgreSQL → SQL aggregation → monthly result → Equinor monthly reference → reconciliation. This section establishes the *expected* result of that chain using pandas, before any PostgreSQL work begins. Differences are quantified, not forced to match, and the known monthly-worksheet data-quality issues (Section 5: a stray non-data row; numeric columns loaded as text) are handled explicitly rather than silently.


In [228]:
# --- TEMPORARY LOADING FOR SECTION 21 ---
# Section 1 (Load source data) has not been implemented yet in this notebook.
# This block loads only what Section 21 needs (both worksheets), and is
# clearly marked so it can be deleted once Section 1 provides daily_df /
# monthly_df for the whole notebook.
from pathlib import Path
import pandas as pd

if "daily_df" not in globals() or "monthly_df" not in globals():
    PROJECT_ROOT = Path.cwd().parent
    WORKBOOK_PATH = PROJECT_ROOT / "data" / "raw" / "Volve production data.xlsx"
    if not WORKBOOK_PATH.exists():
        raise FileNotFoundError(f"Source workbook not found at {WORKBOOK_PATH}")

if "daily_df" not in globals():
    daily_df = pd.read_excel(WORKBOOK_PATH, sheet_name="Daily Production Data")
    print(f"[Section 21 temporary load] daily_df loaded: {daily_df.shape}")
else:
    print(f"Using daily_df already loaded earlier in the notebook: {daily_df.shape}")

if "monthly_df" not in globals():
    monthly_df = pd.read_excel(WORKBOOK_PATH, sheet_name="Monthly Production Data")
    print(f"[Section 21 temporary load] monthly_df loaded: {monthly_df.shape}")
else:
    print(f"Using monthly_df already loaded earlier in the notebook: {monthly_df.shape}")

if "dateprd_parsed" not in globals():
    dateprd_parsed = pd.to_datetime(daily_df["DATEPRD"], errors="coerce")


Using daily_df already loaded earlier in the notebook: (15634, 24)
Using monthly_df already loaded earlier in the notebook: (527, 10)


### 21.1 Aggregate the daily source by wellbore/year/month


In [229]:
daily_monthly_agg = (
    daily_df.assign(year=dateprd_parsed.dt.year, month=dateprd_parsed.dt.month)
    .groupby(["NPD_WELL_BORE_CODE", "year", "month"])
    .agg(
        on_stream_hrs_sum=("ON_STREAM_HRS", "sum"),
        oil_vol_sum=("BORE_OIL_VOL", "sum"),
        gas_vol_sum=("BORE_GAS_VOL", "sum"),
        water_vol_sum=("BORE_WAT_VOL", "sum"),
        wi_vol_sum=("BORE_WI_VOL", "sum"),
        daily_rows_in_month=("DATEPRD", "size"),
    )
    .reset_index()
)
print(f"Daily aggregated to {len(daily_monthly_agg)} wellbore/year/month groups.")
daily_monthly_agg.head(10)


Daily aggregated to 526 wellbore/year/month groups.


,NPD_WELL_BORE_CODE,year,month,on_stream_hrs_sum,oil_vol_sum,gas_vol_sum,water_vol_sum,wi_vol_sum,daily_rows_in_month
0,5351,2008,2,0.00000,0.00,0.00,0.00,0.0,18
1,5351,2008,3,0.00000,0.00,0.00,0.00,0.0,31
2,5351,2008,4,0.00000,0.00,0.00,0.00,0.0,29
3,5351,2008,5,0.00000,0.00,0.00,0.00,0.0,31
4,5351,2008,6,0.00000,0.00,0.00,0.00,0.0,30
5,5351,2008,7,437.57000,51285.08,7538861.07,126.17,0.0,31
6,5351,2008,8,458.40000,67621.20,9780532.34,222.53,0.0,30
7,5351,2008,9,642.70833,114596.26,16340514.75,275.73,0.0,30
8,5351,2008,10,742.50000,142817.06,20326866.32,101.25,0.0,31
9,5351,2008,11,717.83333,104409.61,14906020.01,189.89,0.0,30


### 21.2 Prepare the monthly source for comparison

The stray non-data row identified in Section 5 (all key columns NULL) is excluded from this comparison — it is not a real wellbore-month. The numeric columns, which loaded as text because of that row, are coerced to numeric here **in a derived copy only** — `monthly_df` itself is never modified anywhere in this notebook.


In [230]:
monthly_clean = monthly_df.dropna(subset=["NPDCode", "Year", "Month"]).copy()
monthly_clean["NPDCode"] = monthly_clean["NPDCode"].astype(int)
monthly_clean["Year"] = monthly_clean["Year"].astype(int)
monthly_clean["Month"] = monthly_clean["Month"].astype(int)

for col in ["On Stream", "Oil", "Gas", "Water", "WI"]:
    monthly_clean[col] = pd.to_numeric(monthly_clean[col], errors="coerce")

print(f"Monthly rows excluded (stray non-data row): {len(monthly_df) - len(monthly_clean)}")
print(f"Monthly rows retained for comparison: {len(monthly_clean)}")
monthly_clean.head(5)


Monthly rows excluded (stray non-data row): 1
Monthly rows retained for comparison: 526


,Wellbore name,NPDCode,Year,Month,On Stream,Oil,Gas,Water,GI,WI
1,15/9-F-1 C,7405,2014,4,227.50000,11142.47,1597936.65,0.00,NaN,NaN
2,15/9-F-1 C,7405,2014,5,733.83334,24901.95,3496229.65,783.48,NaN,NaN
3,15/9-F-1 C,7405,2014,6,705.91666,19617.76,2886661.69,2068.48,NaN,NaN
4,15/9-F-1 C,7405,2014,7,742.41666,15085.68,2249365.75,6243.98,NaN,NaN
5,15/9-F-1 C,7405,2014,8,432.99166,6970.43,1048190.80,4529.75,NaN,NaN


### 21.3 Coverage check

Which wellbore-months exist in one source but not the other, before comparing any values.


In [231]:
reconciliation = daily_monthly_agg.merge(
    monthly_clean,
    left_on=["NPD_WELL_BORE_CODE", "year", "month"],
    right_on=["NPDCode", "Year", "Month"],
    how="outer",
    indicator=True,
)

coverage_counts = reconciliation["_merge"].value_counts()
print("Wellbore/year/month coverage:")
print(coverage_counts)


Wellbore/year/month coverage:
_merge
both          526
left_only       0
right_only      0
Name: count, dtype: int64


In [232]:
only_in_daily = reconciliation.loc[reconciliation["_merge"] == "left_only", ["NPD_WELL_BORE_CODE", "year", "month", "daily_rows_in_month"]]
only_in_monthly = reconciliation.loc[reconciliation["_merge"] == "right_only", ["NPDCode", "Year", "Month"]]

print(f"Wellbore/year/month groups only in daily aggregation: {len(only_in_daily)}")
print(f"Wellbore/year/month groups only in monthly source:    {len(only_in_monthly)}")
only_in_daily


Wellbore/year/month groups only in daily aggregation: 0
Wellbore/year/month groups only in monthly source:    0


,NPD_WELL_BORE_CODE,year,month,daily_rows_in_month


### 21.4 Compute absolute and percentage differences

Restricted to wellbore/year/months present in **both** sources — coverage gaps were already reported above and are not re-litigated as value mismatches.


In [233]:
both = reconciliation.loc[reconciliation["_merge"] == "both"].copy()

metric_pairs = [
    ("on_stream_hrs", "on_stream_hrs_sum", "On Stream"),
    ("oil", "oil_vol_sum", "Oil"),
    ("gas", "gas_vol_sum", "Gas"),
    ("water", "water_vol_sum", "Water"),
    ("water_injection", "wi_vol_sum", "WI"),
]

for metric, daily_col, monthly_col in metric_pairs:
    both[f"{metric}_abs_diff"] = both[daily_col] - both[monthly_col]
    both[f"{metric}_pct_diff"] = (both[f"{metric}_abs_diff"] / both[monthly_col].abs().replace(0, pd.NA) * 100)

print(f"Wellbore/year/month groups compared: {len(both)}")


Wellbore/year/month groups compared: 526


### 21.5 Summary statistics per metric

The monthly source leaves a metric **blank (`NaN`), not `0`**, for months where it doesn't apply (e.g. `Oil` is NaN for a pure-injector wellbore, never `0`) — the same NULL≠0 convention this notebook has maintained throughout, now showing up on the *monthly* side. The daily aggregation naturally sums an all-NULL daily column to `0.0`. A naive numeric comparison would treat every one of those as a mismatch (`0.0 - NaN = NaN`), which is wrong — it's a **representation difference, not a reconciliation failure**. Three outcomes are distinguished per row: **MATCH** (both numeric and close), **NULL_CONVENTION** (monthly is NaN, daily sums to ~0 — contextually equivalent), and **MISMATCH** (a real, unexplained numeric difference, or monthly NaN against a non-zero daily sum).


In [234]:
ABS_TOLERANCE = 1e-6  # numerical epsilon, not a business tolerance - separates "identical" from "different"

summary_rows = []
for metric, daily_col, monthly_col in metric_pairs:
    abs_diff = both[f"{metric}_abs_diff"]
    daily_val = both[daily_col]
    monthly_val = both[monthly_col]

    is_match = abs_diff.abs() < ABS_TOLERANCE
    is_null_convention = monthly_val.isna() & (daily_val.abs() < ABS_TOLERANCE)
    is_mismatch = ~is_match & ~is_null_convention

    n_comparable = len(both)
    n_match = int(is_match.sum())
    n_null_convention = int(is_null_convention.sum())
    n_mismatch = int(is_mismatch.sum())

    summary_rows.append(
        {
            "metric": metric,
            "n_comparable": n_comparable,
            "match": n_match,
            "null_convention_0_vs_blank": n_null_convention,
            "mismatch": n_mismatch,
            "accounted_for_rate": round((n_match + n_null_convention) / n_comparable * 100, 2),
            "max_abs_diff_among_matches": abs_diff[is_match].abs().max() if n_match else float("nan"),
        }
    )

reconciliation_summary = pd.DataFrame(summary_rows)
reconciliation_summary


,metric,n_comparable,match,null_convention_0_vs_blank,mismatch,accounted_for_rate,max_abs_diff_among_matches
0,on_stream_hrs,526,515,11,0,100.0,1.136868e-13
1,oil,526,311,215,0,100.0,1.455192e-11
2,gas,526,311,215,0,100.0,9.313226e-10
3,water,526,311,215,0,100.0,1.455192e-11
4,water_injection,526,201,325,0,100.0,4.656613e-10


In [235]:
# Inspect the genuine MISMATCH rows directly, per metric - not just their count.
mismatch_detail_rows = []
for metric, daily_col, monthly_col in metric_pairs:
    abs_diff = both[f"{metric}_abs_diff"]
    daily_val = both[daily_col]
    monthly_val = both[monthly_col]
    is_match = abs_diff.abs() < ABS_TOLERANCE
    is_null_convention = monthly_val.isna() & (daily_val.abs() < ABS_TOLERANCE)
    is_mismatch = ~is_match & ~is_null_convention
    for _, row in both.loc[is_mismatch].iterrows():
        mismatch_detail_rows.append(
            {
                "metric": metric,
                "npd_code": row["NPD_WELL_BORE_CODE"],
                "year": row["year"],
                "month": row["month"],
                "daily_sum": row[daily_col],
                "monthly_value": row[monthly_col],
                "abs_diff": row[f"{metric}_abs_diff"],
                "daily_rows_in_month": row["daily_rows_in_month"],
            }
        )

mismatch_detail = pd.DataFrame(mismatch_detail_rows)
print(f"Genuine mismatches across all 5 metrics: {len(mismatch_detail)}")
mismatch_detail


Genuine mismatches across all 5 metrics: 0


""


### 21.6 Breakdown by wellbore

Is any mismatch systematic (concentrated in specific wellbores) rather than random noise?


In [236]:
by_wellbore_rows = []
for code_value, group in both.groupby("NPD_WELL_BORE_CODE"):
    row = {"npd_code": code_value, "n_months": len(group)}
    for metric, _, _ in metric_pairs:
        row[f"{metric}_median_pct_diff"] = group[f"{metric}_pct_diff"].median()
        row[f"{metric}_max_abs_pct_diff"] = group[f"{metric}_pct_diff"].abs().max()
    by_wellbore_rows.append(row)

by_wellbore = pd.DataFrame(by_wellbore_rows).sort_values("npd_code").reset_index(drop=True)
by_wellbore


,npd_code,n_months,on_stream_hrs_median_pct_diff,on_stream_hrs_max_abs_pct_diff,oil_median_pct_diff,oil_max_abs_pct_diff,gas_median_pct_diff,gas_max_abs_pct_diff,water_median_pct_diff,water_max_abs_pct_diff,water_injection_median_pct_diff,water_injection_max_abs_pct_diff
0,5351,104,0.0,0.000000e+00,0.0,2.094067e-14,0.0,1.998097e-14,0.0,1.558320e-14,NaN,NaN
1,5599,104,0.0,0.000000e+00,0.0,2.023320e-14,0.0,2.209720e-14,0.0,1.140879e-14,NaN,NaN
2,5693,112,0.0,1.487399e-14,NaN,NaN,NaN,NaN,NaN,NaN,0.0,3.045094e-13
3,5769,109,0.0,1.905792e-14,0.0,1.821669e-14,0.0,2.161288e-14,0.0,0.000000e+00,0.0,3.135006e-13
4,7078,39,0.0,1.693512e-14,0.0,0.000000e+00,0.0,2.016332e-14,0.0,1.898812e-14,NaN,NaN
5,7289,33,0.0,0.000000e+00,0.0,1.158294e-14,0.0,2.185623e-14,0.0,0.000000e+00,NaN,NaN
6,7405,25,0.0,0.000000e+00,0.0,1.158834e-14,0.0,1.843845e-14,0.0,1.754956e-14,NaN,NaN


### 21.7 Largest individual mismatches

Investigated for likely cause, not forced to match.


In [237]:
largest_mismatch_rows = []
for metric, daily_col, monthly_col in metric_pairs:
    top = both.reindex(both[f"{metric}_abs_diff"].abs().sort_values(ascending=False).index).head(3)
    for _, row in top.iterrows():
        largest_mismatch_rows.append(
            {
                "metric": metric,
                "npd_code": row["NPD_WELL_BORE_CODE"],
                "year": row["year"],
                "month": row["month"],
                "daily_sum": row[daily_col],
                "monthly_value": row[monthly_col],
                "abs_diff": row[f"{metric}_abs_diff"],
                "pct_diff": row[f"{metric}_pct_diff"],
                "daily_rows_in_month": row["daily_rows_in_month"],
            }
        )

largest_mismatches = pd.DataFrame(largest_mismatch_rows)
largest_mismatches


,metric,npd_code,year,month,daily_sum,monthly_value,abs_diff,pct_diff,daily_rows_in_month
0,on_stream_hrs,7078,2015,11,6.713082e+02,6.713082e+02,-1.136868e-13,-1.693512e-14,30
1,on_stream_hrs,5693,2009,3,1.910833e+02,1.910833e+02,2.842171e-14,1.487399e-14,31
2,on_stream_hrs,5769,2012,9,3.728334e+01,3.728334e+01,-7.105427e-15,-1.905792e-14,30
3,oil,5351,2010,6,7.530982e+04,7.530982e+04,-1.455192e-11,-1.932273e-14,30
4,oil,5351,2012,2,3.679766e+04,3.679766e+04,-7.275958e-12,-1.977288e-14,29
5,oil,5351,2012,3,3.474559e+04,3.474559e+04,7.275958e-12,2.094067e-14,31
6,gas,7078,2014,6,4.990685e+06,4.990685e+06,9.313226e-10,1.866122e-14,30
7,gas,7078,2016,7,2.309447e+06,2.309447e+06,-4.656613e-10,-2.016332e-14,31
8,gas,5351,2013,12,2.330524e+06,2.330524e+06,-4.656613e-10,-1.998097e-14,31
9,water,7078,2016,3,7.663696e+04,7.663696e+04,-1.455192e-11,-1.898812e-14,31


In [238]:
# Is coverage (missing daily rows within a month) a plausible driver of the
# largest mismatches? A month with fewer than a full calendar month's worth
# of daily rows is a concrete, checkable candidate cause.
import calendar

largest_mismatches["expected_days_in_month"] = largest_mismatches.apply(
    lambda r: calendar.monthrange(int(r["year"]), int(r["month"]))[1], axis=1
)
largest_mismatches["daily_rows_vs_expected"] = largest_mismatches["daily_rows_in_month"] - largest_mismatches["expected_days_in_month"]
largest_mismatches[["metric", "npd_code", "year", "month", "pct_diff", "daily_rows_in_month", "expected_days_in_month", "daily_rows_vs_expected"]]


,metric,npd_code,year,month,pct_diff,daily_rows_in_month,expected_days_in_month,daily_rows_vs_expected
0,on_stream_hrs,7078,2015,11,-1.693512e-14,30,30,0
1,on_stream_hrs,5693,2009,3,1.487399e-14,31,31,0
2,on_stream_hrs,5769,2012,9,-1.905792e-14,30,30,0
3,oil,5351,2010,6,-1.932273e-14,30,30,0
4,oil,5351,2012,2,-1.977288e-14,29,29,0
5,oil,5351,2012,3,2.094067e-14,31,31,0
6,gas,7078,2014,6,1.866122e-14,30,30,0
7,gas,7078,2016,7,-2.016332e-14,31,31,0
8,gas,5351,2013,12,-1.998097e-14,31,31,0
9,water,7078,2016,3,-1.898812e-14,31,31,0


### Reconciliation verdict


In [239]:
overall_accounted_for_rate = reconciliation_summary["accounted_for_rate"].mean()
n_coverage_gaps = len(only_in_daily) + len(only_in_monthly)
n_total_mismatches = len(mismatch_detail)

if n_coverage_gaps == 0 and overall_accounted_for_rate >= 99:
    reconciliation_verdict = "PASS"
elif overall_accounted_for_rate >= 90:
    reconciliation_verdict = "PASS WITH REVIEW"
else:
    reconciliation_verdict = "FAIL"

print("=" * 60)
print("SECTION 21 - DAILY-TO-MONTHLY RECONCILIATION SUMMARY")
print("=" * 60)
print(f"Wellbore/year/month groups compared:                {len(both)}")
print(f"Coverage gaps (daily-only + monthly-only):           {n_coverage_gaps}")
print(f"Total genuine mismatches (across all 5 metrics):     {n_total_mismatches}")
print(f"Average accounted-for rate (match + null-convention): {overall_accounted_for_rate:.2f}%")
print()
print(reconciliation_summary[["metric", "match", "null_convention_0_vs_blank", "mismatch", "accounted_for_rate"]].to_string(index=False))
print()
print(f"VERDICT: {reconciliation_verdict}")


SECTION 21 - DAILY-TO-MONTHLY RECONCILIATION SUMMARY
Wellbore/year/month groups compared:                526
Coverage gaps (daily-only + monthly-only):           0
Total genuine mismatches (across all 5 metrics):     0
Average accounted-for rate (match + null-convention): 100.00%

         metric  match  null_convention_0_vs_blank  mismatch  accounted_for_rate
  on_stream_hrs    515                          11         0               100.0
            oil    311                         215         0               100.0
            gas    311                         215         0               100.0
          water    311                         215         0               100.0
water_injection    201                         325         0               100.0

VERDICT: PASS


**Interpretation for later relational modelling:**

**The daily source reproduces Equinor's monthly reference essentially perfectly: 100% of all 526 x 5 = 2,630 wellbore/year/month/metric comparisons are accounted for, 0 coverage gaps, 0 genuine mismatches.** The maximum absolute difference among matched values is on the order of 1e-9 to 1e-13 — floating-point summation noise, not a real discrepancy. This is strong, direct evidence that Equinor's monthly worksheet is itself computed as a `SUM()` aggregation of the same underlying daily records (or a system very close to it), which is exactly what the planned PostgreSQL `GROUP BY` query is designed to test end-to-end once the daily fact table exists.

**Getting to that result required catching two of my own scoring bugs, not just accepting the first number produced — worth documenting because both are reusable lessons.**

1. **First pass**: I computed closeness using percentage difference with a divide-by-zero guard (`.replace(0, pd.NA)`). Every month where the true value was genuinely zero on both sides turned into a `NaN` percentage and got excluded from "close," dragging the reported match rate down to 37-94% and producing a false **FAIL** verdict — while the underlying absolute differences were already ~1e-14. This is the same near-zero-baseline trap Sections 13-17 established for day-to-day jump detection, showing up again here in a different form (a ratio metric, not a change metric).
2. **Second pass**: switching to an absolute-difference tolerance still showed only 38-59% "close" per metric, despite `max_abs_diff` staying at ~1e-9 among the rows that *did* match. The excluded rows weren't numeric mismatches at all — they were `NaN` comparisons. Checking the monthly source directly confirmed why: **it leaves `Oil`/`Gas`/`Water`/`WI` blank (`NaN`), not `0`, for months where that metric doesn't apply** (e.g. `Oil` is NaN for every month of the pure-injector wellbore 5693, never `0`) — the same NULL≠0 convention this notebook has enforced on the daily side since Section 11, now confirmed on the *monthly* side too. `0.0 - NaN = NaN` under a naive numeric diff, which is a representation difference, not a reconciliation failure. Splitting the outcome into three buckets (`match`, `null_convention_0_vs_blank`, `mismatch`) rather than one pass/fail number fixed this: 201-515 rows per metric are numeric matches, 11-325 are the 0-vs-blank convention (entirely expected, concentrated exactly where a wellbore's role makes a metric inapplicable), and genuine `mismatch` is **0 for every metric**.

**For relational modelling**, this is close to the best possible outcome for a reconciliation section: it validates that a straightforward `SUM(...) GROUP BY npd_wellbore_code, year, month` on the daily fact table will reproduce the monthly reference once loaded into PostgreSQL, with no aggregation-logic surprises to design around. The one thing worth carrying forward explicitly: the monthly source's blank-vs-zero convention needs the same care in SQL that it needed here — a `SUM()` over an empty/all-NULL group returns `0` in SQL just as it does in pandas, so a downstream comparison against the raw Excel-derived monthly reference table (if loaded as-is) would need the same `0 vs NULL` accommodation, not a raw equality check.

```
VERDICT: PASS
```

Handing off to **Section 22 (coverage)**, kept compact per your guidance, and then **Section 23**, which formalizes DQ-001 through DQ-006 as the first real deliverable of the issue register.


## 22. Coverage assessment

Compact by design — Section 9 already produced the per-wellbore coverage and gap analysis, and Section 21 already confirmed daily/monthly coverage match exactly (0 gaps in either direction). This section consolidates those results into one summary table rather than re-deriving them.


In [240]:
# --- TEMPORARY LOADING FOR SECTION 22 ---
if "daily_df" not in globals():
    from pathlib import Path
    import pandas as pd

    PROJECT_ROOT = Path.cwd().parent
    WORKBOOK_PATH = PROJECT_ROOT / "data" / "raw" / "Volve production data.xlsx"
    daily_df = pd.read_excel(WORKBOOK_PATH, sheet_name="Daily Production Data")
    print(f"[Section 22 temporary load] daily_df loaded: {daily_df.shape}")
else:
    print(f"Using daily_df already loaded earlier in the notebook: {daily_df.shape}")

if "dateprd_parsed" not in globals():
    dateprd_parsed = pd.to_datetime(daily_df["DATEPRD"], errors="coerce")
if "daily_with_dates" not in globals():
    daily_with_dates = daily_df.copy()
    daily_with_dates["DATEPRD_parsed"] = dateprd_parsed


Using daily_df already loaded earlier in the notebook: (15634, 24)


In [241]:
coverage_rows = []
for code_value, group in daily_with_dates.groupby("NPD_WELL_BORE_CODE"):
    name = group["NPD_WELL_BORE_NAME"].iloc[0]
    dates = group["DATEPRD_parsed"].dropna().drop_duplicates().sort_values()
    first_date, last_date = dates.iloc[0], dates.iloc[-1]
    calendar_span_days = (last_date - first_date).days + 1
    recorded_days = len(dates)
    gap_days_series = dates.diff().dt.days.dropna()
    coverage_rows.append(
        {
            "npd_code": code_value,
            "wellbore_name": name,
            "first_date": first_date.date(),
            "last_date": last_date.date(),
            "calendar_span_days": calendar_span_days,
            "recorded_days": recorded_days,
            "coverage_pct": round(recorded_days / calendar_span_days * 100, 2),
            "max_gap_days": int(gap_days_series.max()) if len(gap_days_series) else 0,
            "gaps_gt_7_days": int((gap_days_series > 7).sum()),
        }
    )

coverage_summary = pd.DataFrame(coverage_rows).sort_values("npd_code").reset_index(drop=True)
coverage_summary


,npd_code,wellbore_name,first_date,last_date,calendar_span_days,recorded_days,coverage_pct,max_gap_days,gaps_gt_7_days
0,5351,15/9-F-14,2008-02-12,2016-09-17,3141,3056,97.29,12,1
1,5599,15/9-F-12,2008-02-12,2016-09-17,3141,3056,97.29,12,1
2,5693,15/9-F-4,2007-09-01,2016-12-01,3380,3327,98.43,30,2
3,5769,15/9-F-5,2007-09-01,2016-09-18,3306,3306,100.00,1,0
4,7078,15/9-F-11,2013-07-08,2016-09-17,1168,1165,99.74,2,0
5,7289,15/9-F-15 D,2014-01-12,2016-09-17,980,978,99.80,2,0
6,7405,15/9-F-1 C,2014-04-07,2016-04-21,746,746,100.00,1,0


In [242]:
print("Field-level coverage: 2007-09-01 to 2016-12-01 (Section 9.2), 53 calendar days with no")
print("record from any wellbore, out of 3,380 total calendar days in the field's recorded span.")
print()
print("Daily-vs-monthly coverage (Section 21): 526/526 wellbore/year/month groups present in both")
print("sources - 0 groups only in daily, 0 groups only in monthly.")


Field-level coverage: 2007-09-01 to 2016-12-01 (Section 9.2), 53 calendar days with no
record from any wellbore, out of 3,380 total calendar days in the field's recorded span.

Daily-vs-monthly coverage (Section 21): 526/526 wellbore/year/month groups present in both
sources - 0 groups only in daily, 0 groups only in monthly.


### Is coverage sufficient for the intended analyses?

**Yes.** Every wellbore has 97.3-100% daily coverage over its own recorded span (Section 9.3), the field-wide gap is 53 days out of 3,380 (1.6%), and the daily and monthly sources agree on which wellbore-months exist with zero exceptions (Section 21). The known gaps (DQ-002's 12-day shared gap, and the wellbores' individually shorter spans reflecting genuine spud/completion/abandonment dates rather than missing reporting) do not threaten the ability to build a daily fact table or reconcile it against the monthly reference. No coverage-driven blocker exists for schema design.


## 23. Data-quality issue register

Formalizing `DQ-001` through `DQ-006` — anticipated since Section 9, referenced throughout Sections 10-21, formally registered here for the first time. **No treatments are implemented in this section** — only structured documentation of what is known, so Section 24 can make schema decisions with a single source of truth instead of re-deriving each issue's history from prose scattered across ten sections.


In [243]:
import pandas as pd

issue_register = pd.DataFrame(
    [
        {
            "issue_id": "DQ-001",
            "affected_dataset": "Daily Production Data",
            "affected_columns": "DATEPRD, WELL_TYPE, FLOW_KIND, ON_STREAM_HRS, BORE_WI_VOL",
            "description": (
                "244 rows (wellbores 5693, 5769) dated before the documented Volve field-life start "
                "(2008-01-01), all WELL_TYPE=WI / FLOW_KIND=injection, all with NULL ON_STREAM_HRS and "
                "zero measurements across every volume/pressure/temperature/choke column (confirmed "
                "Section 14.7: 0/285 DQ-001/DQ-003 rows carry any measurement)."
            ),
            "severity": "Low",
            "affected_records": 244,
            "proposed_treatment": "Retain as-is; do not delete or backfill. Consider a documented flag/comment on load, not a schema constraint.",
            "domain_interpretation_required": "Yes - confirm whether pre-2008 water injection is legitimate (reservoir pressure support before first oil) or a reporting-system artifact.",
            "status": "Open - characterized, not resolved",
        },
        {
            "issue_id": "DQ-002",
            "affected_dataset": "Daily Production Data",
            "affected_columns": "DATEPRD, NPD_WELL_BORE_CODE",
            "description": (
                "Wellbores 5351 and 5599 both show a 12-day reporting gap over the identical calendar "
                "window (2012-01-02 to 2012-01-14) - a coincidence across two independent wells "
                "(Section 9.4). Deliberately not investigated further in Sections 10-18 per explicit "
                "scope discipline (belongs to operational/event interpretation, not a generic rule test)."
            ),
            "severity": "Low",
            "affected_records": 0,
            "proposed_treatment": "No data action - it is an absence of rows, not corrupted data. Document as a known gap; do not backfill or interpolate.",
            "domain_interpretation_required": "Yes - confirm cause (shared facility shutdown/maintenance is the leading hypothesis, unconfirmed).",
            "status": "Open - identified, not investigated",
        },
        {
            "issue_id": "DQ-003",
            "affected_dataset": "Daily Production Data",
            "affected_columns": "ON_STREAM_HRS (and, by extension, every measurement column on the same rows)",
            "description": (
                "285 rows (wellbores 5693, 5769 only) with NULL ON_STREAM_HRS. Overlaps heavily but not "
                "completely with DQ-001 (244/285 are pre-2008; 41 are post-2008 on the same two "
                "wellbores). Confirmed blank: 0/285 rows carry any positive volume measurement "
                "(Section 14.7) and missingness reaches 100% across all five pressure/temperature "
                "fields on these rows too (Section 15.3)."
            ),
            "severity": "Low",
            "affected_records": 285,
            "proposed_treatment": "Load ON_STREAM_HRS as nullable; do not impute. Consider a derived boolean or status column identifying these as structurally blank records.",
            "domain_interpretation_required": "Yes - confirm meaning (pre-commissioning / not-yet-metered period is the leading hypothesis).",
            "status": "Open - characterized, not resolved",
        },
        {
            "issue_id": "DQ-004",
            "affected_dataset": "Daily Production Data",
            "affected_columns": "ON_STREAM_HRS",
            "description": (
                "20 rows across 6 wellbores, on exactly 5 calendar dates, each the last Sunday of "
                "October (EU/Norway DST 'fall back' date) for its year - a precise, non-coincidental "
                "date match (Section 10.7, check 2: confirmed). But the DST hypothesis is only "
                "partially supported: only 13/20 values are exactly 25.0 (check 1: partial), and the "
                "effect is not uniform across wellbores reporting on the same date - e.g. wellbore 5693 "
                "read 0.0 hours on 2010-10-31 while three other wells read 25.0 (check 3: not confirmed)."
            ),
            "severity": "Low-Medium",
            "affected_records": 20,
            "proposed_treatment": (
                "Do not widen a CHECK constraint from 24 to 25 to accommodate these rows without "
                "confirming the DST explanation - that would fit the rule to the data rather than "
                "understanding it. Load ON_STREAM_HRS without an upper-bound constraint pending confirmation."
            ),
            "domain_interpretation_required": "Yes - requires Equinor source/reporting-timezone documentation (Section 10.7, check 5, not testable from data alone).",
            "status": "Open - DST hypothesis partially supported, not confirmed",
        },
        {
            "issue_id": "DQ-005",
            "affected_dataset": "Daily Production Data",
            "affected_columns": "BORE_WAT_VOL, ON_STREAM_HRS, BORE_OIL_VOL, BORE_GAS_VOL",
            "description": (
                "4 negative BORE_WAT_VOL values (wellbores 5351, 5599), 2 of which share an identical "
                "date (2012-08-13) and fractional ON_STREAM_HRS=0.625. Plus 1 unexplained zero-hours/"
                "positive-production row (wellbore 7078, 2015-01-17, substantial oil and gas volume "
                "against 0 recorded hours). Two further zero-hours/positive-production rows "
                "(2011-12-26, wellbores 5599 and 5351) were investigated in Section 13.6 and explained "
                "by a next-day production-recovery jump - not counted in this issue's affected-record total."
            ),
            "severity": "Low",
            "affected_records": 5,
            "proposed_treatment": "Retain as-is - negative volumes may represent legitimate allocation corrections. Do not force to zero or positive. No non-negativity CHECK constraint on BORE_WAT_VOL until reviewed.",
            "domain_interpretation_required": "Yes",
            "status": "Open - characterized, not resolved",
        },
        {
            "issue_id": "DQ-006",
            "affected_dataset": "Daily Production Data",
            "affected_columns": (
                "BORE_WI_VOL, ON_STREAM_HRS, AVG_DOWNHOLE_PRESSURE, AVG_DP_TUBING, AVG_ANNULUS_PRESS, "
                "AVG_WHP_P, AVG_DOWNHOLE_TEMPERATURE, AVG_WHT_P, AVG_CHOKE_SIZE_P, AVG_CHOKE_UOM, DP_CHOKE_SIZE"
            ),
            "description": (
                "2 rows (wellbore 5693 on 2011-06-18: 6,263.69 m3; wellbore 5769 on 2013-03-16: 148.66 m3) "
                "record substantial water injection against ON_STREAM_HRS=0, while every pressure field "
                "(4/4), every temperature field (2/2), and AVG_CHOKE_SIZE_P/AVG_CHOKE_UOM are all NULL, "
                "and DP_CHOKE_SIZE reads exactly 0. A full record-level contradiction: multiple "
                "independent fields all read as 'inactive' while the injection volume alone reads as "
                "substantial activity (Sections 14.4, 15.6, 16.6, 17.6)."
            ),
            "severity": "Medium",
            "affected_records": 2,
            "proposed_treatment": "Retain as-is; flag explicitly (e.g. a data-quality note or review flag on load). Do not zero out or discard the volume.",
            "domain_interpretation_required": "Yes",
            "status": "Open - fully characterized across 4 independent field groups, not resolved",
        },
    ]
)
issue_register


,issue_id,affected_dataset,affected_columns,description,severity,affected_records,proposed_treatment,domain_interpretation_required,status
0,DQ-001,Daily Production Data,"DATEPRD, WELL_TYPE, FLOW_KIND, ON_STREAM_HRS, ...","244 rows (wellbores 5693, 5769) dated before t...",Low,244,Retain as-is; do not delete or backfill. Consi...,Yes - confirm whether pre-2008 water injection...,"Open - characterized, not resolved"
1,DQ-002,Daily Production Data,"DATEPRD, NPD_WELL_BORE_CODE",Wellbores 5351 and 5599 both show a 12-day rep...,Low,0,"No data action - it is an absence of rows, not...",Yes - confirm cause (shared facility shutdown/...,"Open - identified, not investigated"
2,DQ-003,Daily Production Data,"ON_STREAM_HRS (and, by extension, every measur...","285 rows (wellbores 5693, 5769 only) with NULL...",Low,285,Load ON_STREAM_HRS as nullable; do not impute....,Yes - confirm meaning (pre-commissioning / not...,"Open - characterized, not resolved"
3,DQ-004,Daily Production Data,ON_STREAM_HRS,"20 rows across 6 wellbores, on exactly 5 calen...",Low-Medium,20,Do not widen a CHECK constraint from 24 to 25 ...,Yes - requires Equinor source/reporting-timezo...,"Open - DST hypothesis partially supported, not..."
4,DQ-005,Daily Production Data,"BORE_WAT_VOL, ON_STREAM_HRS, BORE_OIL_VOL, BOR...",4 negative BORE_WAT_VOL values (wellbores 5351...,Low,5,Retain as-is - negative volumes may represent ...,Yes,"Open - characterized, not resolved"
5,DQ-006,Daily Production Data,"BORE_WI_VOL, ON_STREAM_HRS, AVG_DOWNHOLE_PRESS...","2 rows (wellbore 5693 on 2011-06-18: 6,263.69 ...",Medium,2,Retain as-is; flag explicitly (e.g. a data-qua...,Yes,Open - fully characterized across 4 independen...


In [244]:
print("Severity distribution:")
print(issue_register["severity"].value_counts())
print()
print(f"Total affected records across all 6 issues (may overlap across issues): {issue_register['affected_records'].sum()}")
print(f"Records fully investigated with no unresolved issue (Section 18): {15634 - 34 - 16} coherent")


Severity distribution:
severity
Low           4
Low-Medium    1
Medium        1
Name: count, dtype: int64

Total affected records across all 6 issues (may overlap across issues): 556
Records fully investigated with no unresolved issue (Section 18): 15584 coherent


**Note on overlap:** DQ-001 and DQ-003 describe overlapping but non-identical populations (244 of 285 DQ-003 rows are also DQ-001; 41 are not) — both are kept as separate entries because they answer different questions (temporal boundary vs. measurement completeness), not because they are independent record counts. Summing `affected_records` across all six issues therefore over-counts the true number of distinct flagged rows; Section 18 (`INCONSISTENT` + `REVIEW` = 34 + 16 = 50 rows) is the correct de-duplicated count of records needing interpretation beyond DQ-001/DQ-003's 285.

**No treatments are implemented here.** Every `proposed_treatment` above is a recommendation for Section 24 and beyond to act on, not an action already taken.


## 24. Database-modelling implications

This section makes **actual decisions**, based on the validated findings of Sections 4-23 — not another round of "it depends." Where evidence doesn't yet support a decision, that is stated explicitly as an open question (Section 23's `domain_interpretation_required` column), not left implicit.


### 24.1 Confirmed grain of each fact table

| Table | Grain | Evidence |
|---|---|---|
| Daily production fact | one row per wellbore per calendar date | Section 4: CONFIRMED, 0 duplicates; Section 19: reconfirmed |
| Monthly reference table | one row per wellbore per year/month | Section 5: REQUIRES REVIEW only because of the 1 stray non-data row (excluded, not a grain violation); 0 duplicates among real rows |

**Decision:** the daily table is the primary fact table. The monthly worksheet is loaded as a separate, smaller **reference/reconciliation table** — not a second fact table for analysis — because Section 21 confirmed it is reproducible from the daily table via `SUM() GROUP BY` with 0 genuine mismatches. Keeping it as a loaded reference table (rather than discarding it) preserves the ability to re-run that reconciliation after any future daily-data changes.


### 24.2 Preferred wellbore identifier

**Decision: `NPD_WELL_BORE_CODE`.** Numeric, confirmed 1:1 with `NPD_WELL_BORE_NAME` and `WELL_BORE_CODE` in both directions (Section 6: CONFIRMED, 0 violations), confirmed to be the same identifier system as the monthly worksheet's `NPDCode` (Section 7: CONFIRMED, all 7 wellbores match with identical names on both sides). `NPD_WELL_BORE_NAME` and `WELL_BORE_CODE` are retained as descriptive attributes, not join keys.


### 24.3 Candidate primary and foreign keys

| Table | Primary key | Foreign keys |
|---|---|---|
| `wellbore` (dimension) | `npd_wellbore_code` | — |
| `daily_production` (fact) | `(npd_wellbore_code, production_date)` | `npd_wellbore_code` → `wellbore.npd_wellbore_code` |
| `monthly_production` (reference) | `(npd_wellbore_code, production_year, production_month)` | `npd_wellbore_code` → `wellbore.npd_wellbore_code` |


### 24.4 Fields that should be NOT NULL

**Decision, `daily_production`:** `npd_wellbore_code`, `production_date`, `well_type`, `flow_kind` — all four are 0% missing across all 15,634 rows (Sections 4, 6, 12) and are structurally required for every row to be interpretable at all.

**Decision, everything else on `daily_production`:** nullable. `on_stream_hrs` and all volume/pressure/temperature/choke columns carry real, evidenced NULL populations (DQ-001/DQ-003's blank records; the CONTEXTUAL columns from Section 11 that are legitimately not applicable depending on `well_type`/`flow_kind`). Forcing NOT NULL on any of these would either reject real source rows or require inventing placeholder values this notebook was explicitly told not to invent.

**Decision, `wellbore`:** `npd_wellbore_code`, `npd_wellbore_name` NOT NULL (0% missing, Section 6).


### 24.5 Possible CHECK constraints

| Column | Constraint | Decision | Evidence |
|---|---|---|---|
| `on_stream_hrs` | `>= 0` | **Enforce** | 0 negative values found anywhere (Section 10.1) |
| `on_stream_hrs` | `<= 24` or `<= 25` | **Do not enforce** | DQ-004 unresolved — enforcing either bound now means guessing; see Section 23 |
| `avg_downhole_pressure`, `avg_dp_tubing`, `avg_annulus_press`, `avg_whp_p` | `>= 0` | **Enforce** | 0 negative values across all four (Section 15.1) |
| `avg_downhole_temperature`, `avg_wht_p` | `>= 0` | **Enforce** | 0 negative values across both (Section 16.1) |
| `avg_choke_size_p` | `>= 0` | **Enforce** | 0 negative values (Section 17.1) |
| `avg_choke_size_p` | `<= 100` | **Do not enforce** | 0 violations observed, but the definition itself is unconfirmed (Section 17.1) — a currently-true fact is not the same as a confirmed rule |
| `bore_oil_vol`, `bore_gas_vol` | `>= 0` | **Enforce** | 0 negative values in either column (Section 13.1) |
| `bore_wat_vol` | `>= 0` | **Do not enforce** | DQ-005: 4 confirmed real negative values in the source (Section 13.1) |
| `bore_wi_vol` | `>= 0` | **Enforce** | 0 negative values (Section 14.1) |
| `avg_downhole_pressure >= avg_whp_p` | cross-column | **Do not enforce** | 2,126 rows violate this (Section 15.4); relationship convention unconfirmed |
| `avg_downhole_temperature >= avg_wht_p` | cross-column | **Do not enforce** | 2,117 rows violate this (Section 16.4); same reason |

**Rule applied throughout:** a constraint is only added where the data has *zero* observed violations across the full 15,634-row history. A rule that is merely "usually true" is not encoded as a hard constraint — it becomes a monitored condition instead (Section 25).


### 24.6 Wellbore / field / facility dimension design

**Decision: fold field and facility into the `wellbore` dimension as plain attributes for v1** (`npd_field_code`, `npd_field_name`, `npd_facility_code`, `npd_facility_name`), rather than separate `field`/`facility` dimension tables. Section 8 confirmed exactly one field (`3420717 / VOLVE`) and one facility (`369304 / MÆRSK INSPIRER`) for the entire dataset, with 0 wellbores ever associated with more than one of either — a separate dimension table would hold exactly one row each, adding join overhead without adding modelling power for this dataset.

This decision is Volve-specific, not a template rule: a reusable version of this schema for a multi-field or multi-facility dataset should reconsider this the moment cardinality exceeds 1, using the same Section 8 methodology (test cardinality and per-wellbore stability before deciding, don't assume).

**Decision: `well_type` and `flow_kind` stay on the fact table, not the dimension.** Section 12 confirmed both are TEMPORAL, not STATIC — 2 of 7 wellbores show real `well_type` transitions and 1 of 7 shows a real `flow_kind` transition, both corresponding to genuine multi-day-to-multi-month operational periods (Section 12.3/12.4), not noise. Putting either on a flat one-row-per-wellbore dimension would silently discard that history. If a wellbore-status history becomes valuable later, it should be a type-2 table keyed by `(npd_wellbore_code, effective_date)`, not an attribute on `wellbore`.


### 24.7 Production and injection: one fact table

**Decision: one unified `daily_production` fact table**, not separate production/injection tables. Reasons, all evidenced:

1. **Same grain, same identity.** Every row — producing, injecting, in transition, or blank — is one wellbore on one date (Section 4). Splitting by mode would require deciding which table owns the 18 `WI`/production transition rows (Section 12), which have production-like activity 9/18 of the time and neither activity the other 9/18 — there is no clean split point.
2. **The measurement columns are already correctly nullable and contextual**, not duplicated. Section 11/15/16/17 confirmed the CONTEXTUAL columns are ~100% populated for their applicable mode and ~100% NULL otherwise — exactly the behavior a single wide nullable table should have, with no need for two narrower tables to avoid sparse columns.
3. **DQ-006's cross-field contradiction (Sections 14-17) depends on comparing injection volume against pressure/temperature/choke on the *same row*.** Splitting production and injection into separate tables would require a join to reproduce a check this notebook could do with a single row lookup.

`well_type` and `flow_kind` remain the columns that distinguish a row's operating mode within the single table.


### 24.8 Unresolved questions before schema creation

Pulled directly from Section 23's `domain_interpretation_required` column — nothing new introduced here.


In [245]:
print("Open questions requiring domain/source confirmation before final constraints can be tightened:")
print()
for _, row in issue_register.iterrows():
    print(f"{row['issue_id']}: {row['domain_interpretation_required']}")
print()
print("Additional open questions from Sections 15-17 (not tied to a specific DQ ID):")
print("  - AVG_DOWNHOLE_PRESSURE / AVG_WHP_P units and measurement convention (2,126 rows show downhole < wellhead)")
print("  - AVG_DOWNHOLE_TEMPERATURE / AVG_WHT_P units and measurement convention (2,117 rows show the same pattern)")
print("  - AVG_CHOKE_SIZE_P's exact definition and confirmed upper bound (currently observed <= 100 but not confirmed as a rule)")


Open questions requiring domain/source confirmation before final constraints can be tightened:

DQ-001: Yes - confirm whether pre-2008 water injection is legitimate (reservoir pressure support before first oil) or a reporting-system artifact.
DQ-002: Yes - confirm cause (shared facility shutdown/maintenance is the leading hypothesis, unconfirmed).
DQ-003: Yes - confirm meaning (pre-commissioning / not-yet-metered period is the leading hypothesis).
DQ-004: Yes - requires Equinor source/reporting-timezone documentation (Section 10.7, check 5, not testable from data alone).
DQ-005: Yes
DQ-006: Yes

Additional open questions from Sections 15-17 (not tied to a specific DQ ID):
  - AVG_DOWNHOLE_PRESSURE / AVG_WHP_P units and measurement convention (2,126 rows show downhole < wellhead)
  - AVG_DOWNHOLE_TEMPERATURE / AVG_WHT_P units and measurement convention (2,117 rows show the same pattern)
  - AVG_CHOKE_SIZE_P's exact definition and confirmed upper bound (currently observed <= 100 but not 

None of these block starting schema construction — they block *tightening* certain constraints later. The schema in Section 24.1-24.7 can be built now with the constraints already decided; each open question above becomes a candidate follow-up migration once answered, not a prerequisite.


## 25. Automated checks to promote into `profile_source.py`

Selection only — **nothing here is implemented in this notebook.** Each check is classified `FAIL` (a deterministic structural rule with zero observed exceptions across all 15,634 rows) or `REVIEW` (a condition needing engineering/domain interpretation, per Section 10's `PASS`/`FAIL`/`REVIEW` model). No check is marked `FAIL` unless this notebook found zero violations for it — a rule that is merely "usually true" is a `REVIEW` candidate, never a hard failure.


In [246]:
import pandas as pd

automated_checks = pd.DataFrame(
    [
        {"check": "Source workbook file exists", "rule": "File present at expected path", "outcome": "FAIL", "evidence": "Structural precondition; nothing else can run without it"},
        {"check": "Expected worksheets exist", "rule": "'Daily Production Data' and 'Monthly Production Data' present", "outcome": "FAIL", "evidence": "Section 1"},
        {"check": "Required columns present", "rule": "All columns this notebook depends on exist per sheet", "outcome": "FAIL", "evidence": "Sections 2-3"},
        {"check": "DATEPRD parses cleanly", "rule": "0 NULL, 0 unparseable values", "outcome": "FAIL", "evidence": "Section 9.1: 0/15,634 violations observed"},
        {"check": "Daily candidate-key uniqueness", "rule": "0 duplicate (npd_wellbore_code, production_date) pairs", "outcome": "FAIL", "evidence": "Sections 4, 19: 0/15,634 violations"},
        {"check": "Monthly candidate-key uniqueness", "rule": "0 duplicate (npd_wellbore_code, year, month) pairs among real rows", "outcome": "FAIL", "evidence": "Section 5: 0/526 violations"},
        {"check": "Monthly stray non-data rows", "rule": "Any row with all key columns NULL", "outcome": "REVIEW", "evidence": "Section 5: 1 such row found (units-header artifact) - flag for inspection, don't assume the same shape every load"},
        {"check": "Wellbore identifier mapping consistency", "rule": "npd_wellbore_code <-> npd_wellbore_name is 1:1 both directions", "outcome": "FAIL", "evidence": "Section 6: 0/7 violations"},
        {"check": "Cross-sheet wellbore identity match", "rule": "Daily and monthly wellbore code sets are identical", "outcome": "FAIL", "evidence": "Section 7: 7/7 match exactly"},
        {"check": "ON_STREAM_HRS non-negative", "rule": "ON_STREAM_HRS >= 0", "outcome": "FAIL", "evidence": "Section 10.1: 0/15,634 violations"},
        {"check": "ON_STREAM_HRS upper bound", "rule": "ON_STREAM_HRS > 24", "outcome": "REVIEW", "evidence": "DQ-004: 20 rows, DST hypothesis unconfirmed - never auto-widen to 25"},
        {"check": "WELL_TYPE valid category", "rule": "WELL_TYPE in {OP, WI}", "outcome": "FAIL", "evidence": "Section 12.1: exactly 2 categories observed, always"},
        {"check": "FLOW_KIND valid category", "rule": "FLOW_KIND in {production, injection}", "outcome": "FAIL", "evidence": "Section 12.1: exactly 2 categories observed, always"},
        {"check": "Oil/gas volume non-negative", "rule": "BORE_OIL_VOL >= 0 and BORE_GAS_VOL >= 0", "outcome": "FAIL", "evidence": "Section 13.1: 0/9,161 violations in either column"},
        {"check": "Water volume non-negative", "rule": "BORE_WAT_VOL >= 0", "outcome": "REVIEW", "evidence": "DQ-005: 4 confirmed real negative values - never a hard rule"},
        {"check": "Injection volume non-negative", "rule": "BORE_WI_VOL >= 0", "outcome": "FAIL", "evidence": "Section 14.1: 0/5,706 violations"},
        {"check": "Injection volume vs. state", "rule": "Positive BORE_WI_VOL only on WELL_TYPE=WI or FLOW_KIND=injection rows", "outcome": "REVIEW", "evidence": "Section 14.3: 0 violations observed today, but this is a state-consistency check, not a magnitude bound - keep as REVIEW, not FAIL"},
        {"check": "Pressure fields non-negative", "rule": "All 4 pressure columns >= 0", "outcome": "FAIL", "evidence": "Section 15.1: 0 violations across all 4"},
        {"check": "Temperature fields non-negative", "rule": "Both temperature columns >= 0", "outcome": "FAIL", "evidence": "Section 16.1: 0 violations across both"},
        {"check": "Choke size lower bound", "rule": "AVG_CHOKE_SIZE_P >= 0", "outcome": "FAIL", "evidence": "Section 17.1: 0/8,919 violations"},
        {"check": "Choke size upper bound", "rule": "AVG_CHOKE_SIZE_P > 100", "outcome": "REVIEW", "evidence": "Section 17.1: 0 violations observed, but the % definition itself is unconfirmed - do not FAIL on an unconfirmed rule"},
        {"check": "Daily-to-monthly reconciliation", "rule": "SUM() of daily by wellbore/year/month matches monthly reference within tolerance", "outcome": "REVIEW", "evidence": "Section 21: 100% matched this run, but a future data revision could introduce a real gap - re-run every load, don't assume"},
        {"check": "Source row-count drift", "rule": "Daily/monthly row counts within an expected range of the prior run", "outcome": "REVIEW", "evidence": "Informational drift signal, not evidenced by this single-snapshot notebook"},
        {"check": "Source date-range sanity", "rule": "DATEPRD outside a documented expected field-life window", "outcome": "REVIEW", "evidence": "DQ-001: legitimate pre-2008 records exist - never FAIL on this alone"},
    ]
)
automated_checks


,check,rule,outcome,evidence
0,Source workbook file exists,File present at expected path,FAIL,Structural precondition; nothing else can run ...
1,Expected worksheets exist,'Daily Production Data' and 'Monthly Productio...,FAIL,Section 1
2,Required columns present,All columns this notebook depends on exist per...,FAIL,Sections 2-3
3,DATEPRD parses cleanly,"0 NULL, 0 unparseable values",FAIL,"Section 9.1: 0/15,634 violations observed"
4,Daily candidate-key uniqueness,"0 duplicate (npd_wellbore_code, production_dat...",FAIL,"Sections 4, 19: 0/15,634 violations"
5,Monthly candidate-key uniqueness,"0 duplicate (npd_wellbore_code, year, month) p...",FAIL,Section 5: 0/526 violations
6,Monthly stray non-data rows,Any row with all key columns NULL,REVIEW,Section 5: 1 such row found (units-header arti...
7,Wellbore identifier mapping consistency,npd_wellbore_code <-> npd_wellbore_name is 1:1...,FAIL,Section 6: 0/7 violations
8,Cross-sheet wellbore identity match,Daily and monthly wellbore code sets are ident...,FAIL,Section 7: 7/7 match exactly
9,ON_STREAM_HRS non-negative,ON_STREAM_HRS >= 0,FAIL,"Section 10.1: 0/15,634 violations"


In [247]:
print("Outcome distribution:")
print(automated_checks["outcome"].value_counts())


Outcome distribution:
outcome
FAIL      16
REVIEW     8
Name: count, dtype: int64


**Not implemented here, by design.** This table is the design handoff to `src/profile_source.py` (Phase 1) — the next step is adding these checks there, each returning `PASS`/`FAIL`/`REVIEW` per row rather than the current single pass/fail-style prints, using exactly the thresholds and rationale documented above. That implementation work is explicitly out of scope for this notebook.


## 26. Quality assessment summary

This section summarizes Sections 4-25. It introduces no new analysis.

### Dataset suitability

**SUITABLE WITH DOCUMENTED EXCEPTIONS**

Not "clean" — it isn't, and it doesn't need to be. Six specific, quantified, traceable exceptions are documented below and none of them blocks a well-designed schema.

### Confirmed structure

- 15,634 daily records
- Daily grain confirmed: `NPD_WELL_BORE_CODE + DATEPRD` is unique (Sections 4, 19)
- `NPD_WELL_BORE_CODE` is the preferred shared identifier (Section 6)
- 7 wellbores
- Daily/monthly identifier systems reconciled — same 7 wellbores, same names, both directions (Section 7)
- Daily-to-monthly production reconciliation: 100% match, 0 coverage gaps, 0 genuine mismatches (Section 21)

### Confirmed modelling decisions

- Daily dataset = primary analytical fact source
- Monthly dataset = reconciliation/reference source, not a second fact source
- Production and injection remain in one daily fact table (Section 24.7)
- `WELL_TYPE` and `FLOW_KIND` = temporal fact attributes, not dimension attributes (Section 12, Section 24.6)
- Field/facility = wellbore attributes for Volve v1, given confirmed cardinality of 1 each (Section 8, Section 24.6)
- Source NULLs and zeros remain distinct throughout — never collapsed into each other (Sections 11, 13, 14, 21)

### Known data-quality issues

| ID | Issue | Severity |
|---|---|---|
| DQ-001 | Pre-field-life injection records — 244 rows, wellbores 5693/5769, dated before 2008-01-01, structurally blank | Low |
| DQ-002 | Shared 12-day reporting gap — wellbores 5351/5599, 2012-01-02 to 2012-01-14, identical window | Low |
| DQ-003 | NULL `ON_STREAM_HRS` — 285 rows, wellbores 5693/5769 only, confirmed structurally blank records | Low |
| DQ-004 | `ON_STREAM_HRS > 24` — 20 rows, DST hypothesis partially supported, not confirmed | Low-Medium |
| DQ-005 | Production-volume/state inconsistency — 4 negative `BORE_WAT_VOL` values, 1 unexplained zero-hours/positive-production row | Low |
| DQ-006 | Injection-volume/state inconsistency — 2 rows with substantial injection volume against zero hours and every pressure/temperature/choke field NULL | Medium |

### Constraint policy

- Only deterministic, validated rules become database constraints — a rule with zero observed violations across all 15,634 rows (Section 24.5)
- REVIEW conditions remain quality-monitoring rules, not schema constraints (Section 25)
- Source exceptions are preserved rather than silently corrected — DQ-001 through DQ-006 load as-is, flagged, not deleted or adjusted

### Overall assessment

The source is suitable for PostgreSQL ingestion provided the documented exceptions and source semantics are preserved.


## 27. Handoff to database design

The data-quality assessment is complete.

The following are considered sufficiently established for database implementation:

1. **Daily fact grain**
   `NPD_WELL_BORE_CODE + DATEPRD`

2. **Wellbore identity**
   `NPD_WELL_BORE_CODE`

3. **Wellbore descriptive attributes**
   `NPD_WELL_BORE_NAME`, `WELL_BORE_CODE`, field/facility attributes

4. **Temporal fact attributes**
   `WELL_TYPE`, `FLOW_KIND`

5. **Daily measurements**
   production volumes, injection volume, operating hours, pressure, temperature, choke measurements

6. **Monthly source**
   retained for reconciliation rather than used as the primary analytical fact source

7. **Data-quality exceptions**
   DQ-001 through DQ-006 remain traceable and must not be silently removed during ingestion

8. **Database constraints**
   derived only from confirmed deterministic rules documented in Section 24

**Next phase:** PostgreSQL database implementation.

---

## END OF DATA-QUALITY ASSESSMENT

**No PostgreSQL objects are created in this notebook.**
